In [1]:
# === Auracelle Charlie - Live 2025: Colab Setup (Reserved Domain) ===
# Installs, cleanup, and ngrok config for aiwargame.ngrok.app
import os

try:
    import pip
    !pip -q install --upgrade pip >/dev/null 2>&1
    !pip -q install streamlit pyngrok==7.2.1 plotly pyvis networkx geopandas folium shapely pyproj rtree >/dev/null 2>&1
except Exception as e:
    print("Dependency install encountered an issue (continuing):", e)

!pkill -f ngrok || true
!pkill -f streamlit || true

try:
    from pyngrok import ngrok
    try:
        ngrok.kill()
        ngrok.disconnect()
    except Exception:
        pass
except Exception:
    pass

os.environ["NGROK_AUTHTOKEN"] = "2vmh7uE9lpuOWrldBNSV68hJKH7_4Ukd3XG92jWofsVoZALiJ"
!ngrok config add-authtoken $NGROK_AUTHTOKEN

print("✅ Setup complete. Reserved domain: aiwargame.ngrok.app")

^C
^C
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
✅ Setup complete. Reserved domain: aiwargame.ngrok.app


# 🧬 Auracelle Charlie - AlphaFold-Style Influence Map

## Visualizing External Policy Pressures & Internal Cultural Forces

### 🎯 What This Shows

Unlike traditional network visualizations that show static relationships, this **AlphaFold-style influence map** reveals the **dynamic forces** acting on each country:

1. **External Policy Influences** (Arrows/Cones)
   - 🔴 Restrictive policies (US export controls, sanctions)
   - 🟠 Regulatory frameworks (EU AI Act, GDPR)
   - 🟡 Technical standards (competing frameworks)
   - 🔵 Governance initiatives (Arctic Council, UN)
   - Arrows point FROM policy source TO affected country
   - Arrow thickness = policy strength/impact

2. **Internal Cultural Forces** (Translucent Spheres)
   - 💙 Democratic accountability pressure
   - ❤️ Authoritarian control imperatives
   - 💚 Indigenous data sovereignty
   - 💜 Tech hub ambitions
   - Sphere size = force strength
   - Sphere encompasses all affected countries

3. **Country Positioning** (3D Space)
   - X-axis: Economic power (GDP)
   - Y-axis: Military capability (% GDP)
   - Z-axis: Tech infrastructure (Internet %)
   - Node size: Overall influence score
   - Node color: Cultural alignment

### 🔬 Key Insights

**Multiple arrows converging** → Country under high external pressure  
**Large encompassing sphere** → Strong internal cultural constraints  
**Arrows + Spheres overlapping** → Complex, constrained policy space  
**Isolated positioning** → Greater policy autonomy

### 📊 E-AGPO-HT Integration

This visualization operationalizes:
- **Stratum III (g-GWC)**: Rapid comprehension of multi-dimensional influence dynamics
- **Stratum II (SAD)**: Alliance detection through spatial clustering + shared force fields
- **Stratum II (STI)**: Tension identification via conflicting force vectors
- **Stratum I (NOF)**: Concrete policy pressures + cultural forces as operational factors

### 🚀 New Countries

- 🇬🇱 **Greenland**: Unique Arctic governance position, minimal external pressure
- 🇻🇪 **Venezuela**: High external pressure (sanctions) + strong internal forces

---

**Author**: Grace Evans | **Organization**: Auracelle AI Governance Labs  
**Framework**: E-AGPO-HT | **Affiliation**: UC Berkeley CLTC


In [2]:
# ========================================
# CELL 1: Install Dependencies
# ========================================
!pip install -q pyngrok streamlit networkx matplotlib numpy pandas torch plotly wbgapi requests scikit-learn PyPDF2 python-docx

# ========================================
# CELL 2: Setup Environment & AGPO Data Package
# ========================================
import os, time, subprocess, sys
from pyngrok import ngrok
from IPython.display import display, HTML

# Cleanup existing processes
!pkill -f streamlit || true
!pkill -f ngrok || true
try:
    ngrok.disconnect("http://localhost:8501")
except:
    pass
try:
    ngrok.kill()
except:
    pass

ngrok.set_auth_token("2vmh7uE9lpuOWrldBNSV68hJKH7_4Ukd3XG92jWofsVoZALiJ")
os.makedirs("pages", exist_ok=True)
os.makedirs("agpo_data", exist_ok=True)
print("✅ Setup complete")

# ========================================
# CELL 3: Create AGPO Data Package - World Bank API Integration
# ========================================
agpo_worldbank = '''"""
AGPO Data Package - World Bank API Integration
Fetches real-world economic and development indicators
"""
import wbgapi as wb
import pandas as pd
import streamlit as st


def _link(label, url):
    return f"- [{label}]({url})"

PROVENANCE_LINKS = [
    ("World Bank API", "https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation"),
    ("IMF Data (WEO)", "https://www.imf.org/en/Publications/WEO/weo-database"),
    ("UN Comtrade", "https://comtradeplus.un.org/"),
    ("SIPRI", "https://www.sipri.org/databases"),
    ("WTO Tariff Data", "https://tao.wto.org/"),
]

COGNITION_FIELDS_LINKS = [
    ("Cognition (overview) – bounded rationality", "https://plato.stanford.edu/entries/bounded-rationality/"),
    ("OODA Loop (Boyd)", "https://en.wikipedia.org/wiki/OODA_loop"),
    ("Sensemaking (Weick)", "https://en.wikipedia.org/wiki/Sensemaking"),
]

BELIEF_FIELDS_LINKS = [
    ("Bayesian belief update (overview)", "https://en.wikipedia.org/wiki/Bayesian_inference"),
    ("Belief state (POMDP concept)", "https://en.wikipedia.org/wiki/Partially_observable_Markov_decision_process"),
]

COGNITION_METRICS_LINKS = [
    ("Calibration (probability)", "https://en.wikipedia.org/wiki/Calibration_(statistics)"),
    ("Brier score", "https://en.wikipedia.org/wiki/Brier_score"),
    ("Entropy", "https://en.wikipedia.org/wiki/Entropy_(information_theory)"),
]

def render_dropdown_links(title, links):
    with st.expander(title, expanded=False):
        st.markdown("
".join([_link(l,u) for l,u in links]))


@st.cache_data(ttl=3600)
def get_world_bank_indicator(indicator_code, country_codes, start_year=2015, end_year=2024):
    """
    Fetch World Bank indicator data for specified countries

    Args:
        indicator_code: WB indicator (e.g., 'NY.GDP.MKTP.CD' for GDP)
        country_codes: List of ISO3 country codes (e.g., ['USA', 'CHN'])
        start_year: Start year for data
        end_year: End year for data

    Returns:
        DataFrame with country, year, indicator value
    """
    try:
        data = wb.data.DataFrame(
            indicator_code,
            country_codes,
            time=range(start_year, end_year + 1),
            labels=True
        )

        # Reshape data
        data_reset = data.reset_index()
        data_melted = data_reset.melt(
            id_vars=['Country'],
            var_name='Year',
            value_name='Value'
        )
        data_melted['Indicator'] = indicator_code

        return data_melted
    except Exception as e:
        st.warning(f"World Bank API error: {e}")
        return pd.DataFrame()

@st.cache_data(ttl=3600)
def get_many_indicators(indicator_codes, country_codes=['USA', 'CHN', 'GBR', 'JPN', 'IND', 'BRA', 'ARE'], start_year=2015, end_year=2024):
    """
    Fetch multiple World Bank indicators at once

    Common indicators:
    - NY.GDP.MKTP.CD: GDP (current US$)
    - MS.MIL.XPND.GD.ZS: Military expenditure (% of GDP)
    - IT.NET.USER.ZS: Internet users (% of population)
    - SP.POP.TOTL: Total population
    - NY.GDP.PCAP.CD: GDP per capita
    """
    all_data = []
    for indicator in indicator_codes:
        df = get_world_bank_indicator(indicator, country_codes, start_year, end_year)
        if not df.empty:
            all_data.append(df)

    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()

def get_latest_gdp(country_code):
    """Get most recent GDP data for a country"""
    try:
        data = get_world_bank_indicator('NY.GDP.MKTP.CD', [country_code], 2020, 2024)
        if not data.empty:
            latest = data.dropna().sort_values('Year', ascending=False).iloc[0]
            value = latest['Value']
            # Handle both string and numeric values
            if isinstance(value, str):
                value = float(value)
            return value / 1e12  # Convert to trillions
        return None
    except Exception as e:
        return None

def get_latest_military_expenditure(country_code):
    """Get most recent military expenditure as % of GDP"""
    try:
        data = get_world_bank_indicator('MS.MIL.XPND.GD.ZS', [country_code], 2020, 2024)
        if not data.empty:
            latest = data.dropna().sort_values('Year', ascending=False).iloc[0]
            value = latest['Value']
            # Handle both string and numeric values
            if isinstance(value, str):
                value = float(value)
            return value
        return None
    except Exception as e:
        return None

def get_internet_penetration(country_code):
    """Get internet users as % of population"""
    try:
        data = get_world_bank_indicator('IT.NET.USER.ZS', [country_code], 2020, 2024)
        if not data.empty:
            latest = data.dropna().sort_values('Year', ascending=False).iloc[0]
            value = latest['Value']
            # Handle both string and numeric values
            if isinstance(value, str):
                value = float(value)
            return value
        return None
    except Exception as e:
        return None
'''

with open('agpo_data/worldbank.py', 'w') as f:
    f.write(agpo_worldbank)

# ========================================
# CELL 4: Create AGPO Data Package - Export Controls API
# ========================================
agpo_exportcontrol = '''"""
AGPO Data Package - US Export Controls Integration
Consolidated Screening List (CSL) from trade.gov
"""
import requests
import pandas as pd
import streamlit as st
from datetime import datetime

@st.cache_data(ttl=86400)  # Cache for 24 hours
def fetch_consolidated_screening_list():
    """
    Fetch US Consolidated Screening List (Entity List, DPL, UVL, etc.)
    Returns DataFrame with sanctioned entities
    Note: This API may require authentication or have rate limits
    """
    url = "https://api.trade.gov/consolidated_screening_list/search"

    try:
        params = {
            "size": 100,
            "offset": 0
        }

        response = requests.get(url, params=params, timeout=10)

        # Check response status
        if response.status_code == 403:
            # API requires authentication
            return pd.DataFrame()

        response.raise_for_status()

        # Check if response has content before parsing
        if not response.text or response.text.strip() == '':
            return pd.DataFrame()

        data = response.json()
        results = data.get('results', [])

        # Parse results
        records = []
        for item in results:
            records.append({
                'name': item.get('name', 'N/A'),
                'country': item.get('addresses', [{}])[0].get('country', 'N/A') if item.get('addresses') else 'N/A',
                'source': item.get('source', 'N/A'),
                'type': item.get('type', 'N/A'),
                'programs': ', '.join(item.get('programs', [])),
                'remarks': item.get('remarks', '')
            })

        return pd.DataFrame(records)

    except Exception as e:
        # Silently return empty DataFrame - API may be unavailable
        return pd.DataFrame()

def check_entity_sanctions(entity_name):
    """Check if an entity appears on screening lists"""
    df = fetch_consolidated_screening_list()
    if df.empty:
        return None

    matches = df[df['name'].str.contains(entity_name, case=False, na=False)]
    return matches

def get_sanctioned_countries():
    """Get list of countries with sanctioned entities"""
    df = fetch_consolidated_screening_list()
    if df.empty:
        return []

    return df['country'].value_counts().to_dict()
'''

with open('agpo_data/exportcontrol.py', 'w') as f:
    f.write(agpo_exportcontrol)

# ========================================
# CELL 5: Create AGPO Data Package - SIPRI Integration
# ========================================
agpo_sipri = '''"""
AGPO Data Package - SIPRI Military Expenditure Integration
Note: SIPRI requires manual CSV download. This module processes the data.
"""
import pandas as pd
import streamlit as st
import io

def parse_sipri_csv(uploaded_file):
    """
    Parse SIPRI military expenditure CSV
    Download from: https://www.sipri.org/databases/milex
    """
    try:
        df = pd.read_csv(uploaded_file, encoding='utf-8')
        return df
    except Exception as e:
        st.error(f"SIPRI CSV parsing error: {e}")
        return pd.DataFrame()

def get_military_expenditure_by_country(df, country_name, year=2023):
    """Extract military expenditure for specific country and year"""
    try:
        if 'Country' in df.columns and str(year) in df.columns:
            country_data = df[df['Country'] == country_name]
            if not country_data.empty:
                value = country_data[str(year)].iloc[0]
                return value
        return None
    except:
        return None

def get_top_military_spenders(df, year=2023, top_n=10):
    """Get top N military spenders for a given year"""
    try:
        if str(year) in df.columns:
            top = df.nlargest(top_n, str(year))[['Country', str(year)]]
            return top
        return pd.DataFrame()
    except:
        return pd.DataFrame()
'''

with open('agpo_data/sipri.py', 'w') as f:
    f.write(agpo_sipri)

# ========================================
# CELL 6: Create AGPO Package Init
# ========================================
agpo_init = '''"""
AGPO Data Package - AI Governance Policy Optimization
Real-world data integration for wargaming simulations
"""
from .worldbank import *
from .exportcontrol import *
from .sipri import *

__version__ = "1.0.0"
'''

with open('agpo_data/__init__.py', 'w') as f:
    f.write(agpo_init)

print("✅ AGPO Data Package created")

# ========================================
# CELL 7: Create Adjudicator Module with API Integration
# ========================================
adjudicator_code = '''import random
import numpy as np
from datetime import datetime
import sys
sys.path.insert(0, ".")

class AgenticAdjudicator:
    """AI Agentic Adjudicator - Neutral referee with real-world data integration"""

    def __init__(self, mode="neutral"):
        self.mode = mode
        self.event_history = []
        self.tension_index = 0.5
        self.shock_types = [
            "economic_sanctions", "cyber_attack", "public_protest",
            "un_resolution", "trade_disruption", "diplomatic_incident",
            "tech_breakthrough", "alliance_shift", "intel_leak"
        ]
        self.real_world_data = {}

    def integrate_real_world_data(self, country_code, gdp=None, mil_exp=None, internet=None, sanctions=None):
        """Store real-world data for adjudication calculations"""
        self.real_world_data[country_code] = {
            'gdp': gdp,
            'military_expenditure_pct': mil_exp,
            'internet_penetration': internet,
            'sanctioned_entities': sanctions
        }

    def calculate_tension_index(self, actor_positions, power_levels, alignment_graph):
        """Calculate geopolitical tension with real-world data weighting"""
        position_divergence = np.std([hash(p) % 100 for p in actor_positions.values()]) / 100
        power_imbalance = np.std(list(power_levels.values()))
        alignment_factor = 1.0 - (sum(alignment_graph.values()) / (len(alignment_graph) + 1e-6))

        # Add real-world military expenditure factor
        mil_exp_factor = 0.0
        if self.real_world_data:
            mil_exps = [d.get('military_expenditure_pct', 2.0) for d in self.real_world_data.values() if d.get('military_expenditure_pct')]
            if mil_exps:
                mil_exp_factor = (np.mean(mil_exps) - 2.0) / 10.0  # Normalize around 2% baseline

        tension = (position_divergence * 0.3 + power_imbalance * 0.2 + alignment_factor * 0.3 + abs(mil_exp_factor) * 0.2)
        self.tension_index = max(0.0, min(1.0, tension))
        return self.tension_index

    def detect_deception(self, stated_position, historical_actions, power_level):
        """Detect potential deception in actor statements"""
        deception_score = 0.0

        if len(historical_actions) > 0:
            consistency = sum([1 for a in historical_actions if a == stated_position]) / len(historical_actions)
            deception_score += (1.0 - consistency) * 0.5

        if power_level < 0.5 and "aggressive" in stated_position.lower():
            deception_score += 0.3

        return min(1.0, deception_score)

    def inject_shock(self, current_round, tension_level):
        """Inject external shock event based on tension and real-world factors"""
        # Increase shock probability if sanctions are detected
        sanctions_multiplier = 1.0
        if self.real_world_data:
            # Handle None values properly when summing sanctions
            total_sanctions = sum([d.get('sanctioned_entities', 0) for d in self.real_world_data.values() if d.get('sanctioned_entities') is not None])
            if total_sanctions > 0:
                sanctions_multiplier = 1.3

        shock_probability = tension_level * 0.3 * sanctions_multiplier

        if random.random() < shock_probability:
            shock_type = random.choice(self.shock_types)
            impact_magnitude = random.uniform(0.1, 0.5) * tension_level

            event = {
                "round": current_round,
                "type": shock_type,
                "magnitude": impact_magnitude,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "narrative": self._generate_narrative(shock_type, impact_magnitude),
                "real_world_triggered": sanctions_multiplier > 1.0
            }

            self.event_history.append(event)
            return event

        return None

    def _generate_narrative(self, shock_type, magnitude):
        """Generate narrative text for shock events"""
        severity = "severe" if magnitude > 0.3 else "moderate"
        narratives = {
            "economic_sanctions": f"Breaking: Coalition imposes {severity} economic sanctions",
            "cyber_attack": f"Alert: {severity.capitalize()} cyber incident targets critical infrastructure",
            "public_protest": f"Developing: {severity.capitalize()} public demonstrations challenge policy",
            "un_resolution": f"UN Security Council debates {severity} resolution",
            "trade_disruption": f"Economic shock: {severity.capitalize()} trade route disruption",
            "diplomatic_incident": f"Diplomatic crisis: {severity.capitalize()} incident strains relations",
            "tech_breakthrough": f"Tech update: {severity.capitalize()} AI capability announced",
            "alliance_shift": f"Geopolitical shift: {severity.capitalize()} realignment detected",
            "intel_leak": f"Intelligence alert: {severity.capitalize()} data leak exposed"
        }
        return narratives.get(shock_type, "Unknown event occurred")

    def adjudicate(self, actor_beliefs, power_levels, alignment_graph, current_round):
        """Main adjudication function with real-world data integration"""
        tension = self.calculate_tension_index(actor_beliefs, power_levels, alignment_graph)
        shock_event = self.inject_shock(current_round, tension)

        confidence = 1.0 - (tension * 0.3)

        deception_scores = {}
        for actor, belief in actor_beliefs.items():
            hist = [e.get(actor) for e in self.event_history if actor in e]
            deception_scores[actor] = self.detect_deception(belief, hist, power_levels.get(actor, 0.5))

        return {
            "tension_index": tension,
            "shock_event": shock_event,
            "confidence_score": confidence,
            "deception_scores": deception_scores,
            "narrative": shock_event["narrative"] if shock_event else "Situation stable",
            "round": current_round,
            "real_world_integrated": len(self.real_world_data) > 0
        }
'''

with open('adjudicator.py', 'w') as f:
    f.write(adjudicator_code)

print("✅ Enhanced Adjudicator with API integration created")

# ========================================
# CELL 8: Create Login Page
# ========================================
app_code = '''import streamlit as st

st.set_page_config(page_title="Auracelle Charlie 3 - War Gaming Stress-Testing Policy Governance Research Simulation/Prototype", layout="wide", initial_sidebar_state="collapsed")
st.title("🔐 Auracelle Charlie 3 - War Gaming Stress-Testing Policy Governance Research Simulation/Prototype")

if "authenticated" not in st.session_state:
    st.session_state["authenticated"] = False

with st.form("login_form"):
    username = st.text_input("Username")
    password = st.text_input("Password", type="password")

    st.markdown("### 🎮 Phase 3 Features and Functionality")

    st.markdown("**Capabilities**")

    capabilities = [
        "🌍 World Bank API (GDP, military expenditure, internet penetration)",
        "🚫 US Export Controls API (sanctions screening)",
        "💥 External shock injection system",
        "🎭 Deception detection with real-world data",
        "🗺️ 3-D Influence Map",
        "🛡️ Red Teaming Foresight",
        "🧠 Evans-AGPO-HT Cognitive Science Framework"
    ]

    for c in capabilities:
        st.write(c)


    submit = st.form_submit_button("Login")

if submit:
    if password == "charlie2025":
        st.session_state["authenticated"] = True
        st.session_state["username"] = username
        st.switch_page("pages/1_Session_Setup.py")
    else:
        st.error("Incorrect password. Access denied.")

if st.session_state.get("authenticated", False):
    if not st.session_state.get("setup_complete", False):
        st.switch_page("pages/1_Session_Setup.py")
    else:
        st.switch_page("pages/2_Simulation.py")
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("✅ Login page created")

# ========================================
# CELL 9: Create Simulation Page with Full API Integration
# ========================================
sim_code = '''import streamlit as st
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sys
sys.path.insert(0, ".")
from adjudicator import AgenticAdjudicator
from agpo_data.research_store import init_db, log_move, set_outcomes, upsert_session


# Import AGPO data modules
try:
    from agpo_data.worldbank import get_latest_gdp, get_latest_military_expenditure, get_internet_penetration, get_many_indicators
    from agpo_data.exportcontrol import fetch_consolidated_screening_list, get_sanctioned_countries
    from agpo_data.sipri import parse_sipri_csv
    AGPO_AVAILABLE = True
except ImportError as e:
    st.warning(f"AGPO modules not fully loaded: {e}")
    AGPO_AVAILABLE = False

st.set_page_config(page_title="Auracelle Charlie 3 - War Gaming Stress-Testing Policy Governance Research Simulation/Prototype", layout="wide", initial_sidebar_state="collapsed")

init_db()
import uuid
if "session_id" not in st.session_state:
    st.session_state["session_id"] = str(uuid.uuid4())
upsert_session(st.session_state["session_id"], scenario=st.session_state.get("scenario"), condition_tag=st.session_state.get("condition_tag"))


if not st.session_state.get("authenticated", False):
    st.warning("Please log in first.")
    st.switch_page("app.py")

if not st.session_state.get("setup_complete", False):
    st.info("Please complete Session Setup before entering the simulation.")
    st.switch_page("pages/1_Session_Setup.py")


st.title("🎯 Auracelle Charlie 3 - Policy Stress-Testing Wargame")

st.header("Auracelle Charlie 3 - War Gaming Stress-Testing Policy Governance Research Simulation/Prototype")




# Initialize session state
if "round" not in st.session_state:
    st.session_state["round"] = 1
if "q_table" not in st.session_state:
    st.session_state["q_table"] = {}
if "adjudicator" not in st.session_state:
    st.session_state["adjudicator"] = AgenticAdjudicator(mode="neutral")
if "event_log" not in st.session_state:
    st.session_state["event_log"] = []
# New: round-level metrics and agent controls
if "round_metrics_trace" not in st.session_state:
    st.session_state["round_metrics_trace"] = []
if "episode_length" not in st.session_state:
    st.session_state["episode_length"] = 5
if "stochastic_exploration" not in st.session_state:
    st.session_state["stochastic_exploration"] = False
if "api_data_loaded" not in st.session_state:
    st.session_state["api_data_loaded"] = False

# Animation and click session state
if "animation_running" not in st.session_state:
    st.session_state["animation_running"] = False
if "selected_country_click" not in st.session_state:
    st.session_state["selected_country_click"] = None

adjudicator = st.session_state["adjudicator"]

# Sidebar for API Data Controls
with st.sidebar:
    st.header("🌍 Real-World Data Integration")

    if st.button("🔄 Refresh API Data"):
        st.session_state["api_data_loaded"] = False
        st.rerun()

    st.markdown("---")
    st.markdown("**Data Sources:**")
    st.markdown("- World Bank API ✅")
    st.markdown("- US Export Controls ✅")
    st.markdown("- SIPRI (CSV Upload)")

    sipri_file = st.file_uploader("Upload SIPRI CSV", type=['csv'])

    with st.expander("📊 Data & Evidence Sources (Traceability)", expanded=False):
        st.markdown("""
- **World Bank (wbgapi)**: macro indicators (GDP, military expenditure, internet penetration) pulled via API.
- **U.S. Consolidated Screening List (CSL)**: export-control / sanctions screening via API wrapper.
- **SIPRI (upload)**: defense / military expenditure reference data (CSV upload).

**Interpretation:** When a score, recommendation, or warning is shown, it should be attributable to one or more of these sources
and the transformations applied inside the sandbox (metric → source → vintage/date → transformation → round/time).
""")



# Country mapping to ISO codes
country_to_iso = {
    "Dubai": "ARE", "United Kingdom": "GBR", "United States": "USA",
    "Japan": "JPN", "China": "CHN", "Brazil": "BRA", "India": "IND",
    "Greenland": "GRL", "Venezuela": "VEN"
}

policy_options = [
    "EU Artificial Intelligence Act (AI Act) - Regulation (EU) 2024/1689",
    "EU General Data Protection Regulation (GDPR) - Regulation (EU) 2016/679",
    "EU Digital Services Act (DSA) - Regulation (EU) 2022/2065",
    "EU NIS2 Directive - Directive (EU) 2022/2555",
    "UNESCO Recommendation on the Ethics of Artificial Intelligence (2021)",
    "OECD Recommendation of the Council on Artificial Intelligence (OECD AI Principles, 2019)",
    "American AI Action Plan",
    "NATO Article 5"
]
selected_policy = st.selectbox("Select Policy Scenario", policy_options)

policy_scenario_explanations = {
    "EU Artificial Intelligence Act (AI Act) - Regulation (EU) 2024/1689": "Risk-tiering, high-risk controls, governance obligations—perfect for stress-testing compliance, innovation tradeoffs, and cross-border adoption.",
    "EU General Data Protection Regulation (GDPR) - Regulation (EU) 2016/679": "The global reference point for privacy, data rights, and cross-border data governance (and it directly collides with AI training / data minimization debates).",
    "EU Digital Services Act (DSA) - Regulation (EU) 2022/2065": "Platform accountability and systemic-risk controls—useful for stress-testing content moderation, transparency, and disinformation pressures in a crisis.",
    "EU NIS2 Directive - Directive (EU) 2022/2555": "Cybersecurity governance obligations for critical and important entities—ideal for stress-testing incident response, resilience duties, and cross-border coordination under attack.",
    "UNESCO Recommendation on the Ethics of Artificial Intelligence (2021)": "A global ethical baseline for AI—useful for stress-testing value conflicts, implementation gaps, and alignment across diverse governance cultures.",
    "OECD Recommendation of the Council on Artificial Intelligence (OECD AI Principles, 2019)": "Widely adopted principles (robustness, transparency, accountability)—useful for stress-testing practical translation of norms into enforceable controls and incentives.",
    "American AI Action Plan": "A U.S.-anchored strategic blueprint for advancing AI while managing risks—useful for stress-testing national competitiveness vs safety, procurement, and cross-sector adoption tradeoffs.",
    "NATO Article 5": "Collective defense clause—useful for stress-testing alliance decision-making if AI-enabled cyber incidents or escalation pressures trigger collective response debates."
}

st.caption("Scenario brief")
st.info(policy_scenario_explanations.get(selected_policy, ""))
country_options = ["Dubai", "United Kingdom", "United States", "Japan", "China", "Brazil", "India", "Russia", "Iraq", "Qatar", "NATO", "Greenland", "Venezuela"]

# Default data structure
default_data = {
    "Dubai": {"gdp": 0.5, "influence": 0.7, "position": "Moderate regulatory stance", "mil_exp": 5.6, "internet": 99.0, "cultural_alignment": "Western-Middle East hybrid"},
    "United Kingdom": {"gdp": 3.2, "influence": 0.85, "position": "Supports EU-style data protection", "mil_exp": 2.2, "internet": 96.0, "cultural_alignment": "Western"},
    "United States": {"gdp": 21.0, "influence": 0.95, "position": "Favors innovation over regulation", "mil_exp": 3.4, "internet": 92.0, "cultural_alignment": "Western"},
    "Japan": {"gdp": 5.1, "influence": 0.88, "position": "Pro-regulation for trust", "mil_exp": 1.0, "internet": 95.0, "cultural_alignment": "Eastern-Western hybrid"},
    "China": {"gdp": 17.7, "influence": 0.93, "position": "Strict state-driven AI governance", "mil_exp": 1.7, "internet": 73.0, "cultural_alignment": "Eastern"},
    "Brazil": {"gdp": 2.0, "influence": 0.75, "position": "Leaning toward EU-style regulation", "mil_exp": 1.4, "internet": 81.0, "cultural_alignment": "Latin American"},
    "India": {"gdp": 3.7, "influence": 0.82, "position": "Strategic tech balancing", "mil_exp": 2.4, "internet": 43.0, "cultural_alignment": "South Asian"},
    "Russia": {"gdp": 1.8, "influence": 0.78, "position": "Sovereign tech control", "mil_exp": 4.3, "internet": 85.0, "cultural_alignment": "Eastern"},
    "Iraq": {"gdp": 0.2, "influence": 0.42, "position": "Developing governance framework", "mil_exp": 3.5, "internet": 49.0, "cultural_alignment": "Middle East"},
    "Qatar": {"gdp": 0.18, "influence": 0.68, "position": "Tech-forward with state oversight", "mil_exp": 3.7, "internet": 99.0, "cultural_alignment": "Middle East"},
    "NATO": {"gdp": 25.0, "influence": 0.97, "position": "Collective security & data interoperability", "mil_exp": 2.5, "internet": 90.0, "cultural_alignment": "Western Alliance"},
    "Greenland": {"gdp": 0.003, "influence": 0.45, "position": "Emerging Arctic tech governance", "mil_exp": 0.0, "internet": 68.0, "cultural_alignment": "Nordic"},
    "Venezuela": {"gdp": 0.048, "influence": 0.58, "position": "State-controlled digital infrastructure", "mil_exp": 0.9, "internet": 72.0, "cultural_alignment": "Latin American"}
}


# Load real-world data
if AGPO_AVAILABLE and not st.session_state["api_data_loaded"]:
    with st.spinner("🌍 Loading real-world data from APIs..."):
        for country, iso_code in country_to_iso.items():
            try:
                # Get World Bank data
                gdp = get_latest_gdp(iso_code)
                mil_exp = get_latest_military_expenditure(iso_code)
                internet = get_internet_penetration(iso_code)

                # Update default data with real values
                if gdp is not None:
                    try:
                        default_data[country]["gdp"] = round(float(gdp), 2)
                    except (ValueError, TypeError):
                        pass
                if mil_exp is not None:
                    try:
                        default_data[country]["mil_exp"] = round(float(mil_exp), 2)
                    except (ValueError, TypeError):
                        pass
                if internet is not None:
                    try:
                        default_data[country]["internet"] = round(float(internet), 1)
                    except (ValueError, TypeError):
                        pass

                # Integrate into adjudicator
                adjudicator.integrate_real_world_data(
                    iso_code,
                    gdp=default_data[country]["gdp"],
                    mil_exp=default_data[country]["mil_exp"],
                    internet=default_data[country]["internet"]
                )

            except Exception as e:
                st.warning(f"Could not load data for {country}: {e}")

        # Get sanctions data
        try:
            sanctions_df = fetch_consolidated_screening_list()
            if not sanctions_df.empty:
                sanctioned_countries = get_sanctioned_countries()
                for country, iso_code in country_to_iso.items():
                    country_name = country if country != "Dubai" else "United Arab Emirates"
                    sanction_count = sanctioned_countries.get(country_name, 0)
                    adjudicator.integrate_real_world_data(
                        iso_code,
                        sanctions=sanction_count
                    )
        except Exception as e:
            # Silently continue if export controls API fails
            pass

        st.session_state["api_data_loaded"] = True
        st.success("✅ Real-world data loaded successfully!")

selected_country_a = st.selectbox("Select Country A", country_options, index=0)
selected_country_b = st.selectbox("Select Country B", country_options, index=1)

role_tags = ["Governance","MilitaryAI","DataPrivacy","ExportControl","Diplomacy","StandardSetting","Surveillance","Trade","TechAlliance"]
role_country_a = st.selectbox(f"Role for {selected_country_a}", role_tags, key="role_a")
role_country_b = st.selectbox(f"Role for {selected_country_b}", role_tags, key="role_b")

player_country = st.selectbox("🎖️ You represent:", country_options, index=0)

# --- Round header + Agent-style controls ---
round_col1, round_col2, round_col3 = st.columns([1, 1, 1])

with round_col1:
    st.subheader(f"🕐 Round: {st.session_state['round']}")

with round_col2:
    # Episode length controls how many rounds conceptually form one episode
    st.session_state["episode_length"] = st.slider(
        "Episode Length (rounds)",
        min_value=1,
        max_value=30,
        value=st.session_state.get("episode_length", 5),
        key="episode_length_slider"
    )

with round_col3:
    st.session_state["stochastic_exploration"] = st.toggle(
        "Enable Stochastic Exploration",
        value=st.session_state.get("stochastic_exploration", False),
        help="When enabled, the adjudication introduces variability into reward and risk.",
        key="stochastic_exploration_toggle"
    )

# --- Reward & Risk computation (lightweight, sim-style) ---
# alignment_score is set later during Policy Position Comparison; default safely here
alignment_score = float(st.session_state.get("alignment_score", 0.5))


# ---- Safe defaults for reward components (prevents NameError in early rounds)
adjudication = st.session_state.get("adjudication")
if not isinstance(adjudication, dict):
    adjudication = {
        "tension_index": 0.5,
        "confidence_score": 0.7,
        "status": "not evaluated",
        "notes": "Agentic adjudicator updates after inputs are provided."
    }
    st.session_state["adjudication"] = adjudication

base_reward = (alignment_score * 0.5
               + (1.0 - adjudication["tension_index"]) * 0.3
               + adjudication["confidence_score"] * 0.2)

if st.session_state.get("stochastic_exploration", False):
    exploration_noise = np.random.normal(0, 0.05)
    base_reward = max(0.0, min(1.0, base_reward + exploration_noise))

reward = float(base_reward)
risk = float(adjudication["tension_index"])

# Log round metrics for batch evaluation
st.session_state["round_metrics_trace"].append({
    "round": int(st.session_state["round"]),
    "reward": reward,
    "risk": risk,
    "tension": float(adjudication["tension_index"]),
    "confidence": float(adjudication["confidence_score"]),
    "alignment": float(alignment_score),
})

# --- Reward & Risk display ---
metric_col1, metric_col2, metric_col3 = st.columns(3)
with metric_col1:
    st.metric("🏆 Reward", f"{reward*100:.1f}")
with metric_col2:
    st.metric("⚠️ Risk", f"{risk*100:.1f}")
with metric_col3:
    # Episode progress within the chosen episode length
    ep_pos = (st.session_state["round"] - 1) % st.session_state["episode_length"] + 1
    st.metric("📏 Episode Progress", f"{ep_pos}/{st.session_state['episode_length']}")

# --- Round navigation controls ---
nav_col1, nav_col2 = st.columns(2)
with nav_col1:
    if st.button("▶️ Next Round"):
        log_move(session_id=st.session_state.get("session_id"), participant_id=st.session_state.get("participant_id"), round_num=st.session_state.get("round_num"), policy=st.session_state.get("policy_selected"), action="next_round", notes="", state={"reward": st.session_state.get("reward"), "risk": st.session_state.get("risk"), "tension": st.session_state.get("tension"), "alignment": st.session_state.get("alignment")})
        st.session_state["round"] += 1
        st.rerun()
with nav_col2:
    if st.button("🔄 Reset Episode"):
        st.session_state["round"] = 1
        st.session_state["round_metrics_trace"] = []
        st.rerun()



# Display real-world data comparison
st.subheader("🤖 AI Agentic Adjudicator Status")

actor_beliefs = {
    selected_country_a: default_data[selected_country_a]["position"],
    selected_country_b: default_data[selected_country_b]["position"]
}
power_levels = {
    selected_country_a: default_data[selected_country_a]["influence"],
    selected_country_b: default_data[selected_country_b]["influence"]
}
alignment_graph = {(selected_country_a, selected_country_b): alignment_score}

adjudication = adjudicator.adjudicate(actor_beliefs, power_levels, alignment_graph, st.session_state["round"])

col1, col2, col3, col4 = st.columns(4)
with col1:
    tension_color = "🔴" if adjudication["tension_index"] > 0.7 else ("🟡" if adjudication["tension_index"] > 0.4 else "🟢")
    st.metric(f"{tension_color} Tension", f"{adjudication['tension_index']*100:.1f}%")
with col2:
    st.metric("🎲 Confidence", f"{adjudication['confidence_score']*100:.1f}%")
with col3:
    st.metric("📊 Alignment", f"{alignment_score*100:.0f}%")
with col4:
    real_world_icon = "✅" if adjudication.get("real_world_integrated") else "⚠️"
    st.metric(f"{real_world_icon} Real Data", "Active" if adjudication.get("real_world_integrated") else "Simulated")

st.markdown("### 🎭 Deception Detection")
deception_df = pd.DataFrame([
    {"Actor": k, "Risk": f"{v*100:.1f}%", "Status": "⚠️ Suspicious" if v > 0.5 else "✅ Consistent"}
    for k, v in adjudication["deception_scores"].items()
])
st.dataframe(deception_df, use_container_width=True)

if adjudication["shock_event"]:
    shock_prefix = "💥 **REAL-WORLD TRIGGERED SHOCK**" if adjudication["shock_event"].get("real_world_triggered") else "💥 **EXTERNAL SHOCK**"
    st.warning(f"{shock_prefix}\\n\\n{adjudication['narrative']}")
    st.session_state["event_log"].append(adjudication["shock_event"])
else:
    st.info(f"ℹ️ {adjudication['narrative']}")

st.subheader("🆚 Policy Position Comparison (Real-World Data)")
comparison_df = pd.DataFrame({
    "Metric": ["GDP (Trillion USD)", "Military Exp (% GDP)", "Internet Penetration (%)", "Influence Score", "AI Policy Position"],
    selected_country_a: [
        f"${default_data[selected_country_a]['gdp']:.2f}T",
        f"{default_data[selected_country_a]['mil_exp']:.1f}%",
        f"{default_data[selected_country_a]['internet']:.1f}%",
        default_data[selected_country_a]["influence"],
        default_data[selected_country_a]["position"]
    ],
    selected_country_b: [
        f"${default_data[selected_country_b]['gdp']:.2f}T",
        f"{default_data[selected_country_b]['mil_exp']:.1f}%",
        f"{default_data[selected_country_b]['internet']:.1f}%",
        default_data[selected_country_b]["influence"],
        default_data[selected_country_b]["position"]
    ]
})
st.table(comparison_df)

player_new_position = st.text_input("📜 Propose New Policy Position", value=default_data[player_country]["position"])
opponent_country = selected_country_b if player_country == selected_country_a else selected_country_a
alignment_score = 1.0 if player_new_position == default_data[opponent_country]["position"] else 0.0


st.session_state["alignment_score"] = float(alignment_score)
st.markdown("---")


st.markdown("---")
# Strategic Analysis
st.markdown("---")
st.subheader("🎯 Strategic Analysis & Recommendations")

col_left, col_right = st.columns(2)

with col_left:
    st.markdown(f"**{selected_country_a} Assessment**")

    # Economic power analysis
    gdp_a = default_data[selected_country_a]["gdp"]
    gdp_b = default_data[selected_country_b]["gdp"]
    gdp_ratio = gdp_a / (gdp_b + 0.01)

    if gdp_ratio > 2.0:
        st.success(f"✅ Strong economic advantage ({gdp_ratio:.1f}x GDP)")
    elif gdp_ratio < 0.5:
        st.warning(f"⚠️ Economic disadvantage ({gdp_ratio:.1f}x GDP)")
    else:
        st.info(f"📊 Comparable economic power ({gdp_ratio:.1f}x GDP)")

    # Military analysis
    mil_a = default_data[selected_country_a]["mil_exp"]
    if mil_a > 3.0:
        st.warning(f"⚠️ High military expenditure ({mil_a:.1f}% GDP) - potential aggression signal")
    elif mil_a < 1.5:
        st.info(f"🕊️ Low military expenditure ({mil_a:.1f}% GDP) - peaceful stance")
    else:
        st.success(f"✅ Moderate military spending ({mil_a:.1f}% GDP)")

    # Digital infrastructure
    internet_a = default_data[selected_country_a]["internet"]
    if internet_a > 90:
        st.success(f"✅ Advanced digital infrastructure ({internet_a:.0f}% penetration)")
    elif internet_a < 60:
        st.warning(f"⚠️ Limited digital infrastructure ({internet_a:.0f}% penetration)")
    else:
        st.info(f"📊 Developing digital infrastructure ({internet_a:.0f}% penetration)")

with col_right:
    st.markdown(f"**{selected_country_b} Assessment**")

    # Economic power analysis
    if gdp_ratio < 0.5:
        st.success(f"✅ Strong economic advantage ({1/gdp_ratio:.1f}x GDP)")
    elif gdp_ratio > 2.0:
        st.warning(f"⚠️ Economic disadvantage ({1/gdp_ratio:.1f}x GDP)")
    else:
        st.info(f"📊 Comparable economic power ({1/gdp_ratio:.1f}x GDP)")

    # Military analysis
    mil_b = default_data[selected_country_b]["mil_exp"]
    if mil_b > 3.0:
        st.warning(f"⚠️ High military expenditure ({mil_b:.1f}% GDP) - potential aggression signal")
    elif mil_b < 1.5:
        st.info(f"🕊️ Low military expenditure ({mil_b:.1f}% GDP) - peaceful stance")
    else:
        st.success(f"✅ Moderate military spending ({mil_b:.1f}% GDP)")

    # Digital infrastructure
    internet_b = default_data[selected_country_b]["internet"]
    if internet_b > 90:
        st.success(f"✅ Advanced digital infrastructure ({internet_b:.0f}% penetration)")
    elif internet_b < 60:
        st.warning(f"⚠️ Limited digital infrastructure ({internet_b:.0f}% penetration)")
    else:
        st.info(f"📊 Developing digital infrastructure ({internet_b:.0f}% penetration)")

# Adjudicator Recommendations
st.markdown("#### 🤖 Adjudicator Recommendations")

if adjudication["tension_index"] > 0.7:
    st.error("""
    **🔴 CRITICAL TENSION LEVEL**
    - Immediate de-escalation measures recommended
    - Consider confidence-building measures
    - Establish backchannel communications
    - Risk of shock events: HIGH
    """)
elif adjudication["tension_index"] > 0.4:
    st.warning("""
    **🟡 ELEVATED TENSION**
    - Monitor situation closely
    - Prepare contingency plans
    - Diplomatic engagement advised
    - Risk of shock events: MODERATE
    """)
else:
    st.success("""
    **🟢 LOW TENSION**
    - Favorable conditions for negotiation
    - Opportunity for comprehensive agreements
    - Continue current engagement strategy
    - Risk of shock events: LOW
    """)

# Export Functionality
st.markdown("---")
st.subheader("📥 Export Simulation Data")

col_export1, col_export2 = st.columns(2)

with col_export1:
    if st.button("📊 Export Full Report"):
        data_status = 'Active ✅' if adjudication.get('real_world_integrated') else 'Simulated ⚠️'

        report = f"""# Auracelle Charlie 3 - War Gaming Stress-Testing Policy Governance Research Simulation/Prototype - Simulation Report
## Generated: {adjudication.get('timestamp', 'N/A')}
### Round: {st.session_state['round']}

## Configuration
- **Policy Scenario:** {selected_policy}
- **Country A:** {selected_country_a} (Role: {role_country_a})
- **Country B:** {selected_country_b} (Role: {role_country_b})
- **Player:** {player_country}

## Real-World Data Integration
- **Data Sources:** World Bank API, US Export Controls API
- **Status:** {data_status}

## Country A: {selected_country_a}
- **GDP:** ${default_data[selected_country_a]['gdp']:.2f} Trillion
- **Military Expenditure:** {default_data[selected_country_a]['mil_exp']:.1f}% of GDP
- **Internet Penetration:** {default_data[selected_country_a]['internet']:.1f}%
- **Influence Score:** {default_data[selected_country_a]['influence']:.2f}
- **Policy Position:** {default_data[selected_country_a]['position']}

## Country B: {selected_country_b}
- **GDP:** ${default_data[selected_country_b]['gdp']:.2f} Trillion
- **Military Expenditure:** {default_data[selected_country_b]['mil_exp']:.1f}% of GDP
- **Internet Penetration:** {default_data[selected_country_b]['internet']:.1f}%
- **Influence Score:** {default_data[selected_country_b]['influence']:.2f}
- **Policy Position:** {default_data[selected_country_b]['position']}

## Adjudication Results
- **Geopolitical Tension:** {adjudication['tension_index']*100:.1f}%
- **Alignment Score:** {alignment_score*100:.0f}%
- **Adjudicator Confidence:** {adjudication['confidence_score']*100:.1f}%

## Deception Analysis
"""
        for actor, score in adjudication['deception_scores'].items():
            risk_pct = score * 100
            report += f"- **{actor}:** {risk_pct:.1f}% risk\\n"

        total_events = len(st.session_state['event_log'])
        report += f"""
## Latest Event
{adjudication['narrative']}

## Total Events: {total_events}
"""

        st.download_button(
            label="📄 Download Report (Markdown)",
            data=report,
            file_name=f"auracelle_report_round_{st.session_state['round']}.md",
            mime="text/markdown"
        )

with col_export2:
    if st.button("📋 Export Event Log"):
        if st.session_state["event_log"]:
            event_df = pd.DataFrame(st.session_state["event_log"])
            csv = event_df.to_csv(index=False)
            st.download_button(
                label="💾 Download CSV",
                data=csv,
                file_name=f"auracelle_events_round_{st.session_state['round']}.csv",
                mime="text/csv"
            )
        else:
            st.info("No events to export yet")

st.markdown("---")
st.caption("🤖 Auracelle Charlie 3 - War Gaming Stress-Testing Policy Governance Research Simulation/Prototype - AI Agentic Adjudicator with Real-World Data Integration | Powered by World Bank, US Export Controls, and SIPRI")
'''

with open('pages/2_Simulation.py', 'w') as f:
    f.write(sim_code)

# ========================================
# NEW: Create Real-World Data Metrics + Instructions Pages
# ========================================
os.makedirs("pages", exist_ok=True)

real_world_metrics_code = r"""import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Real-World Data Metrics", page_icon="🌍", layout="wide", initial_sidebar_state="collapsed")

# Auth gate (same pattern as the app)
if not st.session_state.get("authenticated", False):
    st.warning("Please log in first.")
    st.switch_page("app.py")

st.header("🌍 Real-World Data")

# Actor selector
actor_options = [
    "Dubai","United Kingdom","United States","Japan","China","Brazil","India","NATO",
    "Israel","Paraguay","Belgium","Denmark","Ukraine","Serbia","Argentina",
    "Norway","Switzerland","Poland","Global South"
]
default_a = st.session_state.get("selected_country_a", "United Kingdom")
default_b = st.session_state.get("selected_country_b", "United States")

col1, col2 = st.columns(2)
with col1:
    selected_actor_a = st.selectbox("Actor A", actor_options, index=actor_options.index(default_a) if default_a in actor_options else 1)
with col2:
    selected_actor_b = st.selectbox("Actor B", actor_options, index=actor_options.index(default_b) if default_b in actor_options else 2)

st.session_state["selected_country_a"] = selected_actor_a
st.session_state["selected_country_b"] = selected_actor_b

# Mappings (WB uses ISO3; NATO is not a WB country entity)
ACTOR_TO_ISO3 = {
    "Dubai": "ARE",              # UAE proxy (as in Charlie map)
    "United Kingdom": "GBR",
    "United States": "USA",
    "Japan": "JPN",
    "China": "CHN",
    "Brazil": "BRA",
    "India": "IND",
}

INDICATORS = {
    "gdp": ("NY.GDP.MKTP.CD", "GDP (current US$)"),
    "mil_exp": ("MS.MIL.XPND.GD.ZS", "Military expenditure (% of GDP)"),
    "internet": ("IT.NET.USER.ZS", "Internet users (% of population)"),
}

# Install and import World Bank API
AGPO_AVAILABLE = True
try:
    import wbgapi as wb
except ImportError:
    try:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wbgapi"])
        import wbgapi as wb
        st.success("✅ World Bank API installed successfully")
    except Exception as e:
        AGPO_AVAILABLE = False
        st.error(f"Could not install World Bank API: {e}")
except Exception as e:
    AGPO_AVAILABLE = False
    st.error(f"World Bank API error: {e}")

tabs = st.tabs(["📊 Economic Indicators", "🚫 Export Controls", "📜 Event History", "🧪 Batch Evaluation"])

with tabs[0]:
    st.subheader("📊 Economic Indicators")
    if not AGPO_AVAILABLE:
        st.error("World Bank client not available in this environment.")
    else:
        iso3_list = [ACTOR_TO_ISO3.get(a) for a in [selected_actor_a, selected_actor_b] if ACTOR_TO_ISO3.get(a)]
        label_list = [a for a in [selected_actor_a, selected_actor_b] if ACTOR_TO_ISO3.get(a)]

        if not iso3_list:
            st.info("Selected actors are not available in World Bank country data (e.g., NATO).")
        else:
            start_year, end_year = 2015, 2024

            def fetch_latest(ind_code: str) -> pd.DataFrame:
                try:
                    df = wb.data.DataFrame(ind_code, iso3_list, time=range(start_year, end_year + 1), labels=True)
                    # df index: Country; columns: years (as strings) -> values
                    df = df.reset_index().rename(columns={"economy": "iso3", "Country": "country"})
                    # Keep only year columns
                    year_cols = [c for c in df.columns if str(c).isdigit()]
                    # Find latest non-null per row
                    def latest_nonnull(row):
                        for y in sorted(year_cols, key=lambda x: int(x), reverse=True):
                            v = row.get(y)
                            if pd.notna(v):
                                return float(v), int(y)
                        return None, None

                    latest_vals = df.apply(lambda r: latest_nonnull(r), axis=1, result_type="expand")
                    df["value"], df["year"] = latest_vals[0], latest_vals[1]
                    return df[["country", "value", "year"]]
                except Exception as e:
                    st.warning(f"World Bank API error: {e}")
                    return pd.DataFrame(columns=["country", "value", "year"])

            rows = []
            for key, (code, label) in INDICATORS.items():
                t = fetch_latest(code)
                if not t.empty:
                    t["metric"] = key
                    t["label"] = label
                    rows.append(t)

            if not rows:
                st.warning("No indicator data returned for the selected actors.")
            else:
                long = pd.concat(rows, ignore_index=True)
                wide = long.pivot_table(index="country", columns="metric", values="value", aggfunc="first")
                st.dataframe(wide, use_container_width=True)

                # Plots
                if "gdp" in wide.columns:
                    fig_gdp = px.bar(wide.reset_index(), x="country", y="gdp", title="GDP Comparison (latest available year)")
                    st.plotly_chart(fig_gdp, use_container_width=True)

                if "mil_exp" in wide.columns:
                    fig_mil = px.bar(wide.reset_index(), x="country", y="mil_exp", title="Military Expenditure (% of GDP) (latest available year)")
                    st.plotly_chart(fig_mil, use_container_width=True)

                st.markdown("#### 🌐 Internet Penetration")
                if "internet" in wide.columns:
                    fig_int = px.bar(wide.reset_index(), x="country", y="internet", title="Internet Users (% of population) (latest available year)")
                    st.plotly_chart(fig_int, use_container_width=True)
                else:
                    st.info("Internet penetration data not available for the selected actors.")

with tabs[1]:
    st.subheader("🚫 Export Controls & Sanctions Screening")

    # US Export Controls - BIS Entity List screening
    st.markdown("### 🇺🇸 U.S. Bureau of Industry and Security (BIS) - Entity List")

    # Map actors to potential entities
    entity_search_terms = {
        "China": ["Huawei", "SMIC", "YMTC"],
        "Dubai": ["UAE entities"],
        "United Kingdom": ["No major entities"],
        "United States": ["No entities listed"],
        "Japan": ["No major entities"],
        "Brazil": ["No major entities"],
        "India": ["No major entities"],
        "NATO": ["N/A - Alliance"]
    }

    col1, col2 = st.columns(2)

    with col1:
        st.markdown(f"**{selected_actor_a}**")
        terms = entity_search_terms.get(selected_actor_a, ["No data"])
        if terms[0] == "No major entities" or terms[0] == "No entities listed":
            st.success(f"✅ {terms[0]} on BIS Entity List")
        elif terms[0] == "N/A - Alliance":
            st.info("N/A - Alliance entity")
        else:
            st.warning(f"⚠️ Known entities: {', '.join(terms)}")
            st.caption("Note: Presence on Entity List restricts technology exports")

    with col2:
        st.markdown(f"**{selected_actor_b}**")
        terms = entity_search_terms.get(selected_actor_b, ["No data"])
        if terms[0] == "No major entities" or terms[0] == "No entities listed":
            st.success(f"✅ {terms[0]} on BIS Entity List")
        elif terms[0] == "N/A - Alliance":
            st.info("N/A - Alliance entity")
        else:
            st.warning(f"⚠️ Known entities: {', '.join(terms)}")
            st.caption("Note: Presence on Entity List restricts technology exports")

    st.markdown("---")
    st.markdown("### 🌐 International Sanctions Regimes")
    st.info("💡 **Integration Note**: This section can be enhanced with live API connections to:")
    st.markdown("- **OFAC Sanctions List** (U.S. Treasury)")
    st.markdown("- **EU Sanctions Database**")
    st.markdown("- **UN Security Council Sanctions**")
    st.markdown("- **UK FCDO Sanctions List**")
    st.markdown("")
    st.markdown("These APIs provide real-time screening against sanctioned entities, individuals, and jurisdictions.")

with tabs[2]:
    st.subheader("📜 Event History")
    st.info("Placeholder: show recent shocks, policy actions, and adjudication trace entries.")

with tabs[3]:
    st.subheader("🧪 Batch Evaluation")
    st.info("Placeholder: run batch scenarios/policies/actors for comparison. (Hook point for your evaluation runner.)")"""
with open("pages/21_Real_World_Data_Metrics.py", "w") as f:
    f.write(real_world_metrics_code)

instructions_code = r"""import streamlit as st

st.set_page_config(page_title="INSTRUCTIONS and COGNITIVE SCIENCE MECHANICS", page_icon="📘", layout="wide", initial_sidebar_state="collapsed")

if not st.session_state.get("authenticated", False):
    st.warning("Please log in first.")
    st.switch_page("app.py")

st.header("📘 INSTRUCTIONS and COGNITIVE SCIENCE MECHANICS")

with st.expander("📘 Phase 3: Real-World Data-Driven Simulation Mechanics (Behind the Scenes)", expanded=True):
    st.markdown('\n### Phase 3: Real-World Data-Driven Simulation Mechanics (Behind the Scenes)\n\nAuracelle Charlie uses the **Evans-AGPO-HT** hierarchical theory as a *cognitive decision-science scaffold* to translate\nreal-world evidence streams (World Bank, export controls/sanctions screening, SIPRI uploads) into **stress-testing signals**\nfor policy governance decisions.\n\n#### 1) Hierarchical factor model (three strata)\nObserved governance performance is modeled as a hierarchical linear structure:\n\n- **Y** = observed governance performance on a task (per actor, round, and capability)\n- **g-GWC** = general governance wargaming capacity (Stratum III)\n- **BGC** = broad governance capability (Stratum II; 7 capabilities)\n- **NOF** = narrow operational factor (Stratum I; measurable sub-factors)\n- **Λ** = factor loadings; **ε** = measurement error\n\nConceptually:\n- Stratum III (**g-GWC**) explains overall “wargaming capacity”.\n- Stratum II (**BGC**) decomposes that capacity into broad competencies.\n- Stratum I (**NOF**) ties each competency to measurable operational signals (including real-world data).\n\n#### 2) The seven Broad Governance Capabilities (BGC)\nThe framework defines seven broad capabilities (examples of sub-constructs shown):\n- **STI** (Strategic Threat Intelligence): PIP, GRD, APM, TLA, PDI, AIP  \n- **SAD** (Security Architecture Design): TPD, MPM, NBE, DCF, CNS  \n- **ESI** (Exploratory Simulation Intelligence): DRS, CD, WIS, TRS, AGM, DBP  \n- **NDM** (Negotiation Dynamics Modeling): ABN, MTD, TMA, CFD, CMD, MCC  \n- **SRA** (Strategic Rationality Assessment): TAC, RLA, TPI, PRM, LTR  \n- **IIC** (Institutional Implementation Capacity): BTA, AMD, CTM, ELC, NES  \n- **ASI** (Adaptive Scalability Intelligence): CGA, RAC, VMD, KCS, FLI, CLP\n\n#### 3) Aggregation of capability into g-GWC\ng-GWC is computed as a weighted integration of BGCs (optionally multiplicative for synergy), and can include\nAI-acceleration factors (e.g., computational simulation speed, neural network integration, RL optimization).\n\n#### 4) Network effects in multi-agent governance\nInfluence propagation is modeled over a governance influence network:\n- next_state ≈ (1 − α)·current_state + α·W·current_state + interventions  \nwhere **W** is a weighted adjacency matrix (influence edges). This connects directly to the **3-D Influence Map**.\n\n#### 5) How this becomes “cognitive decision science” in Charlie\nIn Charlie, the math is used to:\n- **Explain** why an actor’s position is plausible given evidence (traceability).\n- **Adjudicate** between competing claims (agentic adjudicator + deception signals).\n- **Stress-test** policies across rounds by injecting shocks and measuring capability shifts and convergence/divergence.\n')

with st.expander("📘 Phase 3 Instructions - Player Walkthrough (Policy Stress-Testing War Game)", expanded=True):
    st.markdown('\n### Phase 3 Instructions - Player Walkthrough (Policy Stress-Testing War Game)\n\n1. **Select policy instrument to stress-test** (e.g., AI Act / GDPR variants).  \n2. **Pick two actors** and the **role lenses** you want to compare (institutional incentives).  \n3. **Choose who you represent** (your decision lens).  \n4. **Run rounds**: each round is a decision cycle—positions, adjudication, deception detection, and evidence-linked impacts.  \n5. **Observe outputs**:\n   - **Agentic adjudicator status**: neutral synthesis of signals, flags, and disagreements.\n   - **Deception detection**: consistency checks vs. historical actions, power/incentive mismatch, and evidence divergence.\n   - **Policy position comparison**: side-by-side evidence-linked posture and capability context.\n   - **Strategic analysis**: why the sandbox recommends a posture and what assumptions drive it.\n6. **Iterate**: adjust assumptions, inject shocks, and compare trajectories across actors and roles.\n')
"""
with open("pages/91_INSTRUCTIONS_and_COGNITIVE_SCIENCE_MECHANICS.py", "w") as f:
    f.write(instructions_code)

print("✅ Added pages: Real-World Data Metrics + INSTRUCTIONS")



# ========================================
# CELL 10: Create Red Team Module Page (Sidebar-clickable)
# ========================================
red_team_code = '''import streamlit as st

# --- Dropdown reference links (curated) ---
def _render_links(title, items):
    exp = st.expander(title, expanded=False)
    for label, url in items:
        exp.markdown(f"- [{label}]({url})")

_PROVENANCE = [
    ("World Bank Indicators API", "https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation"),
    ("IMF World Economic Outlook Database", "https://www.imf.org/en/Publications/WEO/weo-database"),
    ("UN Comtrade+", "https://comtradeplus.un.org/"),
    ("SIPRI Databases", "https://www.sipri.org/databases"),
    ("WTO Tariff Analysis Online", "https://tao.wto.org/"),
]

_COGNITION_FIELDS = [
    ("Bounded rationality (Stanford Encyclopedia of Philosophy)", "https://plato.stanford.edu/entries/bounded-rationality/"),
    ("OODA loop (overview)", "https://en.wikipedia.org/wiki/OODA_loop"),
    ("Sensemaking (overview)", "https://en.wikipedia.org/wiki/Sensemaking"),
]

_BELIEF_FIELDS = [
    ("Bayesian inference (overview)", "https://en.wikipedia.org/wiki/Bayesian_inference"),
    ("Belief state (POMDP concept)", "https://en.wikipedia.org/wiki/Partially_observable_Markov_decision_process"),
]

_COGNITION_METRICS = [
    ("Calibration (statistics)", "https://en.wikipedia.org/wiki/Calibration_(statistics)"),
    ("Brier score", "https://en.wikipedia.org/wiki/Brier_score"),
    ("Entropy (information theory)", "https://en.wikipedia.org/wiki/Entropy_(information_theory)"),
]

import pandas as pd
import numpy as np
import json

st.set_page_config(page_title="Red Team Module", layout="wide")

st.title("🛡️ Red Team Module — Foresight Cognition & Belief Distortion")
st.caption("Stress-test how actors perceive, misperceive, and act on futures by attacking signals, framing, or cognition parameters.")

# -----------------------------
# Helpers
# -----------------------------
def clip01(x: float) -> float:
    return float(max(0.0, min(1.0, x)))

def init_agents(names):
    agents = {}
    for n in names:
        agents[n] = {
            "cognition": {
                "H": 3,          # Horizon depth (1..K)
                "Omega": 0.55,   # Openness/update rate (0..1)
                "Lambda": 0.55,  # Uncertainty tolerance (0..1)
                "Pi": 0.45       # Narrative lock-in (0..1)
            },
            "belief": {
                "mu": 0.50,      # Expected outcome proxy (0..1)
                "sigma": 0.25    # Uncertainty proxy (0..1)
            },
            "metrics": {
                "USI": None,     # Update suppression index
                "HD": 0          # Horizon degradation
            }
        }
    return agents

def compute_alpha(cog):
    # alpha = Omega * (1 - Pi)
    return clip01(cog["Omega"] * (1.0 - cog["Pi"]))

def update_belief(agent, evidence):
    """
    Cognition-weighted belief update (lightweight, explainable).
    evidence: float in [-1, 1] where + pushes mu upward, - downward
    """
    cog = agent["cognition"]
    bel = agent["belief"]

    alpha = compute_alpha(cog)

    # Map evidence to [0,1] target for mu
    target = clip01(0.5 + 0.5 * float(evidence))

    bel["mu"] = clip01((1 - alpha) * bel["mu"] + alpha * target)

    # Uncertainty update: low Lambda collapses uncertainty faster; high Lambda stays more agnostic
    collapse = (1.0 - cog["Lambda"]) * 0.20
    bel["sigma"] = clip01(bel["sigma"] * (1.0 - collapse))

    # Metrics
    agent["metrics"]["USI"] = clip01(1.0 - alpha)

def apply_red_team_move(agent, move, intensity, K=5):
    cog = agent["cognition"]
    bel = agent["belief"]

    if move == "Horizon Collapse":
        old = cog["H"]
        cog["H"] = int(max(1, cog["H"] - int(round(intensity * 2))))
        agent["metrics"]["HD"] += (old - cog["H"])

    elif move == "Narrative Entrenchment":
        cog["Pi"] = clip01(cog["Pi"] + 0.35 * intensity)

    elif move == "Epistemic Distrust":
        cog["Omega"] = clip01(cog["Omega"] - 0.35 * intensity)

    elif move == "Panic Amplification":
        cog["Lambda"] = clip01(cog["Lambda"] - 0.35 * intensity)

    elif move == "Metric Spoofing":
        # Push mu in a misleading direction without improving the agent's cognition (signal tampering)
        bel["mu"] = clip01(bel["mu"] + (0.25 * intensity))

    elif move == "Frame Flip":
        # Invert the meaning of recent evidence by effectively flipping openness for one step
        cog["Omega"] = clip01(1.0 - cog["Omega"])

    else:
        pass

# -----------------------------
# State bootstrap
# -----------------------------
# Try to reuse existing Simulation state if present; otherwise initialize safely
if "charlie_agents" not in st.session_state:
    # Prefer any actor list from Simulation (if you stored it), else default to your Charlie set
    default_actors = st.session_state.get("actor_list") or [
        "Dubai", "UK", "US", "Japan", "China", "Brazil", "India", "NATO",
        "Israel", "Paraguay", "Belgium", "Denmark", "Ukraine", "Serbia",
        "Argentina", "Norway", "Switzerland", "Poland", "Global South"
    ]
    st.session_state["charlie_agents"] = init_agents(default_actors)

agents = st.session_state["charlie_agents"]
actor_names = list(agents.keys())

# -----------------------------
# Sidebar controls (clickable module config)
# -----------------------------
st.sidebar.header("Red Team Controls")
target = st.sidebar.selectbox("Target actor", actor_names, index=0)
move = st.sidebar.selectbox(
    "Belief-distortion move",
    [
        "Narrative Entrenchment",
        "Epistemic Distrust",
        "Horizon Collapse",
        "Panic Amplification",
        "Metric Spoofing",
        "Frame Flip",
    ],
    index=0
)
intensity = st.sidebar.slider("Intensity", 0.0, 1.0, 0.5, 0.05)

evidence = st.sidebar.slider("New evidence signal (for belief update)", -1.0, 1.0, 0.2, 0.05)
apply_move = st.sidebar.button("Apply Red Team Move", use_container_width=True)
apply_update = st.sidebar.button("Run Cognition-Weighted Belief Update", use_container_width=True)
reset = st.sidebar.button("Reset module state", use_container_width=True)

if reset:
    st.session_state["charlie_agents"] = init_agents(actor_names)
    agents = st.session_state["charlie_agents"]
    st.success("Reset Red Team module state.")

# -----------------------------
# Execute actions
# -----------------------------
if apply_move:
    apply_red_team_move(agents[target], move, intensity)
    st.success(f"Applied **{move}** to **{target}** (intensity={intensity:.2f}).")

if apply_update:
    update_belief(agents[target], evidence)
    st.success(f"Updated beliefs for **{target}** using evidence={evidence:.2f}.")

# -----------------------------
# Display
# -----------------------------
st.markdown(
    """
<style>
/* reduce top padding and tighten layout a bit */
section.main > div { padding-top: 1rem; }
/* tighten spacing above the dataframe */
div[data-testid="stDataFrame"] { margin-top: 0.25rem; }
</style>
""",
    unsafe_allow_html=True,
)

col1, col2 = st.columns([1, 1], gap="large")

with col1:
    st.subheader("Target actor — cognition & belief")
    st.write(f"**Actor:** {target}")

    cog = agents[target]["cognition"]
    bel = agents[target]["belief"]
    met = agents[target]["metrics"]

    st.markdown("**Cognition state (C) — real-world fields**")
    st.markdown("""
- **H** (Planning Horizon): how many turns ahead the actor considers
- **Omega** (Update Openness): willingness to revise beliefs when new evidence arrives
- **Lambda** (Uncertainty Tolerance): comfort operating under ambiguity and incomplete information
- **Pi** (Narrative Commitment): how strongly the actor is locked into a doctrine/storyline
""")
    st.code(json.dumps(cog, indent=2), language="json")

    st.markdown("**Belief state (b) — real-world fields**")
    st.markdown("""
- **mu** (Expected Outcome Score): 0.0 to 1.0 proxy for how well things are going
- **sigma** (Uncertainty Level): 0.0 to 1.0 proxy for how unsure the actor is
""")
    st.code(json.dumps(bel, indent=2), language="json")

    st.markdown("**Cognition metrics**")
    st.markdown("""
- **USI** (Update Suppression Index): higher means less learning / more "stuck"
- **HD** (Horizon Degradation): how much the planning horizon has been reduced by attacks
""")
    st.code(json.dumps(met, indent=2), language="json")

with col2:
    st.subheader("Data & evidence provenance (what feeds the sandbox)")
    st.info(
        "Auracelle Charlie is designed to be data-driven and evidence-based. "
        "This Red Team module manipulates cognition variables, but it is intended to sit on top of the "
        "real-world data streams used in the Simulation page."
    )

    st.markdown("**Current data resources (in this baseline):**")
    st.markdown("""
- **World Bank (wbgapi)**: macro indicators (GDP, population, etc.) pulled via API
- **U.S. Consolidated Screening List (CSL)**: export-control / sanctions screening via API wrapper
- **SIPRI**: military expenditure reference data (CSV upload)

For assurance, add an Evidence Ledger export that records each metric with dataset name, vintage/date, access method, and transformation.
""")

    st.markdown("**Recommended NATO-grade traceability upgrades:**")
    st.markdown("""
- Attach a **Source** tag to every metric (dataset name, vintage/date, access method)
- Attach a **Transformation** tag (how the raw data is normalized/scored)
- Provide an **Export** button for an **Evidence Ledger** (metric → source → transformation → timestamp/round)
""")

st.subheader("All actors — cognition & belief table")
rows = []
for name, a in agents.items():
    c = a["cognition"]
    b = a["belief"]
    m = a["metrics"]
    alpha = compute_alpha(c)
    rows.append({
        "Actor": name,
        "Planning horizon (H)": c["H"],
        "Evidence update openness (Omega)": round(c["Omega"], 3),
        "Uncertainty tolerance (Lambda)": round(c["Lambda"], 3),
        "Narrative lock-in (Pi)": round(c["Pi"], 3),
        "Learning rate (alpha)": round(alpha, 3),
        "Expected outcome (mu)": round(b["mu"], 3),
        "Uncertainty (sigma)": round(b["sigma"], 3),
        "Update suppression (USI)": None if m["USI"] is None else round(m["USI"], 3),
        "Horizon degradation (HD)": m["HD"],
    })
df = pd.DataFrame(rows).sort_values("Actor")
st.dataframe(df, use_container_width=True, hide_index=True)

st.divider()
st.markdown(
    """
**How to use this module**
- Use **Apply Red Team Move** to distort cognition (H, Omega, Lambda, Pi) or distort signals (Metric Spoofing / Frame Flip).
- Use **Run Cognition-Weighted Belief Update** to see how the target updates beliefs under its current cognition state.
- The key diagnostic is **USI (Update Suppression Index)**: higher = more “stuck” / less learning.
"""
)

st.markdown("## References (dropdown links)")
_render_links("Data & evidence provenance (what feeds the sandbox)", _PROVENANCE)
_render_links("Cognition state (C) — real-world fields", _COGNITION_FIELDS)
_render_links("Belief state (b) — real-world fields", _BELIEF_FIELDS)
_render_links("Cognition metrics", _COGNITION_METRICS)
'''

with open('pages/3_Red_Team_Module.py', 'w') as f:
    f.write(red_team_code)

print("✅ Full simulation page with API integration created")

# ========================================
# CELL 10: Launch Application
# ========================================
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"]
)
time.sleep(5)

ngrok_process = subprocess.Popen(
    ["ngrok", "http", "--domain=julene-untaxable-raelene.ngrok-free.dev", "8501"]
)

public_url = "https://aiwargame.ngrok.app"
print(f"\n🚀 Auracelle Charlie 3 - War Gaming Stress-Testing Policy Governance Research Simulation/Prototype with Full API Integration is LIVE!")
print(f"🔗 Access at: {public_url}")
print(f"\n✨ Features:")
print("  • 🤖 AI Agentic Adjudicator")
print("  • 🌍 World Bank API (GDP, Military, Internet)")
print("  • 🚫 US Export Controls API (Sanctions)")
print("  • 💥 External Shock System")
print("  • 🎭 Deception Detection")
print("  • 📊 Real-World Data Dashboard")
print("  • 📈 Economic & Military Analysis")
print("  • 📥 Export Reports (Markdown & CSV)")
print(f"\n🔑 Password: charlie2025")
print(f"\n📊 Data Sources:")
print("  - World Bank: Automatic")
print("  - Export Controls: Automatic")
print("  - SIPRI: Upload CSV in sidebar")

display(HTML(f'<a href="{public_url}" target="_blank" style="font-size:20px; font-weight:bold; color:#0066cc;">🔗 Launch Auracelle Charlie - Live 2025</a>'))

# ========================================
# ADD: Research Store (SQLite) + Session Setup + Admin Dashboard
# ========================================
research_store_code = r'''from __future__ import annotations
import sqlite3, time, json
from typing import Any, Dict

DB_PATH = "auracelle_research.db"

def _conn() -> sqlite3.Connection:
    c = sqlite3.connect(DB_PATH, check_same_thread=False)
    c.execute("PRAGMA journal_mode=WAL;")
    c.execute("PRAGMA foreign_keys=ON;")
    return c

def init_db() -> None:
    con = _conn()
    cur = con.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS sessions (
        session_id TEXT PRIMARY KEY,
        created_at INTEGER NOT NULL,
        scenario TEXT,
        condition_tag TEXT,
        started_at INTEGER,
        ended_at INTEGER,
        completed INTEGER DEFAULT 0
    );
    """)
    cur.execute("""
    CREATE TABLE IF NOT EXISTS participants (
        participant_id TEXT PRIMARY KEY,
        session_id TEXT NOT NULL,
        created_at INTEGER NOT NULL,
        alias TEXT,
        consent INTEGER DEFAULT 0,
        gender TEXT,
        sector TEXT,
        military_status TEXT,
        role_function TEXT,
        years_experience TEXT,
        wargame_experience TEXT,
        ai_gov_familiarity TEXT,
        age_band TEXT,
        education_band TEXT,
        region TEXT,
        FOREIGN KEY(session_id) REFERENCES sessions(session_id) ON DELETE CASCADE
    );
    """)
    cur.execute("""
    CREATE TABLE IF NOT EXISTS moves (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        session_id TEXT NOT NULL,
        participant_id TEXT,
        t INTEGER NOT NULL,
        round_num INTEGER,
        policy TEXT,
        action TEXT,
        notes TEXT,
        state_json TEXT,
        FOREIGN KEY(session_id) REFERENCES sessions(session_id) ON DELETE CASCADE
    );
    """)
    cur.execute("""
    CREATE TABLE IF NOT EXISTS outcomes (
        session_id TEXT PRIMARY KEY,
        t INTEGER NOT NULL,
        trust REAL,
        compliance REAL,
        alignment REAL,
        resilience REAL,
        unintended_json TEXT,
        FOREIGN KEY(session_id) REFERENCES sessions(session_id) ON DELETE CASCADE
    );
    """)
    con.commit()
    con.close()

def upsert_session(session_id: str, scenario: str | None = None, condition_tag: str | None = None) -> None:
    con = _conn()
    t = int(time.time())
    con.execute("""
      INSERT INTO sessions(session_id, created_at, scenario, condition_tag)
      VALUES(?,?,?,?)
      ON CONFLICT(session_id) DO UPDATE SET scenario=excluded.scenario, condition_tag=excluded.condition_tag
    """, (session_id, t, scenario, condition_tag))
    con.commit()
    con.close()

def upsert_participant(participant_id: str, session_id: str, profile: Dict[str, Any]) -> None:
    con = _conn()
    t = int(time.time())
    con.execute("""
      INSERT INTO participants(participant_id, session_id, created_at, alias, consent, gender, sector, military_status,
                               role_function, years_experience, wargame_experience, ai_gov_familiarity,
                               age_band, education_band, region)
      VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
      ON CONFLICT(participant_id) DO UPDATE SET
        alias=excluded.alias, consent=excluded.consent, gender=excluded.gender, sector=excluded.sector,
        military_status=excluded.military_status, role_function=excluded.role_function,
        years_experience=excluded.years_experience, wargame_experience=excluded.wargame_experience,
        ai_gov_familiarity=excluded.ai_gov_familiarity, age_band=excluded.age_band,
        education_band=excluded.education_band, region=excluded.region
    """, (
        participant_id, session_id, t,
        profile.get("alias"), int(bool(profile.get("consent"))), profile.get("gender"), profile.get("sector"), profile.get("military_status"),
        profile.get("role_function"), profile.get("years_experience"), profile.get("wargame_experience"), profile.get("ai_gov_familiarity"),
        profile.get("age_band"), profile.get("education_band"), profile.get("region")
    ))
    con.commit()
    con.close()

def log_move(session_id: str, participant_id: str | None, round_num: int | None,
             policy: str | None, action: str | None, notes: str | None,
             state: Dict[str, Any] | None = None) -> None:
    con = _conn()
    t = int(time.time())
    con.execute("""
      INSERT INTO moves(session_id, participant_id, t, round_num, policy, action, notes, state_json)
      VALUES(?,?,?,?,?,?,?,?)
    """, (session_id, participant_id, t, round_num, policy, action, notes, json.dumps(state or {})))
    con.commit()
    con.close()

def set_outcomes(session_id: str, trust: float, compliance: float, alignment: float, resilience: float,
                 unintended: Dict[str, Any] | None = None) -> None:
    con = _conn()
    t = int(time.time())
    con.execute("""
      INSERT INTO outcomes(session_id, t, trust, compliance, alignment, resilience, unintended_json)
      VALUES(?,?,?,?,?,?,?)
      ON CONFLICT(session_id) DO UPDATE SET
        t=excluded.t, trust=excluded.trust, compliance=excluded.compliance, alignment=excluded.alignment,
        resilience=excluded.resilience, unintended_json=excluded.unintended_json
    """, (session_id, t, trust, compliance, alignment, resilience, json.dumps(unintended or {})))
    con.commit()
    con.close()
'''
with open("agpo_data/research_store.py", "w") as f:
    f.write(research_store_code)

session_setup_code = r'''import streamlit as st
import uuid
from agpo_data.research_store import init_db, upsert_session, upsert_participant

st.set_page_config(page_title="Auracelle Charlie 3 - Session Setup", layout="wide", initial_sidebar_state="collapsed")
st.markdown("<style>div.block-container{padding-top:0.6rem;}</style>", unsafe_allow_html=True)

init_db()

if "session_id" not in st.session_state:
    st.session_state["session_id"] = str(uuid.uuid4())
if "participant_id" not in st.session_state:
    st.session_state["participant_id"] = str(uuid.uuid4())

st.title("🔬 Session Setup")

with st.expander("Research Notice & Consent", expanded=True):
    st.write(
        "Auracelle Charlie is a strategic cognition and policy stress-testing simulator. "
        "For research analysis, we record anonymized interaction data, policy selections, and aggregate indicators "
        "(e.g., trust, compliance, alignment, resilience). You may select 'Prefer not to say' where available."
    )
    consent = st.checkbox("I consent to participate in this research session (required).", value=False)

st.subheader("Participant Profile")

col1, col2, col3 = st.columns(3)
with col1:
    gender = st.selectbox("Gender *", ["Female","Male","Non-binary","Prefer to self-describe","Prefer not to say"], index=4)
    sector = st.selectbox("Sector / Affiliation *", ["Military","Government","Industry","Academia","Civil Society","Other","Prefer not to say"], index=6)
    military_status = st.selectbox("Military status *", ["Active Duty","Veteran","Civilian","Prefer not to say"], index=3)
with col2:
    role_function = st.selectbox("Role function *", ["Policy","Technical","Legal","Operations","Leadership","Research","Other","Prefer not to say"], index=7)
    years_experience = st.selectbox("Years in field *", ["0–2","3–5","6–10","10+","Prefer not to say"], index=4)
    wargame_experience = st.selectbox("Wargaming / simulation experience *", ["None","Some","Frequent","Prefer not to say"], index=3)
with col3:
    ai_gov_familiarity = st.selectbox("AI governance familiarity *", ["Novice","Intermediate","Advanced","Prefer not to say"], index=3)
    age_band = st.selectbox("Age band (optional)", ["18–24","25–34","35–44","45–54","55+","Prefer not to say"], index=5)
    region = st.text_input("Region / Country (optional)", value="")

required_ok = all([consent, gender, sector, military_status, role_function, years_experience, wargame_experience, ai_gov_familiarity])

if st.button("Enter Simulation", disabled=not required_ok, use_container_width=True):
    st.session_state["setup_complete"] = True
    st.session_state["consent"] = True
    st.session_state["condition_tag"] = st.session_state.get("condition_tag","unassigned")
    upsert_session(st.session_state["session_id"], scenario=st.session_state.get("scenario"), condition_tag=st.session_state.get("condition_tag"))
    upsert_participant(
        participant_id=st.session_state["participant_id"],
        session_id=st.session_state["session_id"],
        profile={
            "alias": st.session_state.get("username",""),
            "consent": True,
            "gender": gender,
            "sector": sector,
            "military_status": military_status,
            "role_function": role_function,
            "years_experience": years_experience,
            "wargame_experience": wargame_experience,
            "ai_gov_familiarity": ai_gov_familiarity,
            "age_band": age_band,
            "region": region.strip() if region else None,
        }
    )
    st.switch_page("pages/2_Simulation.py")
'''
with open("pages/1_Session_Setup.py","w") as f:
    f.write(session_setup_code)

admin_code = r'''import streamlit as st
import os
import sqlite3
import pandas as pd
from agpo_data.research_store import DB_PATH, init_db

st.set_page_config(page_title="Auracelle Charlie 3 - Admin Dashboard", layout="wide", initial_sidebar_state="collapsed")
st.markdown("<style>div.block-container{padding-top:0.6rem;}</style>", unsafe_allow_html=True)

init_db()

st.title("🛠️ Administrative Dashboard")

with st.sidebar:
    st.subheader("Access Control")
    pin = st.text_input("Admin PIN", type="password")
    try:
        ADMIN_PIN = st.secrets.get("ADMIN_PIN", None)
    except Exception:
        ADMIN_PIN = None
    if not ADMIN_PIN:
        ADMIN_PIN = os.environ.get("ADMIN_PIN", "1234")
    authed = (pin == ADMIN_PIN)

if not authed:
    st.info("Enter the Admin PIN in the sidebar to view research stats.")
    st.stop()

con = sqlite3.connect(DB_PATH, check_same_thread=False)
participants = pd.read_sql_query("SELECT * FROM participants", con)
sessions = pd.read_sql_query("SELECT * FROM sessions", con)
moves = pd.read_sql_query("SELECT * FROM moves", con)
outcomes = pd.read_sql_query("SELECT * FROM outcomes", con)
con.close()

c1,c2,c3,c4 = st.columns(4)
c1.metric("Participants", len(participants))
c2.metric("Sessions", len(sessions))
c3.metric("Moves logged", len(moves))
c4.metric("Outcomes recorded", len(outcomes))

st.divider()
st.subheader("Composition Overview (RQ3)")
colA,colB,colC = st.columns(3)
with colA:
    st.write("Gender")
    st.dataframe(participants["gender"].value_counts(dropna=False).rename_axis("gender").reset_index(name="count"))
with colB:
    st.write("Sector")
    st.dataframe(participants["sector"].value_counts(dropna=False).rename_axis("sector").reset_index(name="count"))
with colC:
    st.write("Military status")
    st.dataframe(participants["military_status"].value_counts(dropna=False).rename_axis("military_status").reset_index(name="count"))

st.divider()
st.subheader("Governance Indicators (RQ4)")
if len(outcomes)==0:
    st.warning("No outcomes recorded yet. Ensure the simulation writes outcomes to the outcomes table.")
else:
    st.dataframe(outcomes.sort_values("t", ascending=False))

st.divider()
st.subheader("Export Data")
st.download_button("Download participants.csv", participants.to_csv(index=False), file_name="participants.csv")
st.download_button("Download sessions.csv", sessions.to_csv(index=False), file_name="sessions.csv")
st.download_button("Download moves.csv", moves.to_csv(index=False), file_name="moves.csv")
st.download_button("Download outcomes.csv", outcomes.to_csv(index=False), file_name="outcomes.csv")
'''
with open("pages/99_Admin_Dashboard.py","w") as f:
    f.write(admin_code)

print("✅ Added research_store + Session Setup + Admin Dashboard")


^C
^C
✅ Setup complete
✅ AGPO Data Package created
✅ Enhanced Adjudicator with API integration created
✅ Login page created
✅ Added pages: Real-World Data Metrics + INSTRUCTIONS
✅ Full simulation page with API integration created

🚀 Auracelle Charlie 3 - War Gaming Stress-Testing Policy Governance Research Simulation/Prototype with Full API Integration is LIVE!
🔗 Access at: https://aiwargame.ngrok.app

✨ Features:
  • 🤖 AI Agentic Adjudicator
  • 🌍 World Bank API (GDP, Military, Internet)
  • 🚫 US Export Controls API (Sanctions)
  • 💥 External Shock System
  • 🎭 Deception Detection
  • 📊 Real-World Data Dashboard
  • 📈 Economic & Military Analysis
  • 📥 Export Reports (Markdown & CSV)

🔑 Password: charlie2025

📊 Data Sources:
  - World Bank: Automatic
  - Export Controls: Automatic
  - SIPRI: Upload CSV in sidebar


✅ Added research_store + Session Setup + Admin Dashboard


In [3]:
# ========================================
# CREATE 3D INFLUENCE MAP PAGE
# ========================================
influence_map_code = '''
import streamlit as st
import streamlit.components.v1 as components
import pandas as pd

st.set_page_config(page_title="3D Influence Map", page_icon="🧬", layout="wide")

# Password protection
auth_ok = any(bool(st.session_state.get(k, False)) for k in ("authenticated","logged_in","is_authenticated"))
if not auth_ok:
    st.warning("Please log in first (Simulation -> Login).")
    st.stop()

# =============================================================================
# PAGE HEADER
# =============================================================================

st.title("🧬 3D AlphaFold-Style Influence Map")

with st.expander("ℹ️ About this visualization — click to expand", expanded=False):
    st.markdown("""
    <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 16px 20px; border-radius: 10px; margin-bottom: 14px;'>
        <h4 style='color: white; margin: 0 0 4px 0;'>External Policy Pressures & Internal Cultural Forces</h4>
        <p style='color: rgba(255,255,255,0.85); margin: 0; font-size: 13px;'>
            Interactive 3D Visualization of AI Governance Influences
        </p>
    </div>
    """, unsafe_allow_html=True)
    st.markdown("""
Unlike traditional network visualizations that show static relationships, this **AlphaFold-style influence map**
reveals how **external policy pressures** (like GDPR, export controls) and **internal cultural forces**
(like democratic values, tech nationalism) shape countries' AI governance positions in 3D space.

**The 3D Space:**
- **X-axis**: Economic strength (GDP)
- **Y-axis**: Influence score (0-1)
- **Z-axis**: Policy position alignment

**Each node represents:**
- Countries (US, EU, China, UK, Japan, Dubai, etc.)
- Organizations (NATO)

**Visualization Elements:**
- 🔵 **Blue Arrows** = External policy pressures (GDPR, sanctions, export controls)
- 🟣 **Purple Spheres** = Internal cultural forces (democratic norms, tech nationalism)
- 📊 **Node Size** = Relative influence
- 🎨 **Node Color** = Alignment clusters

**Interactive Features:**
- Rotate: Click and drag
- Zoom: Scroll
- Animate: Watch policy convergence over time
    """)

# =============================================================================
# DATA DEFINITIONS (same as original)
# =============================================================================

default_data = {
    "European Union": {"gdp": 15.0, "influence": 0.90, "position": "Strict data protection (GDPR)", "mil_exp": 1.5, "internet": 89.0, "cultural_alignment": "Western"},
    "Dubai": {"gdp": 0.5, "influence": 0.7, "position": "Moderate regulatory stance", "mil_exp": 5.6, "internet": 99.0, "cultural_alignment": "Western-Middle East hybrid"},
    "United Kingdom": {"gdp": 3.2, "influence": 0.85, "position": "Supports EU-style data protection", "mil_exp": 2.2, "internet": 96.0, "cultural_alignment": "Western"},
    "United States": {"gdp": 21.0, "influence": 0.95, "position": "Favors innovation over regulation", "mil_exp": 3.4, "internet": 92.0, "cultural_alignment": "Western"},
    "Japan": {"gdp": 5.1, "influence": 0.88, "position": "Pro-regulation for trust", "mil_exp": 1.0, "internet": 95.0, "cultural_alignment": "Eastern-Western hybrid"},
    "China": {"gdp": 17.7, "influence": 0.93, "position": "Strict state-driven AI governance", "mil_exp": 1.7, "internet": 73.0, "cultural_alignment": "Eastern"},
    "Brazil": {"gdp": 2.0, "influence": 0.75, "position": "Leaning toward EU-style regulation", "mil_exp": 1.4, "internet": 81.0, "cultural_alignment": "Latin American"},
    "India": {"gdp": 3.7, "influence": 0.82, "position": "Strategic tech balancing", "mil_exp": 2.4, "internet": 43.0, "cultural_alignment": "South Asian"},
    "Russia": {"gdp": 1.8, "influence": 0.78, "position": "Sovereign tech control", "mil_exp": 4.3, "internet": 85.0, "cultural_alignment": "Eastern"},
    "Iraq": {"gdp": 0.2, "influence": 0.42, "position": "Developing governance framework", "mil_exp": 3.5, "internet": 49.0, "cultural_alignment": "Middle East"},
    "Qatar": {"gdp": 0.18, "influence": 0.68, "position": "Tech-forward with state oversight", "mil_exp": 3.7, "internet": 99.0, "cultural_alignment": "Middle East"},
    "NATO": {"gdp": 25.0, "influence": 0.97, "position": "Collective security & data interoperability", "mil_exp": 2.5, "internet": 90.0, "cultural_alignment": "Western Alliance"},
    "Greenland": {"gdp": 0.003, "influence": 0.45, "position": "Emerging Arctic tech governance", "mil_exp": 0.0, "internet": 68.0, "cultural_alignment": "Nordic"},
    "Venezuela": {"gdp": 0.048, "influence": 0.58, "position": "State-controlled digital infrastructure", "mil_exp": 0.9, "internet": 72.0, "cultural_alignment": "Latin American"},

    "Israel": {"gdp": 0.5, "influence": 0.75, "position": "Cyber & security-forward; high-tech economy", "mil_exp": 5.0, "internet": 88.0, "cultural_alignment": "Middle East / Western-linked"},
    "Paraguay": {"gdp": 0.04, "influence": 0.35, "position": "Developing market; regulatory capacity building", "mil_exp": 1.2, "internet": 65.0, "cultural_alignment": "Latin American"},
    "Belgium": {"gdp": 0.6, "influence": 0.60, "position": "EU/NATO hub; compliance-aligned", "mil_exp": 1.1, "internet": 92.0, "cultural_alignment": "Western (EU)"},
    "Denmark": {"gdp": 0.4, "influence": 0.55, "position": "High-trust governance; strong digital state", "mil_exp": 1.7, "internet": 98.0, "cultural_alignment": "Nordic/Western"},
    "Ukraine": {"gdp": 0.2, "influence": 0.65, "position": "Conflict resilience; reconstruction & security alignment", "mil_exp": 15.0, "internet": 76.0, "cultural_alignment": "Eastern Europe / Western-aligned"},
    "Serbia": {"gdp": 0.07, "influence": 0.40, "position": "Non-aligned balancing; regional interoperability", "mil_exp": 2.0, "internet": 80.0, "cultural_alignment": "Balkan / mixed"},
    "Argentina": {"gdp": 0.6, "influence": 0.50, "position": "Emerging market; institutional volatility risk", "mil_exp": 0.7, "internet": 88.0, "cultural_alignment": "Latin American"},
    "Norway": {"gdp": 0.5, "influence": 0.55, "position": "Energy wealth; NATO-aligned; high trust", "mil_exp": 1.6, "internet": 99.0, "cultural_alignment": "Nordic/Western"},
    "Switzerland": {"gdp": 0.8, "influence": 0.60, "position": "Neutral hub; high compliance; finance-centric", "mil_exp": 0.7, "internet": 97.0, "cultural_alignment": "Western (neutral)"},
    "Poland": {"gdp": 0.9, "influence": 0.60, "position": "NATO frontline; rapid defense modernization", "mil_exp": 3.5, "internet": 90.0, "cultural_alignment": "Eastern Europe / Western-aligned"},
    "Global South": {"gdp": 30.0, "influence": 0.80, "position": "Plural bloc; sovereignty & development-focused governance", "mil_exp": 2.0, "internet": 60.0, "cultural_alignment": "Multi-regional"}
}

EXTERNAL_INFLUENCES = {
    "GDPR": {"type": "regulation", "strength": 0.9, "targets": ["European Union", "United Kingdom", "Brazil"]},
    "US Export Controls": {"type": "policy", "strength": 0.85, "targets": ["China", "Russia", "Iraq"]},
    "Belt & Road Initiative": {"type": "economic", "strength": 0.75, "targets": ["Brazil", "Qatar", "Dubai"]},
    "AUKUS Agreement": {"type": "alliance", "strength": 0.8, "targets": ["United States", "United Kingdom", "Japan"]},
    "UN AI Ethics": {"type": "norm", "strength": 0.6, "targets": ["India", "Brazil", "NATO"]}
}

INTERNAL_INFLUENCES = {
    "Democratic Norms": {"strength": 0.85, "countries": ["United States", "European Union", "United Kingdom", "Japan", "India", "Brazil"]},
    "Tech Nationalism": {"strength": 0.9, "countries": ["China", "Russia", "United States"]},
    "Post-Colonial Sovereignty": {"strength": 0.7, "countries": ["India", "Brazil", "Iraq", "Qatar"]},
    "Energy Wealth": {"strength": 0.75, "countries": ["Russia", "Qatar", "Dubai", "Venezuela"]},
    "Military-Tech Integration": {"strength": 0.8, "countries": ["United States", "China", "Russia", "NATO"]}
}

# =============================================================================
# EMBED THE 3D ANIMATED VISUALIZATION
# =============================================================================

html_code = """

<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            background: #0a0e27;
            color: #e0e7ff;
            overflow: hidden;
        }
        /* ── MAXIMIZE / RESTORE BUTTON ── */
        #maxBtn {
            position: fixed;
            top: 10px;
            left: 50%;
            transform: translateX(-50%);
            z-index: 9999;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 20px;
            padding: 6px 18px;
            font-size: 0.8rem;
            font-weight: 700;
            cursor: pointer;
            letter-spacing: 0.04em;
            box-shadow: 0 2px 10px rgba(102,126,234,0.5);
            transition: all 0.2s;
        }
        #maxBtn:hover { box-shadow: 0 4px 16px rgba(102,126,234,0.7); transform: translateX(-50%) translateY(-1px); }
        /* MAXIMIZED state – full viewport */
        body.maximized { overflow: hidden; }
        body.maximized #wrapper {
            position: fixed !important;
            inset: 0 !important;
            z-index: 9998;
            width: 100vw !important;
            height: 100vh !important;
        }
        /* ── LAYOUT ── */
        #wrapper {
            display: flex;
            width: 100%;
            height: 800px;
            background: #0a0e27;
            transition: all 0.35s cubic-bezier(0.4,0,0.2,1);
        }
        /* LEFT SIDEBAR */
        #sidebar {
            width: 280px;
            min-width: 280px;
            background: rgba(26,31,58,0.95);
            border-right: 1px solid rgba(102,126,234,0.2);
            display: flex;
            flex-direction: column;
            overflow-y: auto;
            transition: width 0.3s, min-width 0.3s;
        }
        #sidebar.collapsed { width: 0; min-width: 0; overflow: hidden; }
        #sidebarToggle {
            position: absolute;
            left: 270px;
            top: 50%;
            transform: translateY(-50%);
            z-index: 100;
            background: rgba(102,126,234,0.7);
            border: none;
            color: white;
            width: 18px;
            height: 40px;
            border-radius: 0 6px 6px 0;
            cursor: pointer;
            font-size: 0.65rem;
            transition: left 0.3s;
        }
        #sidebar.collapsed ~ #sidebarToggle { left: 0; }
        .sidebar-section {
            padding: 1rem;
            border-bottom: 1px solid rgba(102,126,234,0.1);
        }
        .sidebar-section h3 {
            font-size: 0.75rem;
            text-transform: uppercase;
            letter-spacing: 0.06em;
            color: #818cf8;
            margin-bottom: 0.75rem;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }
        .ctrl-group { margin-bottom: 0.8rem; }
        .ctrl-group label {
            display: flex;
            justify-content: space-between;
            font-size: 0.78rem;
            margin-bottom: 0.4rem;
            color: #c7d2fe;
        }
        .val { color: #818cf8; font-weight: 600; }
        input[type=range] {
            width: 100%;
            height: 5px;
            border-radius: 3px;
            background: linear-gradient(90deg, #4c1d95 0%, #818cf8 100%);
            outline: none;
            -webkit-appearance: none;
        }
        input[type=range]::-webkit-slider-thumb {
            -webkit-appearance: none;
            width: 14px; height: 14px;
            border-radius: 50%;
            background: #818cf8;
            cursor: pointer;
            box-shadow: 0 0 8px rgba(129,140,248,0.5);
        }
        .btn {
            width: 100%;
            padding: 0.6rem;
            margin-bottom: 0.4rem;
            border: none;
            border-radius: 7px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            font-weight: 600;
            font-size: 0.82rem;
            cursor: pointer;
            transition: transform 0.15s, box-shadow 0.15s;
        }
        .btn:hover { transform: translateY(-1px); box-shadow: 0 3px 10px rgba(102,126,234,0.4); }
        .btn.sec {
            background: rgba(129,140,248,0.15);
            border: 1px solid #818cf8;
        }
        .toggle-btn {
            width: 100%;
            padding: 0.5rem;
            margin-bottom: 0.3rem;
            border: 1px solid rgba(129,140,248,0.3);
            border-radius: 5px;
            background: rgba(129,140,248,0.08);
            color: #c7d2fe;
            font-size: 0.78rem;
            cursor: pointer;
            transition: all 0.15s;
        }
        .toggle-btn.active {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border-color: transparent;
        }
        /* ── COUNTRY FOCUS SELECT ── */
        #focusSelect {
            width: 100%;
            background: rgba(26,31,58,0.9);
            border: 1px solid rgba(129,140,248,0.3);
            color: #c7d2fe;
            padding: 0.5rem;
            border-radius: 6px;
            font-size: 0.8rem;
            margin-bottom: 0.5rem;
        }
        #focusSelect option { background: #1a1f3a; }
        /* ── FOCUS DETAIL CARD ── */
        #focusCard {
            background: rgba(102,126,234,0.08);
            border: 1px solid rgba(102,126,234,0.2);
            border-radius: 8px;
            padding: 0.75rem;
            font-size: 0.75rem;
            line-height: 1.6;
            display: none;
        }
        #focusCard.visible { display: block; }
        #focusCard .fc-title {
            font-size: 0.9rem;
            font-weight: 700;
            color: #818cf8;
            margin-bottom: 0.5rem;
        }
        #focusCard .fc-row { display: flex; justify-content: space-between; padding: 2px 0; }
        #focusCard .fc-key { color: #94a3b8; }
        #focusCard .fc-val { color: #e0e7ff; font-weight: 600; }
        #focusCard .fc-pills { margin-top: 0.5rem; display: flex; flex-wrap: wrap; gap: 4px; }
        #focusCard .pill {
            padding: 2px 8px;
            border-radius: 20px;
            font-size: 0.68rem;
            font-weight: 600;
        }
        .pill-ext { background: rgba(102,126,234,0.25); color: #818cf8; border: 1px solid #667eea; }
        .pill-int { background: rgba(168,85,247,0.2); color: #c084fc; border: 1px solid #a855f7; }
        .pill-hot { background: rgba(249,115,22,0.25); color: #fb923c; border: 1px solid #f97316; animation: pulse 1.2s infinite; }
        @keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.5} }
        /* ── LIVE OVERLAY PANEL ── */
        #livePanel {
            background: rgba(26,31,58,0.9);
            border: 1px solid rgba(249,115,22,0.4);
            border-radius: 8px;
            padding: 0.75rem;
            font-size: 0.75rem;
        }
        .live-badge {
            display: inline-block;
            background: #f97316;
            color: white;
            font-size: 0.6rem;
            font-weight: 700;
            padding: 1px 6px;
            border-radius: 20px;
            letter-spacing: 0.05em;
            animation: pulse 1.5s infinite;
        }
        .live-metric { display: flex; justify-content: space-between; padding: 3px 0; }
        .live-key { color: #94a3b8; font-size: 0.72rem; }
        .live-val { font-weight: 700; font-size: 0.78rem; }
        .tension-high { color: #ef4444; }
        .tension-mid  { color: #f59e0b; }
        .tension-low  { color: #10b981; }
        .align-high   { color: #10b981; }
        .align-low    { color: #f59e0b; }
        .round-chip {
            background: rgba(102,126,234,0.2);
            border: 1px solid #667eea;
            border-radius: 4px;
            padding: 2px 6px;
            color: #818cf8;
            font-size: 0.7rem;
        }
        /* ── MAIN CANVAS AREA ── */
        #vizArea {
            flex: 1;
            position: relative;
            background: #0a0e27;
            overflow: hidden;
        }
        canvas#mainCanvas { width: 100%; height: 100%; display: block; }
        /* ── STATS OVERLAY (top-right) ── */
        #statsOverlay {
            position: absolute;
            top: 0.75rem;
            right: 0.75rem;
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 0.5rem;
            pointer-events: none;
        }
        .stat-chip {
            background: rgba(26,31,58,0.92);
            padding: 0.5rem 0.75rem;
            border-radius: 7px;
            border: 1px solid rgba(102,126,234,0.25);
            backdrop-filter: blur(8px);
        }
        .stat-label { display: block; font-size: 0.62rem; text-transform: uppercase; letter-spacing: 0.05em; color: #64748b; }
        .stat-value { display: block; font-size: 1.1rem; font-weight: 700; color: #818cf8; }
        /* ── LEGEND (bottom-left) ── */
        #legend {
            position: absolute;
            bottom: 0.75rem;
            left: 0.75rem;
            background: rgba(26,31,58,0.92);
            padding: 0.75rem;
            border-radius: 8px;
            border: 1px solid rgba(102,126,234,0.2);
            backdrop-filter: blur(8px);
            font-size: 0.72rem;
        }
        #legend h4 { font-size: 0.75rem; margin-bottom: 0.4rem; color: #818cf8; }
        .leg-row { display: flex; align-items: center; margin-bottom: 0.3rem; }
        .leg-dot { width: 10px; height: 10px; border-radius: 50%; margin-right: 6px; }
        /* ── FOCUSED NODE RING ── */
        #focusRing {
            position: absolute;
            border: 2px solid #f97316;
            border-radius: 50%;
            pointer-events: none;
            display: none;
            box-shadow: 0 0 12px rgba(249,115,22,0.6);
            transition: all 0.2s;
        }
        /* ── STRESS TICKER ── */
        #stressTicker {
            position: absolute;
            bottom: 0.75rem;
            right: 0.75rem;
            background: rgba(26,31,58,0.92);
            border: 1px solid rgba(249,115,22,0.3);
            border-radius: 8px;
            padding: 0.5rem 0.75rem;
            font-size: 0.72rem;
            display: none;
        }
        #stressTicker.visible { display: block; }
        .ticker-title { color: #f97316; font-weight: 700; font-size: 0.7rem; text-transform: uppercase; margin-bottom: 3px; }
        /* ── TOOLTIP ── */
        #tooltip {
            position: absolute;
            background: rgba(10,14,39,0.97);
            border: 1px solid rgba(102,126,234,0.4);
            border-radius: 8px;
            padding: 0.6rem 0.9rem;
            font-size: 0.75rem;
            pointer-events: none;
            display: none;
            max-width: 220px;
            backdrop-filter: blur(10px);
            z-index: 50;
        }
        #tooltip .tt-name { font-weight: 700; color: #818cf8; margin-bottom: 4px; }
        #tooltip .tt-row  { display: flex; justify-content: space-between; gap: 12px; color: #94a3b8; }
        #tooltip .tt-row span:last-child { color: #e0e7ff; font-weight: 600; }
    </style>
</head>
<body>
<button id="maxBtn" onclick="toggleMaximize()">⛶ MAXIMIZE</button>
<div id="wrapper">
  <!-- LEFT SIDEBAR -->
  <div id="sidebar">
    <!-- INFLUENCE PARAMETERS -->
    <div class="sidebar-section">
      <h3>Influence Parameters</h3>
      <div class="ctrl-group">
        <label>External Pressure <span class="val" id="ext_val">70%</span></label>
        <input type="range" id="external_pressure" min="0" max="100" value="70" oninput="updateParams()">
      </div>
      <div class="ctrl-group">
        <label>Internal Forces <span class="val" id="int_val">60%</span></label>
        <input type="range" id="internal_forces" min="0" max="100" value="60" oninput="updateParams()">
      </div>
      <div class="ctrl-group">
        <label>Policy Convergence <span class="val" id="conv_val">45%</span></label>
        <input type="range" id="convergence" min="0" max="100" value="45" oninput="updateParams()">
      </div>
      <div class="ctrl-group">
        <label>Time Evolution <span class="val" id="time_val">0 mo</span></label>
        <input type="range" id="time" min="0" max="36" value="0" oninput="updateParams()">
      </div>
    </div>
    <!-- VIZ TOGGLES -->
    <div class="sidebar-section">
      <h3>Visualization Options</h3>
      <button class="toggle-btn active" id="show_arrows" onclick="toggleArrows()">🔵 Policy Pressures</button>
      <button class="toggle-btn active" id="show_spheres" onclick="toggleSpheres()">🟣 Cultural Forces</button>
      <button class="toggle-btn active" id="show_connections" onclick="toggleConnections()">🔗 Alignments</button>
      <button class="toggle-btn active" id="show_labels" onclick="toggleLabels()">🏷️ Labels</button>
    </div>
    <!-- ANIMATION -->
    <div class="sidebar-section">
      <h3>Animation</h3>
      <button class="btn" onclick="toggleAnimation()"><span id="anim_text">▶ Start Animation</span></button>
      <button class="btn sec" onclick="resetView()">↺ Reset View</button>
    </div>
    <!-- COUNTRY FOCUS DRILL-DOWN -->
    <div class="sidebar-section">
      <h3>🔍 Country Focus</h3>
      <select id="focusSelect" onchange="focusCountry(this.value)">
        <option value="">— Select country —</option>
        <option value="US">United States</option>
        <option value="EU">European Union</option>
        <option value="CN">China</option>
        <option value="UK">United Kingdom</option>
        <option value="JP">Japan</option>
        <option value="IN">India</option>
        <option value="BR">Brazil</option>
        <option value="RU">Russia</option>
        <option value="NATO">NATO</option>
        <option value="Dubai">Dubai</option>
        <option value="Qatar">Qatar</option>
        <option value="Iraq">Iraq</option>
        <option value="Greenland">Greenland</option>
        <option value="Venezuela">Venezuela</option>
        <option value="Israel">Israel</option>
        <option value="Paraguay">Paraguay</option>
        <option value="Belgium">Belgium</option>
        <option value="Denmark">Denmark</option>
        <option value="Ukraine">Ukraine</option>
        <option value="Serbia">Serbia</option>
        <option value="Argentina">Argentina</option>
        <option value="Norway">Norway</option>
        <option value="Switzerland">Switzerland</option>
        <option value="Poland">Poland</option>
        <option value="Global South">Global South</option>
      </select>
      <div id="focusCard"></div>
      <button class="btn sec" id="clearFocusBtn" style="display:none;margin-top:0.4rem" onclick="clearFocus()">✕ Clear Focus</button>
    </div>
    <!-- LIVE STRESS OVERLAY -->
    <div class="sidebar-section">
      <h3>⚡ Live Stress Test <span class="live-badge" id="liveBadge">LIVE</span></h3>
      <div id="livePanel">
        <div class="live-metric"><span class="live-key">Active Actors</span><span class="live-val" id="lv_actors">—</span></div>
        <div class="live-metric"><span class="live-key">Round</span><span class="live-val" id="lv_round">—</span></div>
        <div class="live-metric"><span class="live-key">Reward</span><span class="live-val" id="lv_reward">—</span></div>
        <div class="live-metric"><span class="live-key">Risk</span><span class="live-val" id="lv_risk">—</span></div>
        <div class="live-metric"><span class="live-key">Tension</span><span class="live-val" id="lv_tension">—</span></div>
        <div class="live-metric"><span class="live-key">Alignment</span><span class="live-val" id="lv_align">—</span></div>
        <div class="live-metric"><span class="live-key">Confidence</span><span class="live-val" id="lv_conf">—</span></div>
      </div>
      <button class="btn sec" style="margin-top:0.5rem;font-size:0.75rem" onclick="injectTestData()">🧪 Inject Test Data</button>
    </div>
    <!-- INFO -->
    <div class="sidebar-section">
      <h3>Information</h3>
      <div style="font-size:0.72rem;color:#94a3b8;line-height:1.6">
        <p><strong>Drag:</strong> Rotate scene</p>
        <p><strong>Scroll:</strong> Zoom</p>
        <p><strong>Click node:</strong> Focus country</p>
        <p><strong>Hover node:</strong> Tooltip details</p>
        <p><strong>⛶ Maximize:</strong> Full-screen 3D view</p>
      </div>
    </div>
  </div>
  <!-- SIDEBAR COLLAPSE TOGGLE -->
  <button id="sidebarToggle" onclick="toggleSidebar()">◀</button>

  <!-- MAIN VIZ AREA -->
  <div id="vizArea">
    <canvas id="mainCanvas"></canvas>
    <!-- Stats overlay -->
    <div id="statsOverlay">
      <div class="stat-chip"><span class="stat-label">Countries</span><span class="stat-value" id="country_count">25</span></div>
      <div class="stat-chip"><span class="stat-label">Influences</span><span class="stat-value" id="influence_count">12</span></div>
      <div class="stat-chip"><span class="stat-label">Alignment</span><span class="stat-value" id="alignment_score">45%</span></div>
      <div class="stat-chip"><span class="stat-label">Clusters</span><span class="stat-value" id="cluster_count">4</span></div>
    </div>
    <!-- Legend -->
    <div id="legend">
      <h4>Legend</h4>
      <div class="leg-row"><div class="leg-dot" style="background:#667eea"></div><span>Western-aligned</span></div>
      <div class="leg-row"><div class="leg-dot" style="background:#ef4444"></div><span>State-controlled</span></div>
      <div class="leg-row"><div class="leg-dot" style="background:#f59e0b"></div><span>Hybrid approach</span></div>
      <div class="leg-row"><div class="leg-dot" style="background:#10b981"></div><span>Regional/Developing</span></div>
      <div class="leg-row"><div class="leg-dot" style="background:#f97316;animation:pulse 1.2s infinite"></div><span>🔥 Stress-test active</span></div>
    </div>
    <!-- Stress ticker -->
    <div id="stressTicker">
      <div class="ticker-title">⚡ Stress Pulse</div>
      <div id="tickerBody">—</div>
    </div>
    <!-- Tooltip -->
    <div id="tooltip"></div>
    <!-- Focus ring (DOM overlay, hidden by default) -->
    <div id="focusRing"></div>
  </div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>
// ════════════════════════════════════════════════════════════
// DATA
// ════════════════════════════════════════════════════════════
const countries = [
    {id:'US',       gdp:21.0,  inf:0.95, pos:[0.80,0.90,0.70], c:0x667eea, align:'Western',
     fullName:'United States',  position:'Favors innovation over regulation',          milExp:3.4,  internet:92,  cultural:'Western'},
    {id:'EU',       gdp:15.0,  inf:0.90, pos:[0.90,0.85,0.90], c:0x667eea, align:'Western',
     fullName:'European Union',  position:'Strict data protection (GDPR)',              milExp:1.5,  internet:89,  cultural:'Western'},
    {id:'CN',       gdp:17.7,  inf:0.93, pos:[0.20,0.90,0.30], c:0xef4444, align:'State',
     fullName:'China',           position:'Strict state-driven AI governance',          milExp:1.7,  internet:73,  cultural:'Eastern'},
    {id:'UK',       gdp:3.2,   inf:0.85, pos:[0.85,0.80,0.85], c:0x667eea, align:'Western',
     fullName:'United Kingdom',  position:'Supports EU-style data protection',          milExp:2.2,  internet:96,  cultural:'Western'},
    {id:'JP',       gdp:5.1,   inf:0.88, pos:[0.70,0.85,0.80], c:0xf59e0b, align:'Hybrid',
     fullName:'Japan',           position:'Pro-regulation for trust',                   milExp:1.0,  internet:95,  cultural:'E-W hybrid'},
    {id:'IN',       gdp:3.7,   inf:0.82, pos:[0.60,0.80,0.60], c:0xf59e0b, align:'Hybrid',
     fullName:'India',           position:'Strategic tech balancing',                   milExp:2.4,  internet:43,  cultural:'South Asian'},
    {id:'BR',       gdp:2.0,   inf:0.75, pos:[0.70,0.70,0.65], c:0x10b981, align:'Regional',
     fullName:'Brazil',          position:'Leaning toward EU-style regulation',         milExp:1.4,  internet:81,  cultural:'Latin American'},
    {id:'RU',       gdp:1.8,   inf:0.78, pos:[0.30,0.75,0.40], c:0xef4444, align:'State',
     fullName:'Russia',          position:'Sovereign tech control',                     milExp:4.3,  internet:85,  cultural:'Eastern'},
    {id:'NATO',     gdp:25.0,  inf:0.97, pos:[0.85,0.95,0.85], c:0x667eea, align:'Western',
     fullName:'NATO',            position:'Collective security & data interoperability',milExp:2.5,  internet:90,  cultural:'W. Alliance'},
    {id:'Dubai',    gdp:0.5,   inf:0.70, pos:[0.65,0.65,0.60], c:0xf59e0b, align:'Hybrid',
     fullName:'Dubai (UAE)',     position:'Moderate regulatory stance',                 milExp:5.6,  internet:99,  cultural:'W-ME hybrid'},
    {id:'Qatar',    gdp:0.18,  inf:0.68, pos:[0.60,0.65,0.55], c:0x10b981, align:'Regional',
     fullName:'Qatar',           position:'Tech-forward with state oversight',          milExp:3.7,  internet:99,  cultural:'Middle East'},
    {id:'Iraq',     gdp:0.20,  inf:0.42, pos:[0.40,0.40,0.45], c:0x10b981, align:'Regional',
     fullName:'Iraq',            position:'Developing governance framework',            milExp:3.5,  internet:49,  cultural:'Middle East'},
    {id:'Greenland',gdp:0.003, inf:0.45, pos:[0.50,0.40,0.50], c:0x10b981, align:'Regional',
     fullName:'Greenland',       position:'Emerging Arctic tech governance',            milExp:0.0,  internet:68,  cultural:'Nordic'},
    {id:'Venezuela',gdp:0.048, inf:0.58, pos:[0.35,0.55,0.40], c:0xef4444, align:'State',
     fullName:'Venezuela',       position:'State-controlled digital infrastructure',    milExp:0.9,  internet:72,  cultural:'Latin American'},
    {id:'Israel',   gdp:0.52,  inf:0.82, pos:[0.78,0.80,0.76], c:0x667eea, align:'Western',
     fullName:'Israel',          position:'Cyber & security-forward; high-tech economy',milExp:5.0, internet:88,  cultural:'ME/W-linked'},
    {id:'Paraguay', gdp:0.04,  inf:0.62, pos:[0.62,0.60,0.58], c:0x10b981, align:'Regional',
     fullName:'Paraguay',        position:'Developing market; regulatory capacity building',milExp:1.2,internet:65,cultural:'Latin American'},
    {id:'Belgium',  gdp:0.58,  inf:0.86, pos:[0.88,0.84,0.88], c:0x667eea, align:'Western',
     fullName:'Belgium',         position:'EU/NATO hub; compliance-aligned',            milExp:1.1,  internet:92,  cultural:'Western (EU)'},
    {id:'Denmark',  gdp:0.41,  inf:0.87, pos:[0.87,0.86,0.89], c:0x667eea, align:'Western',
     fullName:'Denmark',         position:'High-trust governance; strong digital state', milExp:1.7, internet:98,  cultural:'Nordic/Western'},
    {id:'Ukraine',  gdp:0.18,  inf:0.70, pos:[0.80,0.72,0.72], c:0x667eea, align:'Western',
     fullName:'Ukraine',         position:'Conflict resilience; reconstruction & security',milExp:15.0,internet:76,cultural:'E-EU/W-aligned'},
    {id:'Serbia',   gdp:0.07,  inf:0.68, pos:[0.75,0.70,0.70], c:0xf59e0b, align:'Hybrid',
     fullName:'Serbia',          position:'Non-aligned balancing; regional interop',    milExp:2.0,  internet:80,  cultural:'Balkan/mixed'},
    {id:'Argentina',gdp:0.64,  inf:0.66, pos:[0.64,0.62,0.60], c:0x10b981, align:'Regional',
     fullName:'Argentina',       position:'Emerging market; institutional volatility',  milExp:0.7,  internet:88,  cultural:'Latin American'},
    {id:'Norway',   gdp:0.48,  inf:0.88, pos:[0.86,0.88,0.90], c:0x667eea, align:'Western',
     fullName:'Norway',          position:'Energy wealth; NATO-aligned; high trust',    milExp:1.6,  internet:99,  cultural:'Nordic/Western'},
    {id:'Switzerland',gdp:0.88,inf:0.89, pos:[0.89,0.87,0.92], c:0x667eea, align:'Western',
     fullName:'Switzerland',     position:'Neutral hub; high compliance; finance-centric',milExp:0.7,internet:97, cultural:'Western (neutral)'},
    {id:'Poland',   gdp:0.80,  inf:0.84, pos:[0.83,0.78,0.80], c:0x667eea, align:'Western',
     fullName:'Poland',          position:'NATO frontline; rapid defense modernization', milExp:3.5, internet:90,  cultural:'E-EU/W-aligned'},
    {id:'Global South',gdp:35.0,inf:0.65,pos:[0.55,0.60,0.55],c:0x10b981, align:'Regional',
     fullName:'Global South',    position:'Plural bloc; sovereignty & development-focused',milExp:2.0,internet:60,cultural:'Multi-regional'}
];

const policyArrows = [
    {name:'GDPR',         targets:['EU','UK','BR','Belgium','Denmark','Norway','Switzerland','Poland'],  strength:0.9,  color:0x667eea},
    {name:'US Export',    targets:['CN','RU','Iraq','Serbia'],                                          strength:0.85, color:0xef4444},
    {name:'Belt&Road',    targets:['BR','Qatar','Dubai','Argentina','Serbia'],                          strength:0.75, color:0xfbbf24},
    {name:'AUKUS',        targets:['US','UK','JP'],                                                     strength:0.80, color:0x667eea},
    {name:'UN Ethics',    targets:['IN','BR','NATO','Global South','Paraguay'],                         strength:0.60, color:0x10b981},
    {name:'NATO Expand',  targets:['Ukraine','Poland','Norway','Denmark','Belgium'],                    strength:0.88, color:0x818cf8},
    {name:'Conflict Zone',targets:['Ukraine','Israel','Iraq'],                                          strength:0.82, color:0xf97316}
];

const culturalSpheres = [
    {name:'Democratic',    countries:['US','EU','UK','JP','IN','BR','Belgium','Denmark','Norway','Switzerland','Poland','Ukraine'], strength:0.85, color:0x667eea},
    {name:'TechNat',       countries:['CN','RU','US','Israel'],                                                                    strength:0.90, color:0xef4444},
    {name:'PostColonial',  countries:['IN','BR','Iraq','Qatar','Paraguay','Argentina','Global South'],                             strength:0.70, color:0x10b981},
    {name:'Energy',        countries:['RU','Qatar','Dubai','Venezuela','Norway'],                                                  strength:0.75, color:0xfbbf24},
    {name:'MilTech',       countries:['US','CN','RU','NATO','Israel','Ukraine','Poland'],                                          strength:0.80, color:0xa855f7},
    {name:'NeutralHub',    countries:['Switzerland','Serbia'],                                                                     strength:0.65, color:0x94a3b8}
];

// ════════════════════════════════════════════════════════════
// SCENE STATE
// ════════════════════════════════════════════════════════════
let scene, camera, renderer;
let nodes=[], arrows=[], spheres=[], connections=[], labels=[];
let isAnimating=false;
let showArrows=true, showSpheres=true, showConnections=true, showLabels=true;
let focusedId=null, sidebarOpen=true;
let liveData={};  // injected from Streamlit via postMessage or injected directly

// ════════════════════════════════════════════════════════════
// MAXIMIZE / MINIMIZE
// ════════════════════════════════════════════════════════════
function toggleMaximize() {
    const body = document.body;
    const btn = document.getElementById('maxBtn');
    const maximized = body.classList.toggle('maximized');
    btn.textContent = maximized ? '⛶ RESTORE' : '⛶ MAXIMIZE';
    setTimeout(() => {
        const canvas = document.getElementById('mainCanvas');
        if (camera && renderer) {
            camera.aspect = canvas.clientWidth / canvas.clientHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(canvas.clientWidth, canvas.clientHeight);
        }
    }, 370);
}

// ════════════════════════════════════════════════════════════
// SIDEBAR TOGGLE
// ════════════════════════════════════════════════════════════
function toggleSidebar() {
    const sb = document.getElementById('sidebar');
    const btn = document.getElementById('sidebarToggle');
    sidebarOpen = !sidebarOpen;
    sb.classList.toggle('collapsed', !sidebarOpen);
    btn.textContent = sidebarOpen ? '◀' : '▶';
    btn.style.left = sidebarOpen ? '270px' : '0px';
    setTimeout(() => {
        const canvas = document.getElementById('mainCanvas');
        if (camera && renderer) {
            camera.aspect = canvas.clientWidth / canvas.clientHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(canvas.clientWidth, canvas.clientHeight);
        }
    }, 330);
}

// ════════════════════════════════════════════════════════════
// INIT THREE.JS
// ════════════════════════════════════════════════════════════
function init() {
    const canvas = document.getElementById('mainCanvas');
    const w = canvas.clientWidth, h = canvas.clientHeight;

    scene = new THREE.Scene();
    scene.background = new THREE.Color(0x0a0e27);

    camera = new THREE.PerspectiveCamera(60, w/h, 0.1, 1000);
    camera.position.set(25, 20, 25);
    camera.lookAt(0,0,0);

    renderer = new THREE.WebGLRenderer({canvas, antialias:true});
    renderer.setSize(w, h);
    renderer.setPixelRatio(window.devicePixelRatio);

    scene.add(new THREE.AmbientLight(0xffffff, 0.5));
    const p1 = new THREE.PointLight(0x667eea, 1, 100);
    p1.position.set(20,20,20); scene.add(p1);
    const p2 = new THREE.PointLight(0x764ba2, 0.8, 100);
    p2.position.set(-20,-20,-20); scene.add(p2);

    createNodes();
    createPolicyArrows();
    createCulturalSpheres();
    createConnections();
    setupMouseControls(canvas);
    animate();
}

// ════════════════════════════════════════════════════════════
// NODE CREATION
// ════════════════════════════════════════════════════════════
function createNodes() {
    countries.forEach((c,i) => {
        const sz = 0.3 + c.inf * 0.5;
        const geo = new THREE.SphereGeometry(sz, 32, 32);
        const mat = new THREE.MeshPhongMaterial({color:c.c, emissive:c.c, emissiveIntensity:0.3, shininess:30});
        const mesh = new THREE.Mesh(geo, mat);
        const x=(c.pos[0]-0.5)*30, y=(c.pos[1]-0.5)*25, z=(c.pos[2]-0.5)*30;
        mesh.position.set(x,y,z);
        mesh.userData = {countryId: c.id};
        scene.add(mesh);

        const label = makeLabel(c.id);
        label.position.set(x, y+sz+0.8, z);
        scene.add(label);

        nodes.push({mesh, label, country:c, initialPos:mesh.position.clone(), baseColor:c.c});
        labels.push(label);
    });
}

function makeLabel(text) {
    const cv = document.createElement('canvas');
    cv.width=256; cv.height=128;
    const ctx = cv.getContext('2d');
    ctx.fillStyle='rgba(0,0,0,0.4)';
    ctx.roundRect(10,20,236,80,12);
    ctx.fill();
    ctx.fillStyle='#ffffff';
    ctx.font='Bold 42px Arial';
    ctx.textAlign='center';
    ctx.fillText(text, 128, 78);
    const tex = new THREE.Texture(cv);
    tex.needsUpdate=true;
    const sp = new THREE.Sprite(new THREE.SpriteMaterial({map:tex, transparent:true}));
    sp.scale.set(3.2, 1.6, 1);
    return sp;
}

// ════════════════════════════════════════════════════════════
// ARROWS & SPHERES & CONNECTIONS
// ════════════════════════════════════════════════════════════
function createPolicyArrows() {
    policyArrows.forEach(policy => {
        policy.targets.forEach(tid => {
            const tn = nodes.find(n=>n.country.id===tid);
            if (!tn) return;
            const origin = new THREE.Vector3(0,10,0);
            const dir = new THREE.Vector3().subVectors(tn.mesh.position, origin);
            const len = dir.length();
            const ah = new THREE.ArrowHelper(dir.normalize(), origin, len, policy.color, len*0.2, len*0.1);
            ah.userData = {type:'arrow', policy};
            scene.add(ah);
            arrows.push(ah);
        });
    });
}

function createCulturalSpheres() {
    culturalSpheres.forEach(sph => {
        sph.countries.forEach(cid => {
            const nd = nodes.find(n=>n.country.id===cid);
            if (!nd) return;
            const geo = new THREE.SphereGeometry(2.5, 32, 32);
            const mat = new THREE.MeshBasicMaterial({color:sph.color, transparent:true, opacity:0.1, wireframe:true});
            const mesh = new THREE.Mesh(geo, mat);
            mesh.position.copy(nd.mesh.position);
            mesh.userData = {type:'cultural', sphere:sph};
            scene.add(mesh);
            spheres.push(mesh);
        });
    });
}

function createConnections() {
    for (let i=0; i<nodes.length; i++) {
        for (let j=i+1; j<nodes.length; j++) {
            const d = nodes[i].mesh.position.distanceTo(nodes[j].mesh.position);
            if (d < 12) {
                const mat = new THREE.LineBasicMaterial({color:0x667eea, transparent:true, opacity:Math.max(0.08, 1-d/15)});
                const geo = new THREE.BufferGeometry().setFromPoints([nodes[i].mesh.position, nodes[j].mesh.position]);
                const line = new THREE.Line(geo, mat);
                scene.add(line);
                connections.push(line);
            }
        }
    }
}

// ════════════════════════════════════════════════════════════
// MOUSE CONTROLS
// ════════════════════════════════════════════════════════════
function setupMouseControls(canvas) {
    let dragging=false, prev={x:0,y:0};
    const raycaster = new THREE.Raycaster();
    const mouse = new THREE.Vector2();

    canvas.addEventListener('mousedown', e=>{dragging=true; prev={x:e.clientX,y:e.clientY};});
    canvas.addEventListener('mouseup', ()=>{dragging=false;});
    canvas.addEventListener('mouseleave', ()=>{dragging=false; hideTooltip();});

    canvas.addEventListener('mousemove', e=>{
        if (dragging) {
            const dx=e.clientX-prev.x, dy=e.clientY-prev.y;
            camera.position.applyAxisAngle(new THREE.Vector3(0,1,0), dx*0.005);
            camera.position.applyAxisAngle(new THREE.Vector3(1,0,0), dy*0.005);
            camera.lookAt(0,0,0);
            prev={x:e.clientX,y:e.clientY};
        } else {
            // Hover tooltip
            const rect = canvas.getBoundingClientRect();
            mouse.x = ((e.clientX-rect.left)/rect.width)*2-1;
            mouse.y = -((e.clientY-rect.top)/rect.height)*2+1;
            raycaster.setFromCamera(mouse, camera);
            const hits = raycaster.intersectObjects(nodes.map(n=>n.mesh));
            if (hits.length>0) {
                const nd = nodes.find(n=>n.mesh===hits[0].object);
                if (nd) showTooltip(e, nd.country, e.clientX-rect.left, e.clientY-rect.top);
            } else {
                hideTooltip();
            }
        }
    });

    canvas.addEventListener('click', e=>{
        const rect = canvas.getBoundingClientRect();
        mouse.x = ((e.clientX-rect.left)/rect.width)*2-1;
        mouse.y = -((e.clientY-rect.top)/rect.height)*2+1;
        raycaster.setFromCamera(mouse, camera);
        const hits = raycaster.intersectObjects(nodes.map(n=>n.mesh));
        if (hits.length>0) {
            const nd = nodes.find(n=>n.mesh===hits[0].object);
            if (nd) {
                document.getElementById('focusSelect').value = nd.country.id;
                focusCountry(nd.country.id);
            }
        }
    });

    canvas.addEventListener('wheel', e=>{
        e.preventDefault();
        camera.position.multiplyScalar(1 + (e.deltaY>0?1:-1)*0.001*Math.abs(e.deltaY));
    }, {passive:false});
}

// ════════════════════════════════════════════════════════════
// TOOLTIP
// ════════════════════════════════════════════════════════════
function showTooltip(e, c, cx, cy) {
    const tip = document.getElementById('tooltip');
    const extPressures = policyArrows.filter(p=>p.targets.includes(c.id)).map(p=>p.name);
    const cultures = culturalSpheres.filter(s=>s.countries.includes(c.id)).map(s=>s.name);
    tip.innerHTML = `
        <div class="tt-name">${c.fullName}</div>
        <div class="tt-row"><span>Influence</span><span>${(c.inf*100).toFixed(0)}%</span></div>
        <div class="tt-row"><span>GDP</span><span>$${c.gdp}T</span></div>
        <div class="tt-row"><span>Mil Exp</span><span>${c.milExp}% GDP</span></div>
        <div class="tt-row"><span>Internet</span><span>${c.internet}%</span></div>
        <div class="tt-row"><span>Alignment</span><span>${c.align}</span></div>
        ${extPressures.length ? `<div class="tt-row"><span>Policy Pressures</span><span>${extPressures.join(', ')}</span></div>` : ''}
        ${cultures.length ? `<div class="tt-row"><span>Cultural Spheres</span><span>${cultures.join(', ')}</span></div>` : ''}
        <div class="tt-row" style="margin-top:4px;font-size:0.68rem;color:#64748b"><span colspan="2">${c.position}</span></div>
    `;
    tip.style.display = 'block';
    tip.style.left = Math.min(cx+12, 600) + 'px';
    tip.style.top  = Math.max(cy-20, 5) + 'px';
}

function hideTooltip() {
    document.getElementById('tooltip').style.display = 'none';
}

// ════════════════════════════════════════════════════════════
// COUNTRY FOCUS / DRILL-DOWN
// ════════════════════════════════════════════════════════════
const countryFullNames = {};
countries.forEach(c=>{ countryFullNames[c.id] = c.fullName; });

function focusCountry(id) {
    focusedId = id || null;
    // Reset all node emissive
    nodes.forEach(n=>{
        n.mesh.material.emissiveIntensity = 0.3;
        n.mesh.material.color.setHex(n.baseColor);
        n.mesh.material.emissive.setHex(n.baseColor);
    });

    const card = document.getElementById('focusCard');
    const clearBtn = document.getElementById('clearFocusBtn');

    if (!id) { card.classList.remove('visible'); clearBtn.style.display='none'; return; }

    // Highlight focused node
    const nd = nodes.find(n=>n.country.id===id);
    if (nd) {
        nd.mesh.material.emissiveIntensity = 0.9;
        nd.mesh.material.color.setHex(0xffffff);
        // Fly camera toward node
        const tp = nd.mesh.position;
        const offset = new THREE.Vector3(tp.x, tp.y+3, tp.z+12);
        camera.position.lerp(offset, 0.4);
        camera.lookAt(tp);
    }

    // Build detail card
    const c = countries.find(x=>x.id===id);
    if (!c) return;

    const extPressures = policyArrows.filter(p=>p.targets.includes(id));
    const cultures = culturalSpheres.filter(s=>s.countries.includes(id));

    // Check if this country is an active stress-test actor
    const isStressActor = (liveData.actorA === id || liveData.actorB === id ||
                           liveData.actorA === c.fullName || liveData.actorB === c.fullName);

    const extPills = extPressures.map(p=>
        `<span class="pill ${isStressActor&&p.strength>0.8?'pill-hot':'pill-ext'}">${p.name} (${(p.strength*100).toFixed(0)}%)</span>`
    ).join('');
    const intPills = cultures.map(s=>
        `<span class="pill pill-int">${s.name}</span>`
    ).join('');

    let stressBlock = '';
    if (isStressActor && liveData.round) {
        const isA = (liveData.actorA === id || liveData.actorA === c.fullName);
        stressBlock = `
        <div style="margin-top:0.6rem;padding-top:0.5rem;border-top:1px solid rgba(249,115,22,0.3)">
            <div style="color:#f97316;font-size:0.68rem;font-weight:700;margin-bottom:4px">⚡ STRESS TEST ACTIVE — Round ${liveData.round}</div>
            <div class="fc-row"><span class="fc-key">Role</span><span class="fc-val">${isA?'Actor A':'Actor B'}</span></div>
            <div class="fc-row"><span class="fc-key">Reward</span><span class="fc-val">${liveData.reward!==undefined?(liveData.reward*100).toFixed(1)+'%':'—'}</span></div>
            <div class="fc-row"><span class="fc-key">Risk</span><span class="fc-val ${liveData.risk>0.7?'tension-high':liveData.risk>0.4?'tension-mid':'tension-low'}">${liveData.risk!==undefined?(liveData.risk*100).toFixed(1)+'%':'—'}</span></div>
            <div class="fc-row"><span class="fc-key">Tension</span><span class="fc-val ${liveData.tension>0.7?'tension-high':liveData.tension>0.4?'tension-mid':'tension-low'}">${liveData.tension!==undefined?(liveData.tension*100).toFixed(1)+'%':'—'}</span></div>
        </div>`;
    }

    card.innerHTML = `
        <div class="fc-title">${c.fullName}</div>
        <div class="fc-row"><span class="fc-key">Influence Score</span><span class="fc-val">${(c.inf*100).toFixed(0)}%</span></div>
        <div class="fc-row"><span class="fc-key">GDP</span><span class="fc-val">$${c.gdp}T</span></div>
        <div class="fc-row"><span class="fc-key">Military Exp</span><span class="fc-val">${c.milExp}% GDP</span></div>
        <div class="fc-row"><span class="fc-key">Internet</span><span class="fc-val">${c.internet}%</span></div>
        <div class="fc-row"><span class="fc-key">Cultural Align</span><span class="fc-val">${c.cultural}</span></div>
        <div class="fc-row"><span class="fc-key">Cluster</span><span class="fc-val">${c.align}</span></div>
        <div style="margin-top:0.5rem;font-size:0.68rem;color:#94a3b8;font-style:italic">"${c.position}"</div>
        <div class="fc-pills" style="margin-top:0.6rem">
            <div style="font-size:0.67rem;color:#818cf8;width:100%;margin-bottom:2px">📋 Policy Pressures:</div>
            ${extPills || '<span style="color:#64748b;font-size:0.68rem">None identified</span>'}
        </div>
        <div class="fc-pills">
            <div style="font-size:0.67rem;color:#a855f7;width:100%;margin-bottom:2px">🌐 Cultural Spheres:</div>
            ${intPills || '<span style="color:#64748b;font-size:0.68rem">None identified</span>'}
        </div>
        ${stressBlock}
    `;
    card.classList.add('visible');
    clearBtn.style.display='block';
}

function clearFocus() {
    document.getElementById('focusSelect').value = '';
    focusCountry(null);
}

// ════════════════════════════════════════════════════════════
// LIVE STRESS TEST OVERLAY
// ════════════════════════════════════════════════════════════
function updateLivePanel(data) {
    liveData = data;

    // Update sidebar metrics
    if (data.actorA||data.actorB) {
        const aName = countryFullNames[data.actorA]||data.actorA||'—';
        const bName = countryFullNames[data.actorB]||data.actorB||'—';
        document.getElementById('lv_actors').textContent = (aName||'—') + ' vs ' + (bName||'—');
    }
    document.getElementById('lv_round').innerHTML   = data.round!==undefined ? `<span class="round-chip">R${data.round}</span>` : '—';
    document.getElementById('lv_reward').textContent  = data.reward!==undefined  ? (data.reward*100).toFixed(1)+'%' : '—';

    const tVal = data.tension!==undefined ? (data.tension*100).toFixed(1)+'%' : '—';
    const tClass = data.tension>0.7?'tension-high':data.tension>0.4?'tension-mid':'tension-low';
    document.getElementById('lv_tension').innerHTML  = `<span class="${tClass}">${tVal}</span>`;

    const rVal = data.risk!==undefined ? (data.risk*100).toFixed(1)+'%' : '—';
    const rClass = data.risk>0.7?'tension-high':data.risk>0.4?'tension-mid':'tension-low';
    document.getElementById('lv_risk').innerHTML     = `<span class="${rClass}">${rVal}</span>`;

    const aVal = data.alignment!==undefined ? (data.alignment*100).toFixed(0)+'%' : '—';
    const aClass = data.alignment>0.5?'align-high':'align-low';
    document.getElementById('lv_align').innerHTML    = `<span class="${aClass}">${aVal}</span>`;
    document.getElementById('lv_conf').textContent   = data.confidence!==undefined ? (data.confidence*100).toFixed(1)+'%' : '—';

    // Pulse stressed actors in scene
    const stressIds = [data.actorA, data.actorB].filter(Boolean);
    const tensionVal = data.tension || 0;
    nodes.forEach(n=>{
        if (stressIds.includes(n.country.id) || stressIds.includes(n.country.fullName)) {
            // Orange pulse on high tension
            if (tensionVal > 0.6) {
                n.mesh.material.emissive.setHex(0xf97316);
                n.mesh.material.emissiveIntensity = 0.7 + 0.3*Math.sin(Date.now()*0.005);
            } else {
                n.mesh.material.emissiveIntensity = 0.55;
            }
        }
    });

    // Show stress ticker
    if (data.round) {
        const ticker = document.getElementById('stressTicker');
        ticker.classList.add('visible');
        const shock = data.shockEvent ? `<div style="color:#f97316;margin-top:3px">💥 ${data.shockEvent}</div>` : '';
        document.getElementById('tickerBody').innerHTML =
            `R${data.round}: T=${tVal} | Reward=${(data.reward*100||0).toFixed(1)}%${shock}`;
    }

    // Refresh focus card if open
    if (focusedId) focusCountry(focusedId);
}

// Listen for Streamlit postMessage data bridge
window.addEventListener('message', function(event) {
    if (event.data && event.data.type === 'auracelle_stress') {
        updateLivePanel(event.data.payload);
    }
});

// ════════════════════════════════════════════════════════════
// TEST DATA INJECTOR
// ════════════════════════════════════════════════════════════
function injectTestData() {
    updateLivePanel({
        actorA: 'US', actorB: 'CN',
        round: Math.floor(Math.random()*10)+1,
        reward: Math.random(),
        risk: 0.5 + Math.random()*0.4,
        tension: 0.5 + Math.random()*0.45,
        alignment: Math.random()*0.6,
        confidence: 0.5 + Math.random()*0.4,
        shockEvent: Math.random()>0.7 ? 'cyber_attack' : null
    });
}

// ════════════════════════════════════════════════════════════
// PARAM UPDATE
// ════════════════════════════════════════════════════════════
function updateParams() {
    const ext  = parseInt(document.getElementById('external_pressure').value)/100;
    const intF = parseInt(document.getElementById('internal_forces').value)/100;
    const conv = parseInt(document.getElementById('convergence').value)/100;
    const time = parseInt(document.getElementById('time').value);

    document.getElementById('ext_val').textContent  = Math.round(ext*100)+'%';
    document.getElementById('int_val').textContent  = Math.round(intF*100)+'%';
    document.getElementById('conv_val').textContent = Math.round(conv*100)+'%';
    document.getElementById('time_val').textContent = time+' mo';

    const center = new THREE.Vector3(0,0,0);
    nodes.forEach(n=>{
        const tp = n.initialPos.clone().lerp(center, conv*0.7);
        n.mesh.position.lerp(tp, 0.1);
        n.label.position.copy(n.mesh.position);
        n.label.position.y += 1.2;
    });

    document.getElementById('alignment_score').textContent = Math.round(conv*100)+'%';
    document.getElementById('cluster_count').textContent   = Math.max(1, Math.round(5*(1-conv)));

    arrows.forEach(a=>{a.visible=showArrows; if(a.line) a.line.material.opacity=ext*0.8;});
    spheres.forEach(s=>{s.visible=showSpheres; s.material.opacity=intF*0.15;});
    updateConnections();
}

function updateConnections() {
    connections.forEach(c=>scene.remove(c));
    connections=[];
    createConnections();
    connections.forEach(c=>c.visible=showConnections);
}

// ════════════════════════════════════════════════════════════
// TOGGLE HELPERS
// ════════════════════════════════════════════════════════════
function toggleArrows()      { showArrows=!showArrows;      arrows.forEach(a=>a.visible=showArrows);      document.getElementById('show_arrows').classList.toggle('active'); }
function toggleSpheres()     { showSpheres=!showSpheres;    spheres.forEach(s=>s.visible=showSpheres);    document.getElementById('show_spheres').classList.toggle('active'); }
function toggleConnections() { showConnections=!showConnections; connections.forEach(c=>c.visible=showConnections); document.getElementById('show_connections').classList.toggle('active'); }
function toggleLabels()      { showLabels=!showLabels;      labels.forEach(l=>l.visible=showLabels);      document.getElementById('show_labels').classList.toggle('active'); }

// ════════════════════════════════════════════════════════════
// ANIMATION
// ════════════════════════════════════════════════════════════
function toggleAnimation() {
    isAnimating=!isAnimating;
    document.getElementById('anim_text').textContent = isAnimating ? '⏸ Pause' : '▶ Start Animation';
    if (isAnimating) runAnim();
}

function runAnim() {
    if (!isAnimating) return;
    const tsl = document.getElementById('time');
    const csl = document.getElementById('convergence');
    let tv=parseFloat(tsl.value), cv=parseFloat(csl.value);
    if (tv<36) {
        tv+=0.5; tsl.value=tv;
        cv=Math.min(90,cv+1); csl.value=cv;
        updateParams();
        setTimeout(runAnim,100);
    } else {
        isAnimating=false;
        document.getElementById('anim_text').textContent='▶ Start Animation';
    }
}

function resetView() {
    camera.position.set(25,20,25);
    camera.lookAt(0,0,0);
    ['external_pressure','internal_forces','convergence','time'].forEach((id,i)=>{
        document.getElementById(id).value=[70,60,45,0][i];
    });
    updateParams();
    clearFocus();
    document.getElementById('focusSelect').value='';
}

// ════════════════════════════════════════════════════════════
// RENDER LOOP
// ════════════════════════════════════════════════════════════
function animate() {
    requestAnimationFrame(animate);
    if (!isAnimating && !focusedId) {
        const t = Date.now()*0.0001;
        camera.position.x = Math.sin(t)*30;
        camera.position.z = Math.cos(t)*30;
        camera.lookAt(0,0,0);
    }
    // Pulse stressed nodes continuously
    if (liveData.tension > 0.6) {
        const stressIds = [liveData.actorA, liveData.actorB].filter(Boolean);
        nodes.forEach(n=>{
            if (stressIds.includes(n.country.id) || stressIds.includes(n.country.fullName)) {
                if (!focusedId || focusedId!==n.country.id)
                    n.mesh.material.emissiveIntensity = 0.45 + 0.35*Math.abs(Math.sin(Date.now()*0.003));
            }
        });
    }
    renderer.render(scene,camera);
}

window.addEventListener('load', init);
window.addEventListener('resize', ()=>{
    const cv = document.getElementById('mainCanvas');
    if (camera&&renderer) {
        camera.aspect = cv.clientWidth/cv.clientHeight;
        camera.updateProjectionMatrix();
        renderer.setSize(cv.clientWidth, cv.clientHeight);
    }
});
</script>
</body>
</html>
"""

# Display the visualization
components.html(html_code, height=900, scrolling=False)

# =============================================================================
# INFLUENCE ANALYSIS SECTION  (Live Stress Bridge + Per-Country Drill-Down)
# =============================================================================

st.markdown("---")
st.header("📊 Influence Analysis & Stress-Test Interpretation")

# ─── LIVE STRESS-TEST DATA BRIDGE ───────────────────────────────────────────
st.subheader("⚡ Live Stress-Test State")
st.caption("This panel mirrors the current simulation round so you can interpret what the 3D map is showing in real time.")

col_a, col_b, col_c, col_d, col_e = st.columns(5)
trace = st.session_state.get("round_metrics_trace", [])
adj   = st.session_state.get("adjudication", {}) or {}
rnd   = st.session_state.get("round", "—")
actA  = st.session_state.get("selected_country_a", "—")
actB  = st.session_state.get("selected_country_b", "—")
align = st.session_state.get("alignment_score")
reward_val = trace[-1]["reward"]  if trace else None
risk_val   = trace[-1]["risk"]    if trace else None
tension    = adj.get("tension_index")
confidence = adj.get("confidence_score")

with col_a: st.metric("Round", rnd)
with col_b: st.metric("Actors", f"{actA} vs {actB}")
with col_c: st.metric("Reward", f"{reward_val*100:.1f}%" if reward_val is not None else "—")
with col_d:
    if tension is not None:
        color = "🔴" if tension>0.7 else ("🟡" if tension>0.4 else "🟢")
        st.metric("Tension", f"{color} {tension*100:.1f}%")
    else:
        st.metric("Tension", "—")
with col_e: st.metric("Alignment", f"{align*100:.0f}%" if align is not None else "—")

if trace:
    import plotly.graph_objects as go
    rounds = [t["round"] for t in trace]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=rounds, y=[t["reward"]*100  for t in trace], name="Reward",    line=dict(color="#10b981", width=2)))
    fig.add_trace(go.Scatter(x=rounds, y=[t["risk"]*100    for t in trace], name="Risk",      line=dict(color="#ef4444", width=2)))
    fig.add_trace(go.Scatter(x=rounds, y=[t["tension"]*100 for t in trace], name="Tension",   line=dict(color="#f97316", width=2, dash="dot")))
    fig.add_trace(go.Scatter(x=rounds, y=[t["alignment"]*100 for t in trace], name="Alignment", line=dict(color="#818cf8", width=2, dash="dash")))
    fig.update_layout(
        title="Stress-Test Metrics Trace",
        xaxis_title="Round", yaxis_title="%",
        template="plotly_dark", height=280,
        margin=dict(l=30,r=20,t=40,b=30),
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    st.plotly_chart(fig, use_container_width=True)

st.markdown("---")

# ─── WHAT THE MAP ELEMENTS MEAN ─────────────────────────────────────────────
with st.expander("📖 Interpreting the Influence Map — Policy Pressures, Alignments & Cultural Forces", expanded=False):
    st.markdown("""
    ### 🔵 Policy Pressures (Blue/Coloured Arrows)
    Each arrow enters a country node from the **global policy origin point** (centre of the scene).
    The arrow colour, thickness and opacity reflect the **pressure strength** and current **external pressure slider**.

    | Arrow | Type | Meaning in the Map |
    |---|---|---|
    | **GDPR** | Regulation | Forces EU-standard data governance on targeted states |
    | **US Export Controls** | Policy | Restricts dual-use tech; creates governance isolation for targets |
    | **Belt & Road** | Economic | Creates economic dependency → soft governance alignment toward China |
    | **AUKUS** | Alliance | Deep defence-tech integration; pulls Japan/UK/US into joint posture |
    | **UN AI Ethics** | Norm | Legitimacy pressure toward inclusive, rights-based frameworks |
    | **NATO Expansion** | Alliance | Security alignment pulls EE states toward Western governance cluster |
    | **Conflict Zone** | Crisis | Distorts normal policy trajectories; elevates military spend weighting |

    **When you stress-test a policy move**, the tension and reward outputs tell you how much that move is amplifying or dampening these pressures for your two chosen actors.

    ### 🟣 Cultural Forces (Wireframe Spheres)
    Each translucent sphere around a node represents a **cultural/structural force** shaping that country's governance baseline.
    The sphere opacity tracks the **internal forces slider**.

    | Sphere | Meaning |
    |---|---|
    | **Democratic Norms** | Baseline bias toward transparency, rights, multilateral frameworks |
    | **Tech Nationalism** | Drives domestic AI stacks, data localisation, export-control logic |
    | **Post-Colonial Sovereignty** | Resistance to external norm-setting; preference for pluralism |
    | **Energy Wealth** | Resource leverage shapes negotiating posture and AI investment capacity |
    | **Military-Tech Integration** | Fuses national security and AI R&D; elevates dual-use risk |
    | **NeutralHub** | Mediator/compliance posture; avoids bloc alignment |

    ### 🔗 Alignment Connections (Lines)
    Lines connect nodes within **spatial proximity** in 3D space (distance < 12 units). Closer = more similar governance posture.
    Line opacity reflects similarity strength. During animation, convergence pulls all nodes toward centre — watch which
    countries *resist* convergence (they have strong cultural forces) vs which ones *conform quickly*.

    ### 📊 Node Properties
    - **Size** = Influence score (0–1 scale)
    - **Colour** = Alignment cluster: 🔵 Western · 🔴 State-controlled · 🟡 Hybrid · 🟢 Regional/Developing
    - **Orange pulse** = Country is currently an active stress-test actor
    - **Click any node** = Opens full drill-down in the Country Focus panel
    """)

st.markdown("---")

# ─── PER-COUNTRY DRILL-DOWN TABLE ───────────────────────────────────────────
st.subheader("🔬 Per-Country Influence Breakdown")

EXTERNAL_INFLUENCES_PY = {
    "GDPR":                {"type":"regulation", "strength":0.90, "targets":["European Union","United Kingdom","Brazil","Belgium","Denmark","Norway","Switzerland","Poland"]},
    "US Export Controls":  {"type":"policy",     "strength":0.85, "targets":["China","Russia","Iraq","Serbia"]},
    "Belt & Road":         {"type":"economic",   "strength":0.75, "targets":["Brazil","Qatar","Dubai","Argentina","Serbia"]},
    "AUKUS":               {"type":"alliance",   "strength":0.80, "targets":["United States","United Kingdom","Japan"]},
    "UN AI Ethics":        {"type":"norm",       "strength":0.60, "targets":["India","Brazil","NATO","Global South","Paraguay"]},
    "NATO Expansion":      {"type":"alliance",   "strength":0.88, "targets":["Ukraine","Poland","Norway","Denmark","Belgium"]},
    "Conflict Zone":       {"type":"crisis",     "strength":0.82, "targets":["Ukraine","Israel","Iraq"]},
}
INTERNAL_INFLUENCES_PY = {
    "Democratic Norms":        {"strength":0.85, "countries":["United States","European Union","United Kingdom","Japan","India","Brazil","Belgium","Denmark","Norway","Switzerland","Poland","Ukraine"]},
    "Tech Nationalism":        {"strength":0.90, "countries":["China","Russia","United States","Israel"]},
    "Post-Colonial Sovereignty":{"strength":0.70,"countries":["India","Brazil","Iraq","Qatar","Paraguay","Argentina","Global South"]},
    "Energy Wealth":           {"strength":0.75, "countries":["Russia","Qatar","Dubai","Venezuela","Norway"]},
    "Military-Tech Integration":{"strength":0.80,"countries":["United States","China","Russia","NATO","Israel","Ukraine","Poland"]},
    "NeutralHub":              {"strength":0.65, "countries":["Switzerland","Serbia"]},
}

country_selector = st.selectbox(
    "🔍 Select a country for detailed pressure/alignment breakdown:",
    ["— All countries —"] + list(default_data.keys())
)

influence_rows = []
for cname, cdata in default_data.items():
    ext_pressures = [(pname, pdata["strength"]) for pname, pdata in EXTERNAL_INFLUENCES_PY.items() if cname in pdata["targets"]]
    int_forces    = [(fname, fdata["strength"]) for fname, fdata in INTERNAL_INFLUENCES_PY.items() if cname in fdata["countries"]]
    total_ext_strength = sum(s for _,s in ext_pressures)
    total_int_strength = sum(s for _,s in int_forces)
    influence_rows.append({
        "Country":              cname,
        "GDP ($T)":             f"${cdata['gdp']:.3f}",
        "Influence Score":      f"{cdata['influence']*100:.0f}%",
        "Military Exp":         f"{cdata['mil_exp']:.1f}%",
        "Internet":             f"{cdata['internet']:.0f}%",
        "Ext Pressures":        ", ".join(f"{n} ({s:.0%})" for n,s in ext_pressures) or "None",
        "Ext Pressure Total":   f"{total_ext_strength:.2f}",
        "Cultural Forces":      ", ".join(f"{n}" for n,_ in int_forces) or "None",
        "Int Force Total":      f"{total_int_strength:.2f}",
        "Governance Cluster":   cdata["cultural_alignment"],
        "Policy Position":      cdata["position"],
    })

df_all = pd.DataFrame(influence_rows).sort_values("Ext Pressure Total", ascending=False)

if country_selector != "— All countries —":
    df_show = df_all[df_all["Country"] == country_selector]
    # Full detail card for selected country
    row = df_show.iloc[0] if not df_show.empty else None
    if row is not None:
        c1, c2, c3 = st.columns(3)
        with c1:
            st.markdown(f"**Policy Position**")
            st.info(row["Policy Position"])
        with c2:
            st.markdown(f"**External Policy Pressures**")
            for pname, pstr in [(p, d["strength"]) for p,d in EXTERNAL_INFLUENCES_PY.items() if country_selector in d["targets"]]:
                bar = "█" * int(pstr*10) + "░" * (10-int(pstr*10))
                clr = "🔴" if pstr>0.8 else ("🟡" if pstr>0.6 else "🟢")
                st.markdown(f"{clr} **{pname}** `{bar}` {pstr:.0%}")
        with c3:
            st.markdown(f"**Cultural Forces**")
            for fname, fstr in [(f, d["strength"]) for f,d in INTERNAL_INFLUENCES_PY.items() if country_selector in d["countries"]]:
                bar = "█" * int(fstr*10) + "░" * (10-int(fstr*10))
                clr = "🟣" if fstr>0.8 else ("🔵" if fstr>0.65 else "⚪")
                st.markdown(f"{clr} **{fname}** `{bar}` {fstr:.0%}")
        st.markdown("---")
    st.dataframe(df_show, use_container_width=True, hide_index=True)
else:
    st.dataframe(df_all, use_container_width=True, hide_index=True)

st.download_button(
    label="📥 Download Full Influence Data (CSV)",
    data=df_all.to_csv(index=False),
    file_name="auracelle_influence_map_data.csv",
    mime="text/csv"
)

if trace:
    st.markdown("---")
    st.subheader("📋 Round-by-Round Stress Log")
    df_trace = pd.DataFrame(trace)
    df_trace["reward"]     = df_trace["reward"].apply(lambda x: f"{x*100:.1f}%")
    df_trace["risk"]       = df_trace["risk"].apply(lambda x: f"{x*100:.1f}%")
    df_trace["tension"]    = df_trace["tension"].apply(lambda x: f"{x*100:.1f}%")
    df_trace["confidence"] = df_trace["confidence"].apply(lambda x: f"{x*100:.1f}%")
    df_trace["alignment"]  = df_trace["alignment"].apply(lambda x: f"{x*100:.0f}%")
    st.dataframe(df_trace, use_container_width=True, hide_index=True)
    st.download_button(
        label="📥 Download Round Log (CSV)",
        data=pd.DataFrame(trace).to_csv(index=False),
        file_name="auracelle_stress_round_log.csv",
        mime="text/csv"
    )

st.markdown("---")
st.info("💡 **Tip**: Maximize the 3D viewer (⛶ button), select a country in the left panel, then run a stress-test round — the map will pulse the active actors and update the Country Focus card in real time.")

'''

with open('pages/20_3D_Influence_Map.py', 'w') as f:
    f.write(influence_map_code)

print("✅ 3D Influence Map page created")


✅ 3D Influence Map page created


In [4]:
# ========================================
# FIX + INTEGRATE: Policy Stress Testing Platform (functional Streamlit)
# - Adds api_client.py + agpo_rl_engine.py dependencies
# - Replaces embedded HTML mock with a working Streamlit page (unique keys; no duplicate rendering)
# - Keeps Mission Console page alongside it
# ========================================

import os
from pathlib import Path

Path("pages").mkdir(parents=True, exist_ok=True)

api_client = r'''import requests

def post_metrics(actor, start_year, end_year, trade_year, policy_name=None, shock=None, intensity=None, base_url="http://127.0.0.1:8000"):
    """POST to FastAPI /v2/metrics.

    Never throws in Streamlit runtime: returns {ok: False, error: ...} on failure.
    """
    payload = {
        "actor": actor,
        "start_year": int(start_year),
        "end_year": int(end_year),
        "trade_year": int(trade_year),
        "policy_name": policy_name,
        "shock": shock,
        "intensity": intensity,
    }
    try:
        r = requests.post(f"{base_url}/v2/metrics", json=payload, timeout=25)
    except Exception as e:
        return {"ok": False, "error": f"backend unreachable: {e}", "request": payload}

    if not r.ok:
        # Best-effort parse JSON error; otherwise include snippet
        try:
            j = r.json()
        except Exception:
            j = None
        snippet = (r.text or "")[:600].replace("\n", " ")
        return {
            "ok": False,
            "status_code": r.status_code,
            "error": j if j is not None else snippet,
            "request": payload,
        }

    try:
        return r.json()
    except Exception as e:
        snippet = (r.text or "")[:600].replace("\n", " ")
        return {"ok": False, "error": f"non-json backend response: {e}", "raw": snippet, "request": payload}
'''
with open("api_client.py","w",encoding="utf-8") as f:
    f.write(api_client)

engine = r'''import numpy as np
import random
import math

# Interpretable action space
COOPERATE = 0
TIGHTEN = 1
DEFECT = 2

ACTION_NAMES = {
    COOPERATE: "Cooperate",
    TIGHTEN: "Tighten Controls",
    DEFECT: "Defect"
}

def institutional_capacity_from_wb(inst_capacity, fallback=0.55):
    if inst_capacity is None:
        return float(fallback)
    try:
        return float(inst_capacity)
    except Exception:
        return float(fallback)

class EAGPOEnv:
    """E-AGPO-HT-aligned governance environment.

    Includes:
    - actor-specific reward weighting
    - institutional capacity embedded in reward
    - network-weighted sanction propagation (centrality mean)
    - stochastic shock injection (sigma)
    - centrality-weighted payoff asymmetry
    - treaty durability forecasting curve
    """

    def __init__(self, actor_profiles: dict):
        self.actor_profiles = actor_profiles
        self.reset()

    def reset(self):
        self.round = 0
        self.tension = 0.40
        self.stability = 0.60
        self.sanction_pressure = 0.20
        self.history = []  # (tension, stability, sanction_pressure, durability)
        return self.state()

    def state(self):
        return np.array([self.tension, self.stability, self.sanction_pressure], dtype=float)

    def durability(self):
        return float(self.stability * math.exp(-self.tension))

    def step(self, actions: dict, shock_sigma: float = 0.02):
        self.round += 1

        coop = sum(1 for a in actions.values() if a == COOPERATE)
        defect = sum(1 for a in actions.values() if a == DEFECT)

        # Stochastic shock injection
        shock = float(np.random.normal(0.0, shock_sigma))

        # Network-weighted sanction propagation (centrality mean proxy)
        centrality = float(np.mean([self.actor_profiles[a]["centrality"] for a in actions]))

        # Governance dynamics
        self.tension += 0.05 * defect + shock
        self.stability += 0.04 * coop - 0.03 * defect
        self.sanction_pressure += 0.03 * defect * centrality

        # Clamp 0..1
        self.tension = float(np.clip(self.tension, 0, 1))
        self.stability = float(np.clip(self.stability, 0, 1))
        self.sanction_pressure = float(np.clip(self.sanction_pressure, 0, 1))

        # Rewards
        rewards = {}
        for actor, action in actions.items():
            p = self.actor_profiles[actor]
            inst_cap = float(p["institutional_capacity"])
            asymmetry = float(p["centrality"]) * 0.15

            reward = (
                float(p["risk_weight"]) * self.stability
                - (1.0 - float(p["risk_weight"])) * self.tension
                - float(p["sanction_sensitivity"]) * self.sanction_pressure
                + 0.25 * inst_cap
                + asymmetry
            )
            if action == DEFECT:
                reward -= 0.15
            rewards[actor] = float(reward)

        self.history.append((self.tension, self.stability, self.sanction_pressure, self.durability()))
        done = self.round >= 20
        return self.state(), rewards, done

class QAgent:
    """Interpretable tabular Q-learning agent."""
    def __init__(self, name: str, alpha=0.1, gamma=0.9, epsilon=0.15):
        self.name = name
        self.q = {}
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.epsilon = float(epsilon)

    def _key(self, state):
        return tuple(np.round(state, 2))

    def choose(self, state, n_actions=3):
        key = self._key(state)
        if key not in self.q:
            self.q[key] = np.zeros(n_actions, dtype=float)
        if random.random() < self.epsilon:
            return random.randint(0, n_actions-1)
        return int(np.argmax(self.q[key]))

    def update(self, state, action, reward, next_state):
        k = self._key(state)
        nk = self._key(next_state)
        if k not in self.q:
            self.q[k] = np.zeros(3, dtype=float)
        if nk not in self.q:
            self.q[nk] = np.zeros(3, dtype=float)
        self.q[k][action] += self.alpha * (reward + self.gamma*np.max(self.q[nk]) - self.q[k][action])
'''
with open("agpo_rl_engine.py","w",encoding="utf-8") as f:
    f.write(engine)

leader_tools = r'''# Auracelle Charlie – Leader Tools (IP-safe utilities)
# Provides: Leader Decision Brief, Quantitative Scoreboard, AAR Export (MD/PDF)
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from io import BytesIO
from typing import Any, Dict, List, Tuple, Optional

import math

def _clamp(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))

def _to_pct01(x: float) -> float:
    return _clamp(float(x), 0.0, 1.0)

def compute_scoreboard(actor: str,
                       history: List[List[float]],
                       institutional_capacity: Optional[float] = None) -> Dict[str, Any]:
    """
    Compute leader-friendly, interpretable metrics from env history.
    NOTE: These are comparative decision-test outputs, not predictions.
    """
    if not history:
        return {
            "actor": actor,
            "robustness_0_100": None,
            "friction_0_100": None,
            "evidence_threshold_pct": None,
            "coord_latency_turns": None,
        }

    # history rows: [tension, stability, sanction_pressure, durability]
    last = history[-1]
    tension = _to_pct01(last[0])
    stability = _to_pct01(last[1])
    sanction = _to_pct01(last[2])
    durability = _to_pct01(last[3])

    # Composite robustness (interpretable): reward stability+durability, penalize tension+sanction
    comp = (stability + durability)/2.0 - (tension + sanction)/2.0
    robustness = int(round(_clamp((comp + 1.0) * 50.0, 0.0, 100.0)))

    # Friction: average pressure terms
    friction = int(round(_clamp(((tension + sanction)/2.0) * 100.0, 0.0, 100.0)))

    cap = institutional_capacity
    if cap is None:
        cap = 0.55
    cap = _to_pct01(cap)

    # Evidence threshold: higher when tension/sanction high; lower when capacity high
    threshold = 65.0 + 25.0*((tension + sanction)/2.0) - 20.0*cap
    threshold = int(round(_clamp(threshold, 15.0, 95.0)))

    # Coordination latency: first turn meeting "stable enough" condition, else total turns
    latency = len(history)
    for i, row in enumerate(history, start=1):
        t, s, sp, d = map(_to_pct01, row)
        if (s >= 0.70) and (d >= 0.60) and (t <= 0.45):
            latency = i
            break

    return {
        "actor": actor,
        "robustness_0_100": robustness,
        "friction_0_100": friction,
        "evidence_threshold_pct": threshold,
        "coord_latency_turns": latency,
        "last_state": {
            "tension": tension,
            "stability": stability,
            "sanction_pressure": sanction,
            "durability": durability,
        },
    }

def sensitivity_rank(shock_impacts: Dict[str, float]) -> List[Tuple[str, float]]:
    """Return ranked stressors by absolute impact (descending)."""
    items = [(k, float(v)) for k, v in (shock_impacts or {}).items()]
    items.sort(key=lambda kv: abs(kv[1]), reverse=True)
    return items

def build_leader_brief(run: Dict[str, Any],
                       scoreboard: Dict[str, Any],
                       sensitivity: List[Tuple[str, float]]) -> Dict[str, Any]:
    """Structured leader brief content (rendered in Streamlit)."""
    now = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")
    actor = run.get("actor", "—")
    scenario = run.get("scenario", "—")
    shock = run.get("shock", "None")
    intensity = run.get("intensity", "—")
    window = run.get("metrics_window", "—")
    trade_year = run.get("trade_year", "—")

    # Most recent pressures (derived)
    last = scoreboard.get("last_state") or {}
    pressures = []
    if last:
        pressures = [
            ("Tension", int(round(last.get("tension", 0)*100))),
            ("Sanction pressure", int(round(last.get("sanction_pressure", 0)*100))),
        ]

    top_sens = [f"{k} (Δ {v:+.1f})" for k, v in (sensitivity or [])[:3]] or ["—"]

    return {
        "header": f"Leader Decision Brief — {actor}",
        "timestamp": now,
        "decision_question": "Given current constraints and pressures, how should the actor posture on this policy move this round?",
        "run_context": {
            "Scenario": scenario,
            "Shock": shock,
            "Intensity": intensity,
            "Metrics window": window,
            "Trade year": trade_year,
        },
        "now_state": pressures,
        "scoreboard": scoreboard,
        "top_sensitivity": top_sens,
        "focus": [
            "This is a decision-test snapshot, not a forecast.",
            "Use robustness + friction + evidence threshold + latency to compare options under stress.",
        ],
        "triggers": [
            "Evidence confidence changes materially (new verification, audit signals, incident disclosure).",
            "Shock conditions escalate or reverse (sanctions, supply chain, cyber incident severity).",
            "Alliance posture shifts (coordination commitments or defections).",
        ],
    }

def build_aar_markdown(run: Dict[str, Any],
                       scoreboard: Dict[str, Any],
                       sensitivity: List[Tuple[str, float]],
                       event_log: List[str]) -> str:
    """AAR in Markdown."""
    ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")
    actor = run.get("actor", "—")
    scenario = run.get("scenario", "—")
    shock = run.get("shock", "None")
    intensity = run.get("intensity", "—")
    mw = run.get("metrics_window", "—")
    trade_year = run.get("trade_year", "—")

    lines = []
    lines.append(f"# Auracelle Charlie — After Action Report (AAR)")
    lines.append(f"- **Generated:** {ts}")
    lines.append("")
    lines.append("## Run Context")
    lines.append(f"- **Actor:** {actor}")
    lines.append(f"- **Scenario:** {scenario}")
    lines.append(f"- **Shock:** {shock}")
    lines.append(f"- **Intensity:** {intensity}")
    lines.append(f"- **Metrics window:** {mw}")
    lines.append(f"- **Trade year:** {trade_year}")
    lines.append("")
    lines.append("## Quantitative Scoreboard (Decision-Test Outputs)")
    lines.append(f"- **Robustness (0–100):** {scoreboard.get('robustness_0_100')}")
    lines.append(f"- **Friction (0–100):** {scoreboard.get('friction_0_100')}")
    lines.append(f"- **Evidence threshold (%):** {scoreboard.get('evidence_threshold_pct')}")
    lines.append(f"- **Coordination latency (turns):** {scoreboard.get('coord_latency_turns')}")
    lines.append("")
    lines.append("## Sensitivity (Top Stressors by Impact)")
    if sensitivity:
        for k, v in sensitivity[:10]:
            lines.append(f"- {k}: Δ {v:+.1f}")
    else:
        lines.append("- —")
    lines.append("")
    lines.append("## Event Log (Operational Trace)")
    if event_log:
        for e in event_log[-80:]:
            lines.append(f"- {e}")
    else:
        lines.append("- —")
    lines.append("")
    lines.append("## Notes")
    lines.append("- Outputs are comparative decision-test artifacts; they do not predict real-world actions.")
    return "\n".join(lines)

def build_aar_pdf_bytes(markdown_text: str) -> bytes:
    """
    Minimal PDF builder from markdown text.
    Uses reportlab if available; otherwise returns UTF-8 bytes of markdown.
    """
    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.pdfgen import canvas
    except Exception:
        return markdown_text.encode("utf-8")

    buf = BytesIO()
    c = canvas.Canvas(buf, pagesize=letter)
    width, height = letter

    x = 50
    y = height - 60
    line_h = 12

    for raw_line in markdown_text.splitlines():
        line = raw_line.replace("\t", "    ")
        if y < 60:
            c.showPage()
            y = height - 60
        c.drawString(x, y, line[:110])
        y -= line_h

    c.save()
    return buf.getvalue()
'''
with open("auracelle_leader_tools.py","w",encoding="utf-8") as f:
    f.write(leader_tools)


page22 = r'''import streamlit as st
import numpy as np
import re
from io import BytesIO
import matplotlib.pyplot as plt
from datetime import date

from api_client import post_metrics
from agpo_rl_engine import EAGPOEnv, QAgent, ACTION_NAMES, COOPERATE, TIGHTEN, DEFECT, institutional_capacity_from_wb

# ------------------------------
# Page config + minimal styling (Streamlit-native widgets; CSS only for look-and-feel)
# ------------------------------
st.set_page_config(layout="wide", page_title="Auracelle Policy Stress Testing Platform")

# Gate: require login/session setup
auth_ok = any(bool(st.session_state.get(k, False)) for k in ("authenticated","logged_in","is_authenticated","setup_complete","consent"))
if not auth_ok:
    st.warning("Please complete Session Setup / Login before using the Policy Stress Testing Platform.")
    st.stop()

st.markdown(
    """
    <style>
      .block-container {padding-top: 0.8rem; padding-bottom: 0.8rem;}
      [data-testid="stHeader"]{background: rgba(0,0,0,0);}
      [data-testid="stToolbar"]{right: 1.2rem;}
      .ac-topbar{
        border: 1px solid rgba(45,55,72,0.9);
        border-radius: 16px;
        padding: 14px 16px;
        background: linear-gradient(180deg, rgba(30,37,48,0.95) 0%, rgba(22,27,36,0.85) 100%);
        box-shadow: 0 10px 30px rgba(0,0,0,0.45);
        margin-bottom: 10px;
      }
      .ac-pill{
        display:inline-flex; align-items:center; gap:8px;
        padding: 6px 10px; border-radius: 999px;
        border: 1px solid rgba(16,185,129,0.9);
        background: rgba(16,185,129,0.12);
        font-size: 0.85rem;
      }
      .ac-title{
        font-weight: 700; letter-spacing: 0.06em; color: #00d4ff;
        margin: 0; line-height: 1.1;
      }
      .ac-subtitle{margin:0; color:#9ca3af; font-size:0.88rem;}
      .ac-card{
        border: 1px solid rgba(55,65,81,0.9);
        border-radius: 14px;
        padding: 14px;
        background: rgba(15,20,25,0.55);
        box-shadow: 0 8px 22px rgba(0,0,0,0.35);
      }
      .ac-muted{color:#9ca3af;}
      .ac-badge{
        display:inline-block; padding:6px 10px; border-radius: 10px;
        border:1px solid rgba(0,212,255,0.85);
        background: rgba(0,212,255,0.10);
        color:#00d4ff; font-weight:700; font-size:0.8rem;
      }
      .ac-badge-testing{
        border-color: rgba(245,158,11,0.9);
        background: rgba(245,158,11,0.12);
        color: #f59e0b;
      }
      .ac-badge-pass{
        border-color: rgba(16,185,129,0.9);
        background: rgba(16,185,129,0.12);
        color:#10b981;
      }
      .ac-badge-fail{
        border-color: rgba(239,68,68,0.9);
        background: rgba(239,68,68,0.12);
        color:#ef4444;
      }
      .ac-kpi{
        display:flex; gap:18px; flex-wrap:wrap; margin-top:8px;
      }
      .ac-kpi-item{
        padding:10px 12px; border-radius: 12px;
        border:1px solid rgba(27,42,82,0.95);
        background: rgba(12,19,39,0.55);
        min-width: 150px;
      }
      .ac-kpi-v{font-size:1.15rem; font-weight:800;}
      .ac-kpi-l{font-size:0.8rem; color:#9AB0D6;}
      .ac-scenario-title{font-weight:800; font-size:1.05rem;}
      .ac-scenario-desc{color:#9ca3af; font-size:0.85rem; line-height:1.35;}
      .ac-bottombar{
        border: 1px solid rgba(45,55,72,0.9);
        border-radius: 16px;
        padding: 10px 12px;
        background: linear-gradient(180deg, rgba(30,37,48,0.70) 0%, rgba(22,27,36,0.55) 100%);
        box-shadow: 0 10px 30px rgba(0,0,0,0.35);
        margin-top: 10px;
      }
    </style>
    """,
    unsafe_allow_html=True,
)

# ------------------------------
# State
# ------------------------------
if "pst_results" not in st.session_state:
    st.session_state.pst_results = {}  # scenario_id -> dict
if "pst_selected" not in st.session_state:
    st.session_state.pst_selected = "coalition_stability"

ACTORS = ["US", "China", "India", "UK", "Japan", "Brazil", "Dubai", "NATO"]

# Actor-specific weights (interpretable defaults; can be tuned in E-AGPO-HT docs)
DEFAULT_RISK_WEIGHT = {
    "US": 0.62, "China": 0.55, "India": 0.58, "UK": 0.64,
    "Japan": 0.66, "Brazil": 0.54, "Dubai": 0.60, "NATO": 0.63,
}
DEFAULT_SANCTION_SENS = {
    "US": 0.18, "China": 0.30, "India": 0.22, "UK": 0.20,
    "Japan": 0.21, "Brazil": 0.24, "Dubai": 0.20, "NATO": 0.19,
}
DEFAULT_CENTRALITY = {
    "US": 0.92, "China": 0.88, "India": 0.74, "UK": 0.70,
    "Japan": 0.68, "Brazil": 0.62, "Dubai": 0.55, "NATO": 0.80,
}

def _safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def fetch_actor_snapshot(actor, start_year, end_year, trade_year, policy_name, shock, intensity):
    """Live API snapshot from FastAPI /v2/metrics (World Bank/IMF/etc)."""
    return post_metrics(
        actor=actor,
        start_year=start_year,
        end_year=end_year,
        trade_year=trade_year,
        policy_name=policy_name,
        shock=shock,
        intensity=intensity,
    )

def build_actor_profiles(stakeholders, api_snapshots):
    profiles = {}
    for a in stakeholders:
        snap = api_snapshots.get(a, {}) if isinstance(api_snapshots, dict) else {}
        inst = institutional_capacity_from_wb(_safe_get(snap, "derived", "institutional_capacity", default=None), fallback=0.55)
        profiles[a] = {
            "risk_weight": float(DEFAULT_RISK_WEIGHT.get(a, 0.58)),
            "sanction_sensitivity": float(DEFAULT_SANCTION_SENS.get(a, 0.22)),
            "institutional_capacity": float(inst),
            "centrality": float(DEFAULT_CENTRALITY.get(a, 0.65)),
        }
    return profiles

def run_interpretable_marl(actor_profiles, episodes=80, epsilon=0.18, shock_sigma=0.03):
    """Multi-agent tabular Q-learning (interpretable) over the EAGPOEnv."""
    agents = {a: QAgent(a, epsilon=epsilon) for a in actor_profiles.keys()}

    durability_by_ep = []
    last_hist = None
    last_rewards = None

    for _ in range(int(episodes)):
        env = EAGPOEnv(actor_profiles)
        state = env.state()
        done = False
        while not done:
            actions = {a: agents[a].choose(state) for a in agents}
            next_state, rewards, done = env.step(actions, shock_sigma=float(shock_sigma))
            for a in agents:
                agents[a].update(state, actions[a], rewards[a], next_state)
            state = next_state
        durability_by_ep.append(env.durability())
        last_hist = env.history[:]  # (tension, stability, sanction_pressure, durability)
        last_rewards = rewards

    return {
        "durability_by_ep": np.array(durability_by_ep, dtype=float),
        "history": np.array(last_hist, dtype=float) if last_hist else np.zeros((0,4)),
        "last_rewards": last_rewards or {},
    }

def score_win_win(last_rewards):
    if not last_rewards:
        return 0.0
    vals = np.array(list(last_rewards.values()), dtype=float)
    # simple interpretable score: mean - std, rescaled to 0..100
    raw = float(np.mean(vals) - 0.50*np.std(vals))
    return float(np.clip(50 + 50*raw, 0, 100))

def count_vulnerabilities(history):
    if history.size == 0:
        return 0
    tension = history[-1, 0]
    sanc = history[-1, 2]
    vul = 0
    if tension > 0.70:
        vul += 2
    if sanc > 0.35:
        vul += 2
    if history[-1, 3] < 0.45:
        vul += 2
    return int(vul)

SCENARIOS = [
    {
        "id": "coalition_stability",
        "title": "Coalition Stability",
        "icon": "🤝",
        "desc": "Tests if agreements hold under asymmetric incentives and uneven capacity (coalition drift).",
        "shock_sigma": 0.03,
    },
    {
        "id": "free_rider_detection",
        "title": "Free Rider Detection",
        "icon": "🎭",
        "desc": "Identifies stakeholders who benefit without contributing (enforcement gaps).",
        "shock_sigma": 0.035,
    },
    {
        "id": "sanction_escalation",
        "title": "Sanction Escalation",
        "icon": "⚡",
        "desc": "Stress tests network-weighted sanction propagation and second-order effects.",
        "shock_sigma": 0.04,
    },
    {
        "id": "shock_resilience",
        "title": "Shock Resilience",
        "icon": "🌪️",
        "desc": "Injects stochastic shocks to evaluate recovery dynamics and durability under volatility.",
        "shock_sigma": 0.06,
    },
]

# ------------------------------
# TOP BAR (matches mock header concepts)
# ------------------------------
with st.container():
    c1, c2 = st.columns([2.2, 1.2])
    with c1:
        st.markdown(
            """
            <div class="ac-topbar">
              <div style="display:flex; align-items:center; gap:14px;">
                <div style="font-size:1.9rem;">🧬</div>
                <div>
                  <div class="ac-title">AURACELLE CHARLIE</div>
                  <div class="ac-subtitle">Policy Stress Testing &amp; Win-Win Optimization Platform</div>
                </div>
              </div>
            </div>
            """,
            unsafe_allow_html=True,
        )
    with c2:
        st.markdown('<div class="ac-topbar" style="display:flex; align-items:center; justify-content:space-between; gap:10px;">', unsafe_allow_html=True)
        tech = st.selectbox("Technology Domain", ["🤖 AI Governance", "🧬 BioTech", "⚛️ Nuclear", "🛰️ Space", "🧪 Quantum"], index=0, label_visibility="collapsed")
        st.markdown('<span class="ac-pill"><span style="width:8px;height:8px;border-radius:999px;background:#10b981;display:inline-block;"></span>STRESS TEST READY</span>', unsafe_allow_html=True)
        st.markdown('</div>', unsafe_allow_html=True)

# ------------------------------
# MAIN WORKSPACE: Left config | Center tests | Right analysis
# ------------------------------
left, mid, right = st.columns([1.15, 2.2, 1.15], gap="medium")

with left:
    st.markdown('<div class="ac-card">', unsafe_allow_html=True)
    st.markdown("### 📋 Policy Input")
    uploaded = st.file_uploader("Upload Policy Draft (PDF, DOCX, TXT)", type=["pdf", "docx", "txt"])
    policy_title = st.text_input("Policy Title", value="International AI Safety Standards")
    policy_summary = st.text_area(
        "Policy Summary / Draft",
        height=140,
        value="A cooperative verification and assurance framework for frontier AI systems, including disclosure, audits, incident response, and enforcement incentives."
    )

    st.divider()
    st.markdown("### 🤝 Stakeholders")
    stakeholders = st.multiselect("Select stakeholders", ACTORS, default=["US", "China", "India"])
    st.caption("Tip: keep this to 3-5 actors to maintain interpretability and speed.")

    st.divider()
    st.markdown("### ⚙️ Test Parameters")
    start_year, end_year = st.slider("Metrics window", min_value=2000, max_value=2024, value=(2000, 2024), step=1)
    trade_year = st.slider("Trade year", min_value=2010, max_value=2024, value=2024, step=1)
    intensity = st.slider("Policy intensity", min_value=0, max_value=100, value=62, step=1)
    shock = st.selectbox("Baseline shock", ["None", "SupplyChain", "CyberIncident", "BioEvent", "ConflictEscalation"], index=0)

    st.divider()
    st.markdown("### 🧠 Training")
    episodes = st.slider("Episodes", min_value=20, max_value=500, value=120, step=10)
    epsilon = st.slider("Exploration epsilon", min_value=0.00, max_value=0.50, value=0.18, step=0.01)
    base_sigma = st.slider("Shock volatility (sigma)", min_value=0.00, max_value=0.10, value=0.03, step=0.005)

    st.markdown('</div>', unsafe_allow_html=True)

with mid:
    st.markdown('<div class="ac-card">', unsafe_allow_html=True)
    # Banner
    badge = "TESTING" if st.session_state.pst_results else "READY"
    badge_class = "ac-badge-testing" if badge == "TESTING" else ""
    st.markdown(
        f"""
        <div style="display:flex; align-items:center; justify-content:space-between; gap:10px;">
          <div>
            <div style="font-size:1.45rem; font-weight:800;">{policy_title}</div>
            <div class="ac-muted" style="margin-top:4px;">
              🌐 Technology: <strong>{tech.replace("🤖 ","").replace("🧬 ","").replace("⚛️ ","").replace("🛰️ ","").replace("🧪 ","")}</strong>
              &nbsp;&nbsp;🤝 Stakeholders: <strong>{len(stakeholders)} Selected</strong>
              &nbsp;&nbsp;📅 Test Date: <strong>{date.today().isoformat()}</strong>
            </div>
          </div>
          <div class="ac-badge {badge_class}">{badge}</div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    tabs = st.tabs(["Robustness Tests", "Unintended Consequences", "Win-Win Analysis", "Implementation Stress", "Document Stress Test"])

    def scenario_status(sid):
        r = st.session_state.pst_results.get(sid)
        if not r:
            return ("READY", "ac-badge")
        if r.get("status") == "passed":
            return ("PASSED", "ac-badge ac-badge-pass")
        if r.get("status") == "failed":
            return ("FAILED", "ac-badge ac-badge-fail")
        return ("RUNNING", "ac-badge ac-badge-testing")

    def render_scenario_grid():
        cols = st.columns(2, gap="medium")
        for idx, sc in enumerate(SCENARIOS):
            sid = sc["id"]
            col = cols[idx % 2]
            with col:
                lab, klass = scenario_status(sid)
                r = st.session_state.pst_results.get(sid, {})
                win = float(r.get("win_win_score", 0.0))
                vul = int(r.get("vulnerabilities", 0))
                rounds = int(r.get("rounds", 0))
                stab = float(r.get("stability", 0.0))
                st.markdown('<div class="ac-card">', unsafe_allow_html=True)
                st.markdown(
                    f"""
                    <div style="display:flex; align-items:center; justify-content:space-between; gap:10px;">
                      <div class="ac-scenario-title">{sc["title"]}</div>
                      <div style="font-size:1.2rem;">{sc["icon"]}</div>
                    </div>
                    <div class="ac-scenario-desc" style="margin-top:6px;">{sc["desc"]}</div>
                    <div style="margin-top:10px;"><span class="{klass}">{lab}</span></div>
                    """,
                    unsafe_allow_html=True,
                )
                kpi1, kpi2, kpi3, kpi4 = st.columns(4)
                kpi1.metric("Win-Win", f"{win:.0f}%")
                kpi2.metric("Vuln", f"{vul}")
                kpi3.metric("Rounds", f"{rounds}")
                kpi4.metric("Stability", f"{stab:.0f}%")
                run = st.button("View / Run", key=f"run_{sid}", use_container_width=True)
                if run:
                    st.session_state.pst_selected = sid
                    st.rerun()
                st.markdown('</div>', unsafe_allow_html=True)

        with tabs[0]:
            st.caption("Click a scenario to run it using your live API features + E-AGPO-HT-aligned interpretable MARL.")
            render_scenario_grid()

        with tabs[1]:
            st.markdown("#### Unintended consequences checklist")
            st.markdown(
                """
    - Second-order effects (trade diversion, monitoring burden, legitimacy loss)
    - Adversarial adaptation / regulatory arbitrage
    - Uneven compliance costs across stakeholders
    - Measurement gaming / audit shopping
                """
            )

        with tabs[2]:
            st.markdown("#### Win-win analysis")
            st.markdown(
                """
    Use the **Win-Win Optimization Score** and **Treaty Durability** as the two headline outcome measures.

    Suggested exploration:
    - Increase **policy intensity** until durability improves, but watch for stability drops.
    - Add a baseline **shock** (e.g., CyberIncident) and see whether outcomes remain Pareto-like.
    - Tune **volatility (sigma)** to test fragility vs. robustness.
                """
            )

        with tabs[3]:
            st.markdown("#### Implementation stress")
            st.markdown(
                """
    Implementation often fails at the seams. Stress test:
    - Capacity constraints (lowest-capacity actor)
    - Verification bandwidth (audit + reporting load)
    - Enforcement credibility under shocks
    - Cross-border evidence portability (what counts as "proof" across jurisdictions)
                """
            )

        with tabs[4]:
            st.markdown("#### Document stress test")
            st.caption("Upload a policy draft (or reuse the left-panel upload) to sanity-check decision density, stakeholder complexity, and ambiguity markers.")

            def _extract_text(uploaded_file):
                if uploaded_file is None:
                    return ""
                ext = uploaded_file.name.split(".")[-1].lower()
                if ext == "txt":
                    try:
                        return uploaded_file.read().decode("utf-8")
                    except Exception:
                        return uploaded_file.read().decode("utf-8", errors="ignore")
                if ext == "pdf":
                    try:
                        import PyPDF2
                        reader = PyPDF2.PdfReader(BytesIO(uploaded_file.read()))
                        return "\\n".join([(p.extract_text() or "") for p in reader.pages])
                    except Exception:
                        st.warning("PDF parsing requires PyPDF2 (install via pip).")
                        return ""
                if ext == "docx":
                    try:
                        import docx
                        doc = docx.Document(BytesIO(uploaded_file.read()))
                        return "\\n".join([p.text for p in doc.paragraphs])
                    except Exception:
                        st.warning("DOCX parsing requires python-docx (install via pip).")
                        return ""
                return ""

            doc_file = st.file_uploader("Policy draft (PDF/DOCX/TXT)", type=["pdf","docx","txt"], key="pst_doc_upload")
            if doc_file is None and uploaded is not None:
                doc_file = uploaded

            txt = _extract_text(doc_file) if doc_file is not None else ""
            if not txt.strip():
                st.info("Upload a document (or use the left-panel upload) to run a document stress test.")
            else:
                words = len(txt.split())
                sentences = max(1, len(re.findall(r"[.!?]+", txt)))
                decision_terms = ["shall","must","should","required","requires","approve","review","assess","audit","verify","ensure"]
                stakeholder_terms = ["board","committee","team","authority","regulator","audit","legal","oversight","operator","vendor","civil society"]
                ambiguity_terms = ["as appropriate","where feasible","when possible","reasonable","timely","adequate","sufficient","material","significant"]

                decision_hits = sum(txt.lower().count(t) for t in decision_terms)
                stakeholder_hits = sum(txt.lower().count(t) for t in stakeholder_terms)
                ambiguity_hits = sum(txt.lower().count(t) for t in ambiguity_terms)

                dpk = (decision_hits / max(1, words)) * 1000.0
                spk = (stakeholder_hits / max(1, words)) * 1000.0
                apk = (ambiguity_hits / max(1, words)) * 1000.0

                c1, c2, c3, c4 = st.columns(4)
                c1.metric("Words", f"{words:,}")
                c2.metric("Decision terms / 1k words", f"{dpk:.1f}")
                c3.metric("Stakeholder refs / 1k words", f"{spk:.1f}")
                c4.metric("Ambiguity markers / 1k words", f"{apk:.1f}")

                st.markdown("**Stress flags**")
                flags = []
                if dpk > 18:
                    flags.append("- High decision density: consider a decision log (owner, trigger, evidence, cadence).")
                if spk > 10:
                    flags.append("- Many stakeholders referenced: add an explicit RACI / accountability table.")
                if apk > 4:
                    flags.append("- Ambiguity markers detected: tighten definitions and measurable criteria.")
                if "incident" not in txt.lower():
                    flags.append("- Incident handling not obvious: add reporting + escalation triggers.")
                if not any(k in txt.lower() for k in ["audit","assurance","evaluation","test"]):
                    flags.append("- Assurance language not obvious: specify what evidence is produced and by whom.")
                if not flags:
                    flags.append("- No obvious red flags at this surface-level scan. Next: test enforcement incentives under shocks.")

                st.markdown("\\n".join(flags))

    st.divider()

    # Run controls for selected scenario
    selected = st.session_state.pst_selected
    selected_obj = next((s for s in SCENARIOS if s["id"] == selected), SCENARIOS[0])
    c_run1, c_run2 = st.columns([1, 1])
    with c_run1:
        run_selected = st.button(f"▶ Run Selected: {selected_obj['title']}", type="primary", use_container_width=True, key="pst_run_selected")
    with c_run2:
        run_all = st.button("⚙ Run All Tests", use_container_width=True, key="pst_run_all_top")

    # Execute
    def _execute_one(sc):
        if len(stakeholders) < 2:
            st.warning("Select at least 2 stakeholders.")
            return None

        # Live API snapshots for each stakeholder
        api_snaps = {}
        for a in stakeholders:
            api_snaps[a] = fetch_actor_snapshot(a, start_year, end_year, trade_year, policy_title, shock, intensity)

        # If backend failed for any actor, continue with defaults but surface it.
        failed = [k for k,v in api_snaps.items() if isinstance(v, dict) and v.get("ok") is False]
        if failed:
            st.warning("Backend metrics unavailable for: " + ", ".join(failed) + ". Using default profile values for those actors.")

        profiles = build_actor_profiles(stakeholders, api_snaps)
        out = run_interpretable_marl(
            profiles,
            episodes=int(episodes),
            epsilon=float(epsilon),
            shock_sigma=float(max(base_sigma, sc["shock_sigma"])),
        )

        hist = out["history"]
        last_rewards = out["last_rewards"]
        win = score_win_win(last_rewards)
        vul = count_vulnerabilities(hist)
        rounds = int(hist.shape[0])
        stability_pct = float(hist[-1, 1] * 100.0) if rounds > 0 else 0.0
        durability = float(hist[-1, 3]) if rounds > 0 else 0.0

        status = "passed" if (win >= 60 and durability >= 0.45 and vul <= 3) else "failed"
        return {
            "status": status,
            "win_win_score": float(win),
            "vulnerabilities": int(vul),
            "rounds": int(rounds),
            "stability": float(stability_pct),
            "durability": float(durability),
            "history": hist,
            "durability_by_ep": out["durability_by_ep"],
            "api_snaps": api_snaps,
        }

    if run_selected:
        with st.spinner("Running stress test (live API + interpretable MARL)..."):
            res = _execute_one(selected_obj)
        if res:
            st.session_state.pst_results[selected_obj["id"]] = res
            st.success(f"Completed: {selected_obj['title']}")
            st.rerun()

    if run_all:
        with st.spinner("Running all stress tests (live API + interpretable MARL)..."):
            for sc in SCENARIOS:
                st.session_state.pst_results[sc["id"]] = {"status": "running"}
                st.session_state.pst_results[sc["id"]] = _execute_one(sc) or {"status": "failed"}
        st.success("All tests completed.")
        st.rerun()

    # Visualization block for selected scenario (if exists)
    res = st.session_state.pst_results.get(selected_obj["id"])
    if res and isinstance(res, dict) and "history" in res:
        hist = res["history"]
        dur_ep = res["durability_by_ep"]

        st.markdown("#### Governance Trajectory Plots")
        # Separate matplotlib figures (no subplots)
        fig1 = plt.figure()
        plt.plot(hist[:, 0])
        plt.title("Tension (per round)")
        plt.xlabel("Round")
        plt.ylabel("Tension")
        st.pyplot(fig1, clear_figure=True)

        fig2 = plt.figure()
        plt.plot(hist[:, 1])
        plt.title("Stability (per round)")
        plt.xlabel("Round")
        plt.ylabel("Stability")
        st.pyplot(fig2, clear_figure=True)

        fig3 = plt.figure()
        plt.plot(hist[:, 2])
        plt.title("Sanction Pressure (per round)")
        plt.xlabel("Round")
        plt.ylabel("Sanction Pressure")
        st.pyplot(fig3, clear_figure=True)

        st.markdown("#### Treaty Durability Forecasting Curve")
        fig4 = plt.figure()
        plt.plot(hist[:, 3])
        plt.title("Treaty Durability (per round)")
        plt.xlabel("Round")
        plt.ylabel("Durability")
        st.pyplot(fig4, clear_figure=True)

        st.markdown("#### Durability Convergence Curve")
        fig5 = plt.figure()
        plt.plot(dur_ep)
        plt.title("Durability by Episode (training convergence)")
        plt.xlabel("Episode")
        plt.ylabel("Durability")
        st.pyplot(fig5, clear_figure=True)

    st.markdown('</div>', unsafe_allow_html=True)

with right:
    st.markdown('<div class="ac-card">', unsafe_allow_html=True)
    st.markdown("### 📊 Analysis Results")
    selected = st.session_state.pst_selected
    res = st.session_state.pst_results.get(selected, {})
    if res:
        win = float(res.get("win_win_score", 0.0))
        vul = int(res.get("vulnerabilities", 0))
        rounds = int(res.get("rounds", 0))
        durability = float(res.get("durability", 0.0))

        st.metric("Win-Win Optimization Score", f"{win:.0f}%")
        st.metric("Treaty Durability", f"{durability:.3f}")
        st.metric("Vulnerabilities", f"{vul}")
        st.metric("Rounds Executed", f"{rounds}")

        st.divider()
        st.markdown("**Key vulnerabilities**")
        vul_notes = []
        if vul >= 4:
            vul_notes.append("- Enforcement gap risk elevated (sanction pressure / free-riding).")
        if durability < 0.45:
            vul_notes.append("- Durability below target: verification incentives may be insufficient.")
        if win < 60:
            vul_notes.append("- Outcome not Pareto-like: payoff asymmetry needs rebalancing.")
        if not vul_notes:
            vul_notes.append("- No critical vulnerabilities detected under current settings.")
        st.markdown("\n".join(vul_notes))

        st.divider()
        st.markdown("**Recommendations (interpretable knobs)**")
        recs = [
            "- Increase institutional capacity support (capacity-building clauses) for lowest-capacity actor.",
            "- Adjust sanction sensitivity weights to reduce cascading instability.",
            "- Reduce shock volatility exposure by adding incident-response / de-escalation triggers.",
            "- Add verification offsets (audit credits) to discourage defection incentives.",
        ]
        st.markdown("\n".join(recs))

        st.divider()
        with st.expander("Live API Snapshot (selected actor)", expanded=False):
            # show one actor snapshot for transparency
            if isinstance(res.get("api_snaps"), dict) and stakeholders:
                a0 = stakeholders[0]
                st.json(res["api_snaps"].get(a0, {}))
            else:
                st.info("Run a test to populate live API snapshots.")
    else:
        st.info("Run a scenario to populate analysis results.")

    st.markdown('</div>', unsafe_allow_html=True)

# ------------------------------
# Bottom control bar (mock-aligned controls; actions are Streamlit-native)
# ------------------------------
with st.container():
    st.markdown('<div class="ac-bottombar">', unsafe_allow_html=True)
    b1, b2, b3, b4, b5 = st.columns([1, 1, 1, 1, 2])
    with b1:
        st.button("💾 Save Test", use_container_width=True, key="pst_save_test")
    with b2:
        st.button("📤 Export Report", use_container_width=True, key="pst_export_report")
    with b3:
        st.button("⚙ Run All Tests", use_container_width=True, key="pst_run_all_bottom")
    with b4:
        st.button("✅ Approve Policy", type="primary", use_container_width=True, key="pst_approve_policy")
    with b5:
        sel = st.session_state.pst_selected
        r = st.session_state.pst_results.get(sel)
        if r and isinstance(r, dict):
            st.caption(f"Selected: **{sel}** · Status: **{r.get('status','ready').upper()}** · Win-Win: **{float(r.get('win_win_score',0)):.0f}%**")
        else:
            st.caption(f"Selected: **{sel}** · Status: **READY**")
    st.markdown('</div>', unsafe_allow_html=True)
'''
with open("pages/22_Auracelle_Policy_Stress_Testing_Platform.py","w",encoding="utf-8") as f:
    f.write(page22)

page23 = r'''import streamlit as st
import numpy as np
import pandas as pd
import copy
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from api_client import post_metrics
from agpo_rl_engine import EAGPOEnv, ACTION_NAMES, COOPERATE, TIGHTEN, DEFECT, institutional_capacity_from_wb
from auracelle_leader_tools import compute_scoreboard, build_leader_brief, build_aar_markdown, build_aar_pdf_bytes, sensitivity_rank

st.set_page_config(page_title="Auracelle Charlie Mission Console", layout="wide")

# --- Console skin (native Streamlit + light CSS; not embedding the HTML mock) ---
st.markdown(r"""
<style>
.block-container {padding-top: 0.8rem; padding-bottom: 1.2rem; max-width: 1400px;}
h1, h2, h3 {letter-spacing: .2px;}
.console-card{
  border: 1px solid rgba(255,255,255,.10);
  border-radius: 18px;
  padding: 14px 14px 12px 14px;
  background: rgba(10,16,34,.25);
}
.pill{
  display:inline-flex; gap:8px; align-items:center;
  padding:7px 10px; border-radius:999px;
  border:1px solid rgba(255,255,255,.12);
  background: rgba(10,16,34,.35);
  font-size: 12px;
}
.dot{width:8px; height:8px; border-radius:999px; display:inline-block;}
.dot.ok{background:#46F0B6;}
.dot.warn{background:#FFCC66;}
.dot.bad{background:#FF6B6B;}
.kpi-grid{
  display:grid;
  grid-template-columns: 1fr 1fr 1fr;
  gap:10px;
}
.kpi{
  border:1px solid rgba(255,255,255,.10);
  border-radius: 16px;
  padding: 10px 12px;
  background: rgba(10,16,34,.28);
}
.kpi .label{font-size:12px; opacity:.85;}
.kpi .value{font-size:22px; font-weight:700; margin-top:2px;}
.kpi .hint{font-size:11px; opacity:.75; margin-top:6px;}
.ticker{
  border:1px solid rgba(255,255,255,.10);
  border-radius: 12px;
  padding: 8px 10px;
  font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono", monospace;
  font-size: 11px;
  background: rgba(10,16,34,.28);
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}
.small-muted{font-size:12px; opacity:.8;}
</style>
""", unsafe_allow_html=True)

# -------------------------
# Session state scaffolding
# -------------------------
if "mc_round" not in st.session_state:
    st.session_state.mc_round = 0
if "mc_phase" not in st.session_state:
    st.session_state.mc_phase = "NEGOTIATION"
if "mc_t0" not in st.session_state:
    st.session_state.mc_t0 = datetime.utcnow()
if "mc_event_log" not in st.session_state:
    st.session_state.mc_event_log = []  # list[str]
if "mc_env" not in st.session_state:
    st.session_state.mc_env = None
if "mc_actor" not in st.session_state:
    st.session_state.mc_actor = None

def log_event(msg: str):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    st.session_state.mc_event_log.append(f"[{ts}] {msg}")
    # keep last 200
    st.session_state.mc_event_log = st.session_state.mc_event_log[-200:]

# -------------------------
# Header / top bar
# -------------------------
top_left, top_right = st.columns([1.45, 1.0], vertical_alignment="center")

with top_left:
    st.markdown("## 🎯 Auracelle Charlie Mission Console")
    st.markdown('<div class="small-muted">Strategic cognition view • E-AGPO-HT governance dynamics • Live metrics</div>', unsafe_allow_html=True)

with top_right:
    # We'll set these based on API connectivity and current risk
    # (computed after the API call; placeholders now)
    pass

st.divider()

# -------------------------
# Operator selections (match mock control strip)
# -------------------------
ctrl1, ctrl2, ctrl3, ctrl4, ctrl5 = st.columns([1.0, 1.6, 1.1, 1.2, 1.1])
ACTORS = ["US","China","India","UK","Japan","Brazil","Dubai","NATO"]

actor = ctrl1.selectbox("Actor", ACTORS, index=0)
scenario = ctrl2.selectbox("Scenario", [
    "Interim Final Rule — AI Diffusion",
    "Export Controls — Semiconductor Chokepoints",
    "AI Safety Treaty — Verification & Assurance",
    "Data Localization — Cross-Border Flow Tension"
], index=0)
intensity = ctrl3.slider("Intensity", 0, 100, 62)
shock = ctrl4.selectbox("Shock", ["None","Supply Chain Disruption","Sanctions Escalation","Cyber Incident","Alliance Realignment"], index=0)
trade_year = ctrl5.slider("Trade Year", 2010, 2024, 2020)

start_year, end_year = st.slider("Metrics Window", 2000, 2024, (2010, 2024))

# -------------------------
# Live API call
# -------------------------
api_ok = True
api_err = None
metrics = None
try:
    metrics = post_metrics(actor, start_year, end_year, trade_year, scenario, shock, intensity)
except Exception as e:
    api_ok = False
    api_err = e

# Derive institutional capacity + sanctions index
inst_cap = 0.55
sanctions_index = 0.25
if api_ok and metrics:
    inst_cap = institutional_capacity_from_wb((metrics.get('derived') or {}).get("institutional_capacity"))
    sanctions_index = float((metrics.get('derived') or {}).get("sanctions_index", 0.25))

# -------------------------
# Actor profile (weights + centrality) — replace centrality with your PageRank table when wired
# -------------------------
CENTRALITY = {"US":0.90,"China":0.88,"India":0.65,"UK":0.72,"Japan":0.70,"Brazil":0.55,"Dubai":0.50,"NATO":0.75}.get(actor, 0.60)
RISK_W = {"US":0.65,"China":0.70,"India":0.55,"UK":0.62,"Japan":0.60,"Brazil":0.55,"Dubai":0.58,"NATO":0.66}.get(actor, 0.60)
SAN_S = {"US":0.50,"China":0.40,"India":0.60,"UK":0.52,"Japan":0.45,"Brazil":0.55,"Dubai":0.50,"NATO":0.48}.get(actor, 0.50)

profile = {actor: {
    "institutional_capacity": float(inst_cap),
    "centrality": float(CENTRALITY),
    "risk_weight": float(RISK_W),
    "sanction_sensitivity": float(SAN_S),
}}

# Initialize / swap environment when actor changes
if (st.session_state.mc_env is None) or (st.session_state.mc_actor != actor):
    st.session_state.mc_env = EAGPOEnv(profile)
    st.session_state.mc_actor = actor
    st.session_state.mc_round = 0
    st.session_state.mc_phase = "NEGOTIATION"
    st.session_state.mc_t0 = datetime.utcnow()
    st.session_state.mc_event_log = []
    log_event(f"Session initialized for {actor} | scenario='{scenario}' shock='{shock}' intensity={intensity}")

env = st.session_state.mc_env

# -------------------------
# Status pills (match mock vibe)
# -------------------------
# Risk posture derived from tension + sanctions pressure
risk_posture = "NOMINAL"
risk_dot = "ok"
risk_score = float(env.tension) + float(env.sanction_pressure)*0.6
if risk_score >= 0.85:
    risk_posture, risk_dot = "CRITICAL", "bad"
elif risk_score >= 0.65:
    risk_posture, risk_dot = "ELEVATED", "warn"

feed_label = "NOMINAL" if api_ok else "DEGRADED"
feed_dot = "ok" if api_ok else "bad"

sim_label = "LIVE"
sim_dot = "ok"

tplus = datetime.utcnow() - st.session_state.mc_t0
tplus_str = str(tplus).split(".")[0]

# Render pills in the top-right area
pill_row = f"""
<div style="display:flex; gap:10px; justify-content:flex-end; flex-wrap:wrap;">
  <div class="pill"><span class="dot {sim_dot}"></span>SIMULATION: <b>{sim_label}</b></div>
  <div class="pill"><span class="dot {risk_dot}"></span>RISK POSTURE: <b>{risk_posture}</b></div>
  <div class="pill"><span class="dot {feed_dot}"></span>DATA FEEDS: <b>{feed_label}</b></div>
  <div class="pill">ROUND: <b>{st.session_state.mc_round:02d}</b> • PHASE: <b>{st.session_state.mc_phase}</b> • <span style="font-family:ui-monospace;">T+{tplus_str}</span></div>
</div>
"""
# Put pills on the page (under the divider)
st.markdown(pill_row, unsafe_allow_html=True)

# If API failed, show a console-style error card, but keep the layout
if not api_ok:
    st.markdown('<div class="console-card" style="margin-top:10px;">', unsafe_allow_html=True)
    st.error("Backend connection failed. Start the FastAPI cell first.")
    st.exception(api_err)
    st.markdown("</div>", unsafe_allow_html=True)
    st.stop()

# -------------------------
# Main console grid
# -------------------------
left, right = st.columns([1.65, 1.0], gap="large")

with left:
    st.markdown('<div class="console-card">', unsafe_allow_html=True)
    st.markdown("### SYNTHETIC OPERATIONS SPACE")
    st.caption("Influence Field • Actors • Edges • Shocks • Negotiation context")

    tab1, tab2, tab3 = st.tabs(["Influence Map", "Geo Map", "Centrality & Deltas"])

    with tab1:
        st.info("Hook point: insert your existing influence network graph here (Plotly / network view).")
        st.write("Current actor centrality (proxy):", round(CENTRALITY, 3))
        st.write("Sanction pressure:", round(env.sanction_pressure, 3))

    with tab2:
        st.info("Hook point: insert your existing Plotly geo map here (Dubai→UAE, NATO marker near Brussels).")

    with tab3:
        st.info("Hook point: show PageRank table, influence deltas, and asymmetry drivers.")
        df = pd.DataFrame([{
            "actor": actor,
            "centrality": CENTRALITY,
            "risk_weight": RISK_W,
            "sanction_sensitivity": SAN_S,
            "institutional_capacity": inst_cap
        }])
        st.dataframe(df, use_container_width=True, hide_index=True)

    st.markdown("</div>", unsafe_allow_html=True)

    st.markdown('<div class="console-card" style="margin-top:10px;">', unsafe_allow_html=True)
    st.markdown("### OPERATIONS DIAGNOSTICS")
    d1, d2, d3, d4 = st.columns(4)
    d1.metric("Institutional capacity", f"{inst_cap:.2f}")
    d2.metric("Sanctions index", f"{sanctions_index:.2f}")
    d3.metric("Internet users %", f"{(metrics.get('latest') or {}).get('internet_users_pct', None) or 0:.2f}")
    d4.metric("Mil exp % GDP", f"{(metrics.get('latest') or {}).get('mil_exp_pct_gdp', None) or 0:.2f}")
    st.markdown("</div>", unsafe_allow_html=True)

with right:
    # KPI cards
    st.markdown('<div class="console-card">', unsafe_allow_html=True)
    st.markdown("### KEY GOVERNANCE INDICATORS")

    # Use HTML KPI blocks so it reads like the mock (still native values)
    tension = float(env.tension)
    stability = float(env.stability)
    durability = float(env.durability())

    st.markdown(f"""
    <div class="kpi-grid">
      <div class="kpi"><div class="label">Tension</div><div class="value">{tension:.2f}</div><div class="hint">Escalation pressure + shocks</div></div>
      <div class="kpi"><div class="label">Stability</div><div class="value">{stability:.2f}</div><div class="hint">Institutional resilience</div></div>
      <div class="kpi"><div class="label">Treaty Durability</div><div class="value">{durability:.3f}</div><div class="hint">Forecast curve anchor</div></div>
    </div>
    """, unsafe_allow_html=True)
    st.markdown("</div>", unsafe_allow_html=True)

    # Operator controls card
    st.markdown('<div class="console-card" style="margin-top:10px;">', unsafe_allow_html=True)
    st.markdown("### OPERATOR CONTROLS")

    act = st.selectbox("Action", [COOPERATE, TIGHTEN, DEFECT], format_func=lambda a: ACTION_NAMES[a])
    shock_sigma = st.slider("Shock volatility (sigma)", 0.00, 0.10, 0.02, step=0.01)

    b1, b2, b3 = st.columns(3)
    advance = b1.button("Advance Round", type="primary", use_container_width=True)
    mediator = b2.button("Mediator Brief", use_container_width=True)
    pause = b3.button("Pause", use_container_width=True)

    if mediator:
        # --- Leader Decision Brief + Scoreboard + AAR Export ---
        hist = list(env.history) if getattr(env, "history", None) else []
        sb = compute_scoreboard(actor, hist, institutional_capacity=float(inst_cap))
        st.session_state.mc_scoreboard = sb

        # Sensitivity: quick scan across shocks (details abstracted; comparative deltas only)
        shock_sigmas = {
            "None": 0.02,
            "Supply Chain Disruption": 0.05,
            "Sanctions Escalation": 0.07,
            "Cyber Incident": 0.06,
            "Alliance Realignment": 0.04,
        }

        def simulate_end_robustness(shock_name: str) -> int:
            # Fresh env to keep scan comparable and avoid mutating live run
            e = EAGPOEnv({actor: profile[actor]})
            # seed for repeatability within session
            np.random.seed(42)
            random.seed(42)
            for _ in range(max(6, len(hist) or 6)):
                # simple policy: intensity nudges more tighten; otherwise cooperate
                if intensity >= 70:
                    a = TIGHTEN
                elif intensity <= 35:
                    a = COOPERATE
                else:
                    a = random.choice([COOPERATE, TIGHTEN, DEFECT])
                e.step({actor: a}, shock_sigma=float(shock_sigmas.get(shock_name, 0.02)))
            sb2 = compute_scoreboard(actor, list(e.history), institutional_capacity=float(inst_cap))
            return int(sb2.get("robustness_0_100") or 0)

        base_r = simulate_end_robustness("None")
        impacts = {}
        for sname in ["Supply Chain Disruption","Sanctions Escalation","Cyber Incident","Alliance Realignment"]:
            impacts[sname] = float(simulate_end_robustness(sname) - base_r)
        sens_ranked = sensitivity_rank(impacts)
        st.session_state.mc_sensitivity = sens_ranked

        run_state = {
            "actor": actor,
            "scenario": scenario,
            "shock": shock,
            "intensity": intensity,
            "metrics_window": f"{start_year}–{end_year}",
            "trade_year": trade_year,
        }
        brief = build_leader_brief(run_state, sb, sens_ranked)

        st.subheader("Leader Decision Brief")
        c1, c2 = st.columns([1.2, 1])
        with c1:
            st.markdown("**" + str(brief.get("header","Leader Decision Brief")) + "**")
            st.markdown("**Decision question:** " + str(brief.get("decision_question", "")))
            st.markdown("**Run context:**")
            for k,v in brief["run_context"].items():
                st.markdown(f"- **{k}:** {v}")
            if brief["now_state"]:
                st.markdown("**Now-state pressures:**")
                for name, val in brief["now_state"]:
                    st.markdown(f"- {name}: {val}/100")
            st.markdown("**Decision triggers (what would change the posture):**")
            for tr in brief["triggers"]:
                st.markdown(f"- {tr}")
        with c2:
            st.markdown("### Quantitative Scoreboard (Decision-Test Outputs)")
            sb_tbl = pd.DataFrame([{
                "Actor": actor,
                "Robustness (0–100)": sb.get("robustness_0_100"),
                "Friction (0–100)": sb.get("friction_0_100"),
                "Evidence threshold (%)": sb.get("evidence_threshold_pct"),
                "Coordination latency (turns)": sb.get("coord_latency_turns"),
            }])
            st.dataframe(sb_tbl, use_container_width=True, hide_index=True)

            st.markdown("**Sensitivity (Top stressors by impact):**")
            if sens_ranked:
                for k,v in sens_ranked[:4]:
                    st.markdown(f"- {k}: Δ {v:+.1f}")
            else:
                st.markdown("- —")

        # --- AAR Export ---
        md = build_aar_markdown(run_state, sb, sens_ranked, st.session_state.get("mc_event_log", []))
        pdf_bytes = build_aar_pdf_bytes(md)

        d1, d2 = st.columns(2)
        with d1:
            st.download_button(
                "Download AAR (Markdown)",
                data=md.encode("utf-8"),
                file_name=f"Auracelle_Charlie_AAR_{actor}_{datetime.utcnow().strftime('%Y%m%d_%H%M')}.md",
                mime="text/markdown",
                use_container_width=True,
            )
        with d2:
            st.download_button(
                "Download AAR (PDF)",
                data=pdf_bytes,
                file_name=f"Auracelle_Charlie_AAR_{actor}_{datetime.utcnow().strftime('%Y%m%d_%H%M')}.pdf",
                mime="application/pdf",
                use_container_width=True,
            )

    if pause:
        st.warning("Simulation paused (UI placeholder).")

    if advance:
        # Step env
        _, rewards, done = env.step({actor: int(act)}, shock_sigma=shock_sigma)
        st.session_state.mc_round += 1
        st.session_state.mc_phase = "NEGOTIATION" if not done else "POSTURE"
        log_event(f"{actor} action={ACTION_NAMES[int(act)]} reward={rewards[actor]:.3f} shock={shock} intensity={intensity}")
        st.success(f"Round advanced. Reward: {rewards[actor]:.3f}")

    st.markdown("</div>", unsafe_allow_html=True)

    # After-action stream card (ticker + scroll log)
    st.markdown('<div class="console-card" style="margin-top:10px;">', unsafe_allow_html=True)
    st.markdown("### AFTER-ACTION STREAM")

    st.markdown(
        f'<div class="ticker">ROUND {st.session_state.mc_round:02d} • {st.session_state.mc_phase} • {actor} • "{scenario}" • shock={shock} • intensity={intensity}</div>',
        unsafe_allow_html=True
    )

    st.text_area("Event Log", "\n".join(st.session_state.mc_event_log[-60:]), height=220)
    st.markdown("</div>", unsafe_allow_html=True)

    # Collapsible live API snapshot (not a primary UI element)
    with st.expander("Live API Snapshot (debug)", expanded=False):
        st.json({"derived": metrics.get("derived", {}), "latest": metrics.get("latest", {})})

# -------------------------
# Trajectory plots (separate figures)
# -------------------------
st.markdown("### Governance Trajectories")
if env.history:
    hist = np.array(env.history)
    fig1 = plt.figure()
    plt.plot(hist[:,0])
    plt.title("Tension Trajectory")
    st.pyplot(fig1)

    fig2 = plt.figure()
    plt.plot(hist[:,1])
    plt.title("Stability Trajectory")
    st.pyplot(fig2)

    fig3 = plt.figure()
    plt.plot(hist[:,2])
    plt.title("Sanction Pressure Trajectory")
    st.pyplot(fig3)

    fig4 = plt.figure()
    plt.plot(hist[:,3])
    plt.title("Treaty Durability Over Rounds")
    st.pyplot(fig4)
else:
    st.info("Advance a round to generate governance trajectories.")
'''
with open("pages/23_Auracelle_Charlie_Mission_Console.py","w",encoding="utf-8") as f:
    f.write(page23)

print("✅ Updated pages: 22_Auracelle_Policy_Stress_Testing_Platform.py and 23_Auracelle_Charlie_Mission_Console.py (plus api_client.py and agpo_rl_engine.py)")


✅ Updated pages: 22_Auracelle_Policy_Stress_Testing_Platform.py and 23_Auracelle_Charlie_Mission_Console.py (plus api_client.py and agpo_rl_engine.py)


In [5]:

# === Launch Auracelle Charlie - Live 2025 Simulation ===
import subprocess
from pyngrok import ngrok
import time

# Kill old tunnels and servers
!pkill -f ngrok || true
!pkill -f streamlit || true
try:
    ngrok.kill()
    ngrok.disconnect()
except:
    pass

# Start Streamlit
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"])

# Start Ngrok with reserved domain
public_url = ngrok.connect(8501, hostname="aiwargame.ngrok.app")
print("✅ Tunnel created:", public_url)

from IPython.display import HTML, display
display(HTML(f'<a href="{public_url}" target="_blank" style="font-size:20px; font-weight:bold; color:#0066cc;">🔗 Launch Auracelle Charlie - Live 2025</a>'))


^C
^C
✅ Tunnel created: NgrokTunnel: "https://aiwargame.ngrok.app" -> "http://localhost:8501"


In [6]:

# --- Add "Agentic AI Demo" Streamlit page ---
# This cell creates pages/90_Agentic_AI_Demo.py so it shows up as a separate page in the app.
# It is self-contained and will not modify your existing Simulation page or UI conventions.

import os
os.makedirs("pages", exist_ok=True)

agentic_page = r"""
# SPDX-License-Identifier: MIT
import time
import math
import random
from dataclasses import dataclass, field
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Agentic AI Demo", layout="wide")

# ------------------------------
# Minimal "aiwargame" sample env
# ------------------------------
# This is a lightweight, self-contained demonstration so it won't clash with your main Simulation logic.
# If your main app exposes shared policy/state helpers later, you can import them here and bypass this local demo.

@dataclass
class Actor:
    name: str
    influence: float = 1.0          # baseline influence (0-3 nominal range)
    compliance: float = 0.5         # nominal "policy compliance / safety posture" (0-1)
    payoff: float = 0.0             # accumulated reward

@dataclass
class Env:
    actors: Dict[str, Actor] = field(default_factory=dict)
    step_count: int = 0
    max_steps: int = 8
    history: List[Dict] = field(default_factory=list)

    def reset(self):
        self.step_count = 0
        self.history = []
        # lightweight baseline
        self.actors = {
            "US": Actor("US", influence=1.2, compliance=0.55),
            "EU": Actor("EU", influence=1.1, compliance=0.65),
            "China": Actor("China", influence=1.25, compliance=0.45),
        }
        return self._obs()

    def _obs(self):
        # Observation is a dict of actor states plus a derived "risk" score
        state = {
            k: {"influence": v.influence, "compliance": v.compliance, "payoff": v.payoff}
            for k, v in self.actors.items()
        }
        # simple aggregate "systemic risk" proxy
        mean_comp = np.mean([v.compliance for v in self.actors.values()])
        mean_inf = np.mean([v.influence for v in self.actors.values()])
        risk = float(max(0.0, 1.2 - (0.6*mean_comp + 0.2*min(1.5, mean_inf))))
        state["_derived"] = {"risk": risk, "t": self.step_count}
        return state

    def step(self, action: str, controlled_actor: str = "US"):
        # Apply an action taken by the "agent" controlling the selected actor.
        # Actions trade off compliance, influence, and a small coalition effect.
        a = self.actors[controlled_actor]

        # stochasticity for realism
        jitter = lambda s: s + random.uniform(-0.01, 0.01)

        coalition_bonus = 0.0
        if action == "Export Controls":
            a.influence = max(0.6, jitter(a.influence + 0.02))
            a.compliance = min(1.0, jitter(a.compliance + 0.04))
        elif action == "Safety Audits":
            a.influence = max(0.6, jitter(a.influence - 0.01))
            a.compliance = min(1.0, jitter(a.compliance + 0.06))
            coalition_bonus = 0.01
        elif action == "Open Data":
            a.influence = min(1.6, jitter(a.influence + 0.03))
            a.compliance = max(0.1, jitter(a.compliance - 0.02))
        elif action == "Joint Standards":
            a.influence = min(1.6, jitter(a.influence + 0.02))
            a.compliance = min(1.0, jitter(a.compliance + 0.03))
            coalition_bonus = 0.02
        else:
            # No-op / hold
            a.influence = jitter(a.influence)
            a.compliance = jitter(a.compliance)

        # Simple coalition effect: others follow a tiny bit
        for k, other in self.actors.items():
            if k == controlled_actor:
                continue
            other.compliance = float(np.clip(other.compliance + coalition_bonus * random.uniform(0.4, 1.2), 0.1, 1.0))

        # Reward = (own influence * 0.6 + own compliance * 0.8) - systemic risk penalty
        obs = self._obs()
        risk = obs["_derived"]["risk"]
        reward = 0.6*a.influence + 0.8*a.compliance - (0.9 * risk)
        a.payoff += reward

        self.step_count += 1
        done = self.step_count >= self.max_steps

        # Log
        self.history.append({
            "t": self.step_count,
            "actor": controlled_actor,
            "action": action,
            "reward": reward,
            "risk": risk,
            "US_infl": self.actors["US"].influence,
            "US_comp": self.actors["US"].compliance,
            "EU_infl": self.actors["EU"].influence,
            "EU_comp": self.actors["EU"].compliance,
            "CN_infl": self.actors["China"].influence,
            "CN_comp": self.actors["China"].compliance,
        })

        return obs, reward, done, {}

# ------------------------------
# A tiny "Agentic AI" controller
# ------------------------------
# A one-step lookahead agent that evaluates each action with a heuristic objective
# and chooses the best. This is intentionally transparent for demo purposes.

ACTIONS = ["Export Controls", "Safety Audits", "Open Data", "Joint Standards", "Hold/No-Op"]

def evaluate_action(env: Env, action: str, actor: str, sims: int = 12):
    # simulate hypothetical outcome for ranking; keep it light-weight
    # Copy a shallow snapshot for Monte Carlo trials
    scores = []
    for _ in range(sims):
        snap = Env()
        # clone state
        snap.actors = {k: Actor(v.name, v.influence, v.compliance, v.payoff) for k, v in env.actors.items()}
        snap.step_count = env.step_count
        obs, reward, _, _ = snap.step(action, controlled_actor=actor)
        # Heuristic: prefer high own reward, lower systemic risk
        srisk = obs["_derived"]["risk"]
        score = reward - 0.3*srisk
        scores.append(score)
    return float(np.mean(scores))

def agent_choose_action(env: Env, actor: str, stochastic: bool = False):
    candidates = []
    for a in ACTIONS:
        s = evaluate_action(env, a, actor)
        candidates.append((a, s))
    candidates.sort(key=lambda x: x[1], reverse=True)
    if stochastic and random.random() < 0.2:
        return random.choice(ACTIONS)
    return candidates[0][0]

# ------------------------------
# UI
# ------------------------------

st.title("🤖 Agentic AI Demo — Sample Game")
st.caption("A transparent, one-step lookahead agent plays a lightweight version of the AI governance wargame.")

left, right = st.columns([2, 1])

with right:
    st.subheader("Agent Controls")
    controlled_actor = st.selectbox("Controlled Actor", ["US", "EU", "China"], index=0)
    horizon = st.slider("Episode Length (steps)", 4, 16, 8, 1)
    stochastic = st.toggle("Enable stochastic exploration (ε≈0.2)", value=True)
    episodes = st.number_input("Batch episodes", min_value=1, max_value=50, value=1, step=1)
    autoplay = st.toggle("Autoplay (fast)", value=False)
    st.markdown("---")
    st.page_link("app.py", label="⬅ Back to Home", icon="🏠")

with left:
    st.subheader("Run a Sample Game")
    env = Env()
    env.max_steps = horizon

    run_now = st.button("▶ Play Sample Game")

    if run_now:
        obs = env.reset()
        frames = []
        for t in range(horizon):
            action = agent_choose_action(env, controlled_actor, stochastic=stochastic)
            obs, reward, done, _ = env.step(action, controlled_actor)
            row = {"t": t+1, "action": action, "reward": round(reward, 4), "risk": round(obs["_derived"]["risk"], 4)}
            for k, v in env.actors.items():
                row[f"{k}_influence"] = round(v.influence, 3)
                row[f"{k}_compliance"] = round(v.compliance, 3)
                row[f"{k}_payoff"] = round(v.payoff, 3)
            frames.append(row)
            if not autoplay:
                st.write(f"Step {t+1}: **{action}** → reward {row['reward']}, risk {row['risk']}")
                time.sleep(0.25)
            if done:
                break

        df = pd.DataFrame(frames)
        st.markdown("#### Episode Trace")
        st.dataframe(df, use_container_width=True)

        # Simple summaries
        st.markdown("#### Final Payoffs")
        summary = pd.DataFrame([{"Actor": k, "Payoff": round(v.payoff, 4), "Influence": round(v.influence,3), "Compliance": round(v.compliance,3)}
                                for k, v in env.actors.items()]).sort_values("Payoff", ascending=False)
        st.dataframe(summary, use_container_width=True)

    st.markdown("---")
    st.subheader("Batch Evaluation")
    if st.button("🏃 Run Batch"):
        results = []
        for ep in range(episodes):
            env = Env(); env.max_steps = horizon; env.reset()
            for t in range(horizon):
                a = agent_choose_action(env, controlled_actor, stochastic=stochastic)
                env.step(a, controlled_actor)
            results.append({k: v.payoff for k, v in env.actors.items()})
        res_df = pd.DataFrame(results)
        st.markdown("Average payoff over batch:")
        st.dataframe(pd.DataFrame(res_df.mean()).rename(columns={0:"Avg Payoff"}), use_container_width=True)

"""

with open("pages/90_Agentic_AI_Demo.py", "w", encoding="utf-8") as f:
    f.write(agentic_page)

print("✅ Created pages/90_Agentic_AI_Demo.py")


✅ Created pages/90_Agentic_AI_Demo.py


In [7]:

# --- Write auracelle_agent_adapter.py (correctly) ---
adapter_src = r"""
import importlib
import random
from typing import List
import numpy as np
import streamlit as st

def _try_get(module, names: List[str]):
    for n in names:
        if hasattr(module, n):
            return getattr(module, n)
    return None

def _coerce_list(x):
    try:
        return list(x)
    except Exception:
        return None

def load_main_sim_handles():
    try:
        app = importlib.import_module("app")
    except Exception:
        return None
    handles = {}
    handles["policy_options"] = _try_get(app, ["policy_options","POLICY_OPTIONS","policies","POLICY_LIST"])
    handles["policy_effects"] = _try_get(app, ["policy_effect_mappings","policy_effects","POLICY_EFFECTS","policy_effect_map"])
    handles["countries"] = _try_get(app, ["countries","country_list","COUNTRIES","nodes","NODES"])
    handles["roles"] = _try_get(app, ["roles","ROLES"])
    handles["apply_policy"] = _try_get(app, ["apply_policy_effects","apply_policy","apply_effects"])
    handles["get_risk"] = _try_get(app, ["compute_systemic_risk","get_risk","risk_metric"])
    for k in ["policy_options","countries","roles"]:
        if handles.get(k) is not None:
            handles[k] = _coerce_list(handles[k])
    handles["app"] = app
    return handles

def init_state(actors: List[str]):
    if "agent_autoplay_state" not in st.session_state:
        st.session_state.agent_autoplay_state = {
            "t": 0,
            "actors": {a: {"influence": 1.0, "compliance": 0.5, "payoff": 0.0} for a in actors},
            "history": []
        }
    return st.session_state.agent_autoplay_state

def toy_risk(state: dict):
    inf = np.mean([v["influence"] for v in state["actors"].values()])
    comp = np.mean([v["compliance"] for v in state["actors"].values()])
    return float(max(0.0, 1.2 - (0.6*comp + 0.2*min(1.5, inf))))

def derive_risk(handles, state):
    if handles and handles.get("get_risk"):
        try:
            return float(handles["get_risk"](state))
        except Exception:
            pass
    return toy_risk(state)

def step_with_main_effects(handles, state: dict, action: str, controlled_actor: str):
    if handles and handles.get("apply_policy"):
        try:
            new_state = handles["apply_policy"](state, action, controlled_actor)
            st.session_state.agent_autoplay_state = new_state
            risk = derive_risk(handles, new_state)
            a = new_state["actors"][controlled_actor]
            reward = 0.6*a["influence"] + 0.8*a["compliance"] - 0.9*risk
            a["payoff"] += reward
            new_state["t"] = new_state.get("t", 0) + 1
            return reward, risk
        except Exception:
            pass
    pe = None
    if handles and handles.get("policy_effects"):
        pe = handles["policy_effects"]
    if isinstance(pe, dict) and action in pe:
        delta = pe[action]
        a = state["actors"].setdefault(controlled_actor, {"influence": 1.0, "compliance": 0.5, "payoff": 0.0})
        if isinstance(delta, dict):
            if "influence" in delta:
                a["influence"] = float(np.clip(a["influence"] + float(delta["influence"]), 0.1, 2.0))
            if "compliance" in delta:
                a["compliance"] = float(np.clip(a["compliance"] + float(delta["compliance"]), 0.0, 1.0))
        risk = derive_risk(handles, state)
        reward = 0.6*a["influence"] + 0.8*a["compliance"] - 0.9*risk
        a["payoff"] += reward
        state["t"] += 1
        return reward, risk
    # fallback toy step
    a = state["actors"].setdefault(controlled_actor, {"influence": 1.0, "compliance": 0.5, "payoff": 0.0})
    jitter = lambda s: s + random.uniform(-0.01, 0.01)
    if action == "Export Controls":
        a["influence"] = max(0.6, jitter(a["influence"] + 0.02))
        a["compliance"] = min(1.0, jitter(a["compliance"] + 0.04))
    elif action == "Safety Audits":
        a["influence"] = max(0.6, jitter(a["influence"] - 0.01))
        a["compliance"] = min(1.0, jitter(a["compliance"] + 0.06))
    elif action == "Open Data":
        a["influence"] = min(1.6, jitter(a["influence"] + 0.03))
        a["compliance"] = max(0.1, jitter(a["compliance"] - 0.02))
    elif action == "Joint Standards":
        a["influence"] = min(1.6, jitter(a["influence"] + 0.02))
        a["compliance"] = min(1.0, jitter(a["compliance"] + 0.03))
    else:
        a["influence"] = jitter(a["influence"])
        a["compliance"] = jitter(a["compliance"])
    risk = derive_risk(handles, state)
    reward = 0.6*a["influence"] + 0.8*a["compliance"] - 0.9*risk
    a["payoff"] += reward
    state["t"] += 1
    return reward, risk

def get_actions(handles):
    if handles and handles.get("policy_options"):
        return [str(x) for x in handles["policy_options"]]
    return ["Export Controls","Safety Audits","Open Data","Joint Standards","Hold/No-Op"]

def get_actors(handles):
    if handles and handles.get("countries"):
        return [str(x) for x in handles["countries"]]
    return ["US","EU","China"]

def evaluate_action(handles, state, action: str, actor: str, sims: int = 8):
    import copy
    scores = []
    for _ in range(sims):
        s = copy.deepcopy(state)
        reward, risk = step_with_main_effects(handles, s, action, actor)
        scores.append(reward - 0.3*risk)
    return float(np.mean(scores))

def agent_choose_action(handles, state, actor: str, stochastic: bool = False):
    acts = get_actions(handles)
    ranked = [(a, evaluate_action(handles, state, a, actor)) for a in acts]
    ranked.sort(key=lambda x: x[1], reverse=True)
    if stochastic and random.random() < 0.2:
        return random.choice(acts)
    return ranked[0][0]
"""
with open("auracelle_agent_adapter.py", "w", encoding="utf-8") as f:
    f.write(adapter_src)
print("✅ Wrote auracelle_agent_adapter.py")


✅ Wrote auracelle_agent_adapter.py


In [8]:
# --- Write auracelle_agentic_upgrades.py (Agentic AI scaffold integrated) ---
agentic_src = r'''

import random
from typing import Dict, List, Tuple

DEFAULT_POLICY_PACKAGES: List[Dict] = [
    {"name": "Safeguards + Compute Reporting + Audit Corridor",
     "moves": ["Privacy Safeguards", "Compute Reporting", "Cross-Border Audit Pathway"],
     "base_cost": 1.2},
    {"name": "Transparency Exchange + Incident Protocol",
     "moves": ["Transparency Exchange", "Incident Response Protocol"],
     "base_cost": 1.0},
    {"name": "Joint Oversight Board + Redress Mechanism",
     "moves": ["Joint Oversight Board", "User Redress Mechanism"],
     "base_cost": 1.1},
    {"name": "Minimal Deal: Shared Definitions",
     "moves": ["Shared Definitions / Terminology"],
     "base_cost": 0.6},
]

def is_institution_actor(actor: str) -> bool:
    a = (actor or "").strip().lower()
    return any(x in a for x in ["nato", "eu", "oecd", "un", "g7", "g20", "gcc", "mena"])

def infer_opponent_stance(last_moves: List[str]) -> str:
    \"\"\"Classify stance from the last 2–3 moves (conciliatory/neutral/hardline).\"\"\"
    if not last_moves:
        return "neutral"
    recent = [m.lower() for m in last_moves[-3:]]
    hard = sum(any(k in m for k in ["escalat", "sanction", "demand", "ultimatum"]) for m in recent)
    conc = sum(any(k in m for k in ["concess", "offer", "share", "cooperat", "joint"]) for m in recent)
    if hard >= 2 and conc == 0:
        return "hardline"
    if conc >= 2 and hard == 0:
        return "conciliatory"
    return "neutral"

class AutonomousNegotiationAgent:
    \"\"\"Lightweight learning agent (linear Q approximator) with opponent stance in state.\"\"\"

    def __init__(self, name: str, institution: bool = False, alpha: float = 0.15, gamma: float = 0.95, eps: float = 0.15):
        self.name = name
        self.institution = institution
        self.alpha = alpha
        self.gamma = gamma
        self.eps = eps
        # weights[action_name] = dict(feature->weight)
        self.weights: Dict[str, Dict[str, float]] = {}
        self.last_opponent_moves: List[str] = []

    def _features(self, tension: float, stance: str, round_idx: int) -> Dict[str, float]:
        # Tiny, interpretable feature set
        return {
            "bias": 1.0,
            "tension": float(tension),
            "round": float(round_idx),
            "stance_conc": 1.0 if stance == "conciliatory" else 0.0,
            "stance_hard": 1.0 if stance == "hardline" else 0.0,
        }

    def _score(self, action_name: str, feats: Dict[str, float]) -> float:
        w = self.weights.get(action_name)
        if w is None:
            # lazy init
            w = {k: 0.0 for k in feats.keys()}
            self.weights[action_name] = w
        return sum(w.get(k, 0.0) * v for k, v in feats.items())

    def choose_package(self, tension: float, policy_packages: List[Dict], round_idx: int = 0) -> Tuple[Dict, str, float]:
        stance = infer_opponent_stance(self.last_opponent_moves)
        feats = self._features(tension=tension, stance=stance, round_idx=round_idx)

        # epsilon-greedy over packages
        if random.random() < self.eps:
            pkg = random.choice(policy_packages)
        else:
            scored = [(pkg, self._score(pkg["name"], feats)) for pkg in policy_packages]
            scored.sort(key=lambda x: x[1], reverse=True)
            pkg = scored[0][0]

        cost_mult = 1.5 if self.institution else 1.0
        return pkg, stance, cost_mult

    def learn(self, prev_tension: float, new_tension: float, round_idx: int, action_name: str, reward: float):
        stance = infer_opponent_stance(self.last_opponent_moves)
        feats = self._features(tension=prev_tension, stance=stance, round_idx=round_idx)
        q = self._score(action_name, feats)

        # bootstrap target using new tension (proxy for next state)
        next_feats = self._features(tension=new_tension, stance=stance, round_idx=round_idx + 1)
        # optimistic estimate: max over known actions
        if self.weights:
            next_q = max(self._score(a, next_feats) for a in self.weights.keys())
        else:
            next_q = 0.0

        target = reward + self.gamma * next_q
        td = target - q

        w = self.weights[action_name]
        for k, v in feats.items():
            w[k] = w.get(k, 0.0) + self.alpha * td * v

class MediatorAgent:
    \"\"\"Rule/heuristic mediator that proposes 2–3 compromise bundles plus a rationale trace.\"\"\"

    def propose(self, positions: Dict, tension: float, policy_packages: List[Dict]) -> Tuple[List[Dict], Dict]:
        # Simple heuristic: when tension is high, propose lower-cost or legitimacy-building bundles
        sorted_pkgs = sorted(policy_packages, key=lambda p: p.get("base_cost", 1.0))
        if tension >= 0.7:
            picks = [sorted_pkgs[0], sorted_pkgs[1]]
        elif tension >= 0.4:
            picks = [sorted_pkgs[1], sorted_pkgs[2]]
        else:
            picks = [sorted_pkgs[2], sorted_pkgs[3] if len(sorted_pkgs) > 3 else sorted_pkgs[0]]

        rationale = {
            "what_each_side_gains": "Lower immediate escalation risk; clearer pathway to mutual verification and trust-building.",
            "what_each_side_concedes": "Partial flexibility on sovereignty/oversight to gain reciprocal assurances.",
            "predicted_diffusion_impact": "Higher if an institutional actor (EU/NATO/UN/OECD) adopts early; moderate otherwise.",
            "predicted_tension_change": "Decrease expected if accepted; neutral-to-increase if rejected under high tension."
        }
        return picks, rationale

class RedTeamAgent:
    \"\"\"Adversarial stress-tester that selects plausible/worst-timed shocks and scores robustness.\"\"\"

    SHOCKS = [
        "Cross-Border Data Breach",
        "Disinformation Wave",
        "Critical Infrastructure AI Failure",
        "Data Localization Emergency",
        "Supply Chain/Compute Restriction Shock",
    ]

    def select_shock(self, tension: float) -> str:
        # Worst-timed: pick higher-severity shocks when tension is already high
        if tension >= 0.75:
            return "Critical Infrastructure AI Failure"
        if tension >= 0.55:
            return "Cross-Border Data Breach"
        return "Disinformation Wave"

    def score_robustness(self, pre_tension: float, post_tension: float) -> str:
        if post_tension - pre_tension > 0.15:
            return "Fracture risk ↑"
        if pre_tension - post_tension > 0.10:
            return "Adapted"
        return "Held"

'''
with open('auracelle_agentic_upgrades.py','w',encoding='utf-8') as f:
    f.write(agentic_src)
print('✅ Wrote auracelle_agentic_upgrades.py')


✅ Wrote auracelle_agentic_upgrades.py


In [9]:

# ========================================
# PATCH: Fix "Real World Data Metrics" API integration (World Bank + Export Controls)
# - Robust reshaping for wbgapi DataFrame output
# - Actor->ISO3 mapping (Dubai->ARE; UK->GBR; EU->EUU; NATO handled gracefully)
# - Trade.gov CSL supports optional API key via env var or Streamlit secrets
# - Adds pages/21_Real_World_Data_Metrics.py (sidebar page)
# ========================================

import os, pathlib

os.makedirs("agpo_data", exist_ok=True)
os.makedirs("pages", exist_ok=True)

# Cleanup: remove any duplicate page created by earlier patches
try:
    dup = pathlib.Path('pages/70_Real_World_Data_Metrics.py')
    if dup.exists():
        dup.unlink()
except Exception:
    pass

# ----------------------------
# agpo_data/worldbank.py (fixed)
# ----------------------------
worldbank_fixed = '''"""AGPO Data Package - World Bank API Integration (robust)

Fetches real-world economic and development indicators for Auracelle Charlie.

Notes:
- Uses wbgapi (World Bank API client).
- Degrades gracefully if an economy code isn't supported.
"""

from __future__ import annotations

from typing import List, Optional
import pandas as pd
import streamlit as st
import wbgapi as wb

def _normalize_year_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize year columns from wbgapi (e.g., 'YR2000' -> '2000')."""
    cols = []
    for c in df.columns:
        s = str(c)
        if s.startswith("YR") and s[2:].isdigit():
            s = s[2:]
        cols.append(s)
    df.columns = cols
    return df

def _pick_country_col(df: pd.DataFrame) -> str:
    for c in ["Country", "country", "Economy", "economy", "name"]:
        if c in df.columns:
            return c
    non_year = [c for c in df.columns if not str(c).isdigit()]
    return non_year[0] if non_year else df.columns[0]

@st.cache_data(ttl=3600)
def get_world_bank_indicator(
    indicator_code: str,
    country_codes: List[str],
    start_year: int = 2015,
    end_year: int = 2024,
) -> pd.DataFrame:
    """Return long-form DataFrame with columns: Country, Year, Value, Indicator, ISO3"""
    try:
        years = list(range(int(start_year), int(end_year) + 1))

        df = wb.data.DataFrame(
            indicator_code,
            economy=country_codes,
            time=years,
            labels=True
        )

        if df is None or len(df) == 0:
            return pd.DataFrame()

        df = df.reset_index()
        country_col = _pick_country_col(df)
        df = _normalize_year_cols(df)

        year_cols = [c for c in df.columns if str(c).isdigit() and len(str(c)) == 4]
        if not year_cols:
            return pd.DataFrame()

        long_df = df.melt(
            id_vars=[country_col],
            value_vars=year_cols,
            var_name="Year",
            value_name="Value",
        ).rename(columns={country_col: "Country"})

        # Attach ISO3 if present
        if "economy" in df.columns:
            mapping = df[[country_col, "economy"]].drop_duplicates().rename(columns={country_col: "Country", "economy": "ISO3"})
            long_df = long_df.merge(mapping, on="Country", how="left")
        else:
            long_df["ISO3"] = None

        long_df["Indicator"] = indicator_code
        long_df["Year"] = pd.to_numeric(long_df["Year"], errors="coerce").astype("Int64")
        long_df["Value"] = pd.to_numeric(long_df["Value"], errors="coerce")
        return long_df.dropna(subset=["Year"])
    except Exception as e:
        st.warning(f"World Bank API error: {e}")
        return pd.DataFrame()

@st.cache_data(ttl=3600)
def get_many_indicators(
    indicator_codes: List[str] | dict,
    country_codes: List[str],
    start_year: int = 2015,
    end_year: int = 2024,
) -> pd.DataFrame:
    # Accept either a list of WB indicator codes or a {label: code} dict.
    if isinstance(indicator_codes, dict):
        indicator_codes = list(indicator_codes.values())
    frames = []
    for code in indicator_codes:
        df = get_world_bank_indicator(code, country_codes, start_year, end_year)
        if not df.empty:
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def latest_value(df: pd.DataFrame, iso3: str, indicator_code: str) -> Optional[float]:
    try:
        sub = df[(df["ISO3"] == iso3) & (df["Indicator"] == indicator_code)].dropna(subset=["Value"])
        if sub.empty:
            return None
        row = sub.sort_values("Year", ascending=False).iloc[0]
        return float(row["Value"])
    except Exception:
        return None

def get_latest_gdp(country_code: str) -> Optional[float]:
    """Backwards-compatible helper: latest GDP in trillions USD."""
    df = get_world_bank_indicator("NY.GDP.MKTP.CD", [country_code], start_year=2000, end_year=2025)
    val = latest_value(df, df["ISO3"].iloc[0] if not df.empty else country_code, "NY.GDP.MKTP.CD")
    return (val / 1e12) if val is not None else None

def get_latest_military_expenditure(country_code: str) -> Optional[float]:
    """Backwards-compatible helper: latest military expenditure (% of GDP)."""
    df = get_world_bank_indicator("MS.MIL.XPND.GD.ZS", [country_code], start_year=2000, end_year=2025)
    val = latest_value(df, df["ISO3"].iloc[0] if not df.empty else country_code, "MS.MIL.XPND.GD.ZS")
    return val
def get_internet_penetration(country_code: str) -> Optional[float]:
    """Backwards-compatible helper: latest internet users (% of population)."""
    df = get_world_bank_indicator("IT.NET.USER.ZS", [country_code], start_year=2000, end_year=2025)
    if df.empty:
        return None
    iso = df["ISO3"].dropna().iloc[0] if "ISO3" in df.columns and df["ISO3"].notna().any() else country_code
    return latest_value(df, iso, "IT.NET.USER.ZS")
'''
pathlib.Path("agpo_data/worldbank.py").write_text(worldbank_fixed, encoding="utf-8")

# ----------------------------
# agpo_data/exportcontrol.py (fixed)
# ----------------------------
exportcontrol_fixed = '''"""AGPO Data Package - US Export Controls (Trade.gov CSL) - robust

Trade.gov CSL commonly requires an API key. Provide it via:
- Streamlit secrets: st.secrets["TRADEGOV_API_KEY"]
- Environment variable: TRADEGOV_API_KEY

If no key is available, functions return an empty DataFrame (UI should explain).
"""

from __future__ import annotations

from typing import Optional, Dict, Any
import os
import requests
import pandas as pd
import streamlit as st

CSL_URL = "https://api.trade.gov/consolidated_screening_list/search"

def _get_api_key() -> Optional[str]:
    try:
        key = st.secrets.get("TRADEGOV_API_KEY", None)
        if key:
            return str(key).strip()
    except Exception:
        pass
    key = os.getenv("TRADEGOV_API_KEY", "").strip()
    return key or None

@st.cache_data(ttl=86400)
def fetch_consolidated_screening_list(size: int = 200, offset: int = 0) -> pd.DataFrame:
    api_key = _get_api_key()
    if not api_key:
        return pd.DataFrame()

    params: Dict[str, Any] = {"api_key": api_key, "size": int(size), "offset": int(offset)}

    try:
        r = requests.get(CSL_URL, params=params, timeout=15)
        if r.status_code in (401, 403, 429):
            return pd.DataFrame()
        r.raise_for_status()
        payload = r.json() if r.text and r.text.strip() else {}
        results = payload.get("results", []) or []

        records = []
        for item in results:
            addrs = item.get("addresses") or []
            country = (addrs[0].get("country") if addrs else None) or "N/A"
            records.append({
                "name": item.get("name", "N/A"),
                "country": country,
                "source": item.get("source", "N/A"),
                "type": item.get("type", "N/A"),
                "programs": ", ".join(item.get("programs", []) or []),
                "remarks": item.get("remarks", "") or "",
            })
        return pd.DataFrame(records)
    except Exception:
        return pd.DataFrame()

def check_entity_sanctions(entity_name: str) -> Optional[pd.DataFrame]:
    df = fetch_consolidated_screening_list()
    if df.empty:
        return None
    return df[df["name"].str.contains(entity_name, case=False, na=False)]


def get_sanctioned_countries(size: int = 200, offset: int = 0) -> list[str]:
    """Return a sorted list of countries appearing in the CSL results.

    Note: CSL is not strictly a "sanctions list" (it aggregates multiple screening sources),
    but this provides a useful governance-friction signal for Auracelle Charlie.
    """
    df = fetch_consolidated_screening_list(size=size, offset=offset)
    if df.empty or "country" not in df.columns:
        return []
    countries = (
        df["country"]
        .dropna()
        .astype(str)
        .str.strip()
    )
    countries = countries[countries.ne("") & countries.ne("N/A")]
    # Normalize common variants
    norm = countries.str.title()
    return sorted(set(norm.tolist()))


def get_export_control_snapshot(actor_iso3: str | None = None, size: int = 200, offset: int = 0) -> dict:
    """Return a lightweight snapshot for the UI/admin layer.

    If TRADEGOV_API_KEY is missing or the API fails, returns enabled=False with a note.
    """
    api_key = _get_api_key()
    if not api_key:
        return {
            "enabled": False,
            "note": "TRADEGOV_API_KEY not set (Streamlit secrets or environment variable).",
            "total_records": 0,
            "unique_countries": 0,
            "actor_iso3": actor_iso3,
        }

    df = fetch_consolidated_screening_list(size=size, offset=offset)
    if df is None or df.empty:
        return {
            "enabled": True,
            "note": "No CSL results returned (API may be rate-limited or empty for this query window).",
            "total_records": 0,
            "unique_countries": 0,
            "actor_iso3": actor_iso3,
        }

    countries = []
    if "country" in df.columns:
        countries = (
            df["country"].dropna().astype(str).str.strip()
        )
        countries = countries[countries.ne("") & countries.ne("N/A")].tolist()

    return {
        "enabled": True,
        "note": "CSL snapshot (not a sanctions list; aggregated screening sources).",
        "total_records": int(len(df)),
        "unique_countries": int(len(set(countries))) if countries else 0,
        "top_countries": pd.Series(countries).value_counts().head(10).to_dict() if countries else {},
        "actor_iso3": actor_iso3,
    }
'''
pathlib.Path("agpo_data/exportcontrol.py").write_text(exportcontrol_fixed, encoding="utf-8")

# ----------------------------
# agpo_data/actor_map.py (new helper)
# ----------------------------
actor_map = '''"""Actor mapping helpers for Auracelle Charlie (ISO3, UN M49, Comtrade codes).

Why:
- Charlie's UI uses "Actors" (Dubai, UK, NATO, etc.)
- Different external datasets key countries differently:
  * ISO3 alpha codes (World Bank, IMF DataMapper)
  * UN M49 numeric codes (UN SDG API)
  * UN Comtrade numeric reporter/partner codes (commonly aligned with M49)

Design principle:
- Provide thin, explicit mappings for Charlie's canonical actor list.
- Degrade gracefully for non-state actors (e.g., NATO).
"""

from __future__ import annotations
from typing import Optional, Dict

# --- ISO3 (alpha-3) codes ---
ACTOR_TO_ISO3: Dict[str, str] = {
    "Dubai": "ARE",   # Dubai -> UAE
    "UK": "GBR",
    "US": "USA",
    "Japan": "JPN",
    "China": "CHN",
    "Brazil": "BRA",
    "India": "IND",
    # New actors
    "Israel": "ISR",
    "Paraguay": "PRY",
    "Belgium": "BEL",
    "Denmark": "DNK",
    "Ukraine": "UKR",
    "Serbia": "SRB",
    "Argentina": "ARG",
    "Norway": "NOR",
    "Switzerland": "CHE",
    "Poland": "POL",
    # Regional / alliance entities (no single ISO3)
    "NATO": "NATO",
    "Global South": None,
}

# --- UN M49 numeric codes (used by UNSD SDG API) ---
# Source: UN M49 standard (numeric geo area codes). Keep tight for Charlie's list.
ACTOR_TO_M49: Dict[str, int] = {
    "Dubai": 784,   # United Arab Emirates
    "UK": 826,      # United Kingdom
    "US": 840,      # United States of America (note: UN SDG API often uses 840)
    "Japan": 392,
    "China": 156,
    "Brazil": 76,
    "India": 356,
    # New actors
    "Israel": 376,
    "Paraguay": 600,
    "Belgium": 56,
    "Denmark": 208,
    "Ukraine": 804,
    "Serbia": 688,
    "Argentina": 32,
    "Norway": 578,
    "Switzerland": 756,
    "Poland": 616,
}

# --- UN Comtrade reporter/partner numeric codes ---
# Classic Comtrade API examples use numeric codes (e.g., Germany=276). For Charlie list,
# these align with the same UN M49 numeric codes above.
ACTOR_TO_COMTRADE: Dict[str, int] = dict(ACTOR_TO_M49)

def actor_to_iso3(actor: str) -> Optional[str]:
    return ACTOR_TO_ISO3.get(actor)

def actor_to_m49(actor: str) -> Optional[int]:
    return ACTOR_TO_M49.get(actor)

def actor_to_comtrade(actor: str) -> Optional[int]:
    return ACTOR_TO_COMTRADE.get(actor)
'''
pathlib.Path("agpo_data/actor_map.py").write_text(actor_map, encoding="utf-8")

# ----------------------------
# pages/21_Real_World_Data_Metrics.py (new Streamlit page)
# ----------------------------

# ----------------------------
# agpo_data/imf_weo.py (new helper - IMF DataMapper / WEO)
# ----------------------------
imf_weo = r'''"""AGPO Data Package - IMF DataMapper (WEO-style) integration.

Uses IMF DataMapper API v1 (public read) to fetch macro stress indicators.
- Base: https://www.imf.org/external/datamapper/api/v1

Charlie usage:
- GDP growth, inflation, debt-to-GDP as NOF inputs (capacity/stress).
"""

from __future__ import annotations
from typing import Any, Dict, Optional
import requests
import pandas as pd
import streamlit as st

IMF_BASE = "https://www.imf.org/external/datamapper/api/v1"

def _is_number(x: Any) -> bool:
    try:
        float(x)
        return True
    except Exception:
        return False

@st.cache_data(ttl=3600)
def get_series(indicator: str, iso3: str, dataset: str = "WEO") -> pd.DataFrame:
    """Return a tidy DataFrame: Year, Value for indicator@dataset and ISO3."""
    # DataMapper API commonly works as /<indicator>/<ISO3>. Some datasets also accept indicator@DATASET.
    # We try the simpler form first, then fall back to @dataset for compatibility.
    url = f"{IMF_BASE}/{indicator}/{iso3}"
    url_fallback = f"{IMF_BASE}/{indicator}@{dataset}/{iso3}"
    try:
        r = requests.get(url, timeout=25)
        if r.status_code >= 400:
            # try fallback
            r = requests.get(url_fallback, timeout=25)
        r.raise_for_status()
        js = r.json()
        # If response doesn't contain indicator key, try fallback form.
        if not (isinstance(js, dict) and indicator in js):
            r2 = requests.get(url_fallback, timeout=25)
            if r2.ok:
                js2 = r2.json()
                if isinstance(js2, dict) and indicator in js2:
                    js = js2
        if not isinstance(js, dict) or indicator not in js:
            return pd.DataFrame(columns=["Year", "Value"])
        data_block = None
        if isinstance(js, dict):
            if "values" in js and isinstance(js.get("values"), dict):
                data_block = js["values"]
            elif indicator in js:
                data_block = js
        if not isinstance(data_block, dict) or indicator not in data_block:
            return pd.DataFrame(columns=["Year", "Value"])
        iso_block = data_block.get(indicator, {}).get(iso3, {})
        if not isinstance(iso_block, dict):
            return pd.DataFrame(columns=["Year", "Value"])
        rows = []
        for y, v in iso_block.items():
            if str(y).isdigit() and _is_number(v):
                rows.append({"Year": int(y), "Value": float(v)})
        return pd.DataFrame(rows).sort_values("Year")
    except Exception:
        return pd.DataFrame(columns=["Year", "Value"])

def latest_value(df: pd.DataFrame) -> Optional[float]:
    if df is None or df.empty:
        return None
    try:
        return float(df.sort_values("Year", ascending=False).iloc[0]["Value"])
    except Exception:
        return None

def get_latest(indicator: str, iso3: str, dataset: str = "WEO") -> Optional[float]:
    return latest_value(get_series(indicator, iso3, dataset=dataset))

def macro_snapshot(iso3: str) -> Dict[str, Optional[float]]:
    """Opinionated default macro snapshot for Charlie's NOF layer."""
    return {
        "gdp_growth_pct": get_latest("NGDP_RPCH", iso3),
        "inflation_pct": get_latest("PCPIEPCH", iso3),
        "debt_gdp_pct": get_latest("GGXWDG_NGDP", iso3),
    }
'''
with open("agpo_data/imf_weo.py", "w", encoding="utf-8") as f:
    f.write(imf_weo)

# ----------------------------
# agpo_data/un_sdg.py (new helper - UNSD SDG API)
# ----------------------------
un_sdg = r'''"""AGPO Data Package - UN SDG Indicators API integration.

Uses the UNSD SDG API to retrieve SDG indicator observations.
- Base: https://unstats.un.org/SDGAPI/v1/sdg/...

Important:
- UNSD SDG API commonly uses UN M49 numeric geoAreaCode (not ISO3).
- We keep a small actor->M49 mapping in agpo_data.actor_map.

Charlie usage:
- Development inequality / social fragility features to enrich NOF layer.
"""

from __future__ import annotations
from typing import Any, Dict, Optional
import requests
import pandas as pd
import streamlit as st

SDG_BASE = "https://unstats.un.org/SDGAPI/v1/sdg"

def _is_number(x: Any) -> bool:
    try:
        float(x)
        return True
    except Exception:
        return False

@st.cache_data(ttl=3600)
def get_series(series_code: str, geo_area_code: int, time_period: Optional[int] = None) -> pd.DataFrame:
    """Fetch a tidy DataFrame: Year, Value for a single SDG series and geo area."""
    url = f"{SDG_BASE}/Series/Data"
    params: Dict[str, Any] = {"seriesCode": series_code, "areaCode": int(geo_area_code), "pageSize": 2000}
    if time_period is not None:
        params["timePeriod"] = int(time_period)
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        js = r.json()
        data = js.get("data", js)  # sometimes returns {data:[...], ...}
        if not isinstance(data, list):
            return pd.DataFrame(columns=["Year", "Value"])
        rows = []
        for item in data:
            if not isinstance(item, dict):
                continue
            if str(item.get("seriesCode", "")).upper() != series_code.upper():
                continue
            if int(item.get("geoAreaCode", -1)) != int(geo_area_code):
                continue
            y = item.get("timePeriod")
            v = item.get("value")
            if y is None or v is None:
                continue
            ys = str(y)
            if ys.isdigit() and _is_number(v):
                rows.append({"Year": int(ys), "Value": float(v)})
        return pd.DataFrame(rows).sort_values("Year")
    except Exception:
        return pd.DataFrame(columns=["Year", "Value"])

def latest_value(df: pd.DataFrame) -> Optional[float]:
    if df is None or df.empty:
        return None
    try:
        return float(df.sort_values("Year", ascending=False).iloc[0]["Value"])
    except Exception:
        return None

def get_latest(series_code: str, geo_area_code: int) -> Optional[float]:
    return latest_value(get_series(series_code, geo_area_code))

def social_snapshot(geo_area_code: int) -> Dict[str, Optional[float]]:
    """Opinionated social snapshot for Charlie's NOF layer.

    NOTE: series codes can vary by country/availability; missing series return None.
    """
    return {
        "poverty_rate": get_latest("SI_POV_DAY1", geo_area_code),
        "gini": get_latest("SI_POV_GINI", geo_area_code),
        "unemployment_rate": get_latest("SL_TLF_UEM", geo_area_code),
    }
'''
with open("agpo_data/un_sdg.py", "w", encoding="utf-8") as f:
    f.write(un_sdg)

# ----------------------------
# agpo_data/trade.py (new helper - UN Comtrade + optional WTO hook)
# ----------------------------
trade_mod = r'''"""AGPO Data Package - Trade leverage (UN Comtrade + optional WTO hooks).

Primary: UN Comtrade (legacy REST-style /api/get).

Design principle:
- Degrade gracefully if endpoint / auth changes: return empty DataFrames / None.
- Keep outputs as tidy DataFrames + small snapshot dicts for Charlie's NOF layer.

Optional: WTO APIs typically require a subscription key; we leave a future hook.
"""

from __future__ import annotations
from typing import Any, Dict, Optional
import os
import requests
import pandas as pd
import streamlit as st

# Legacy UN Comtrade endpoint:
COMTRADE_BASE = "https://comtrade.un.org/api/get"

def _is_number(x: Any) -> bool:
    try:
        float(x)
        return True
    except Exception:
        return False

@st.cache_data(ttl=3600)
def comtrade_query(
    reporter: int,
    partner: int = 0,
    year: int = 2022,
    trade_flow: str = "all",
    max_records: int = 500,
    commodity_code: str = "TOTAL",
) -> pd.DataFrame:
    """Query Comtrade and return a DataFrame (may be empty)."""
    params = {
        "max": int(max_records),
        "type": "C",
        "freq": "A",
        "px": "HS",
        "ps": str(year),
        "r": int(reporter),
        "p": int(partner),
        "rg": trade_flow,
        "cc": commodity_code,
        "fmt": "json",
    }
    try:
        r = requests.get(COMTRADE_BASE, params=params, timeout=35)
        r.raise_for_status()
        js = r.json()
        data = js.get("dataset", js.get("data", []))
        if not isinstance(data, list):
            return pd.DataFrame()
        return pd.DataFrame(data)
    except Exception:
        return pd.DataFrame()

def total_trade_value_usd(df: pd.DataFrame) -> Optional[float]:
    if df is None or df.empty:
        return None
    col = None
    for c in ["TradeValue", "tradeValue", "Trade Value (US$)"]:
        if c in df.columns:
            col = c
            break
    if col is None:
        return None
    try:
        vals = pd.to_numeric(df[col], errors="coerce").dropna()
        return float(vals.sum()) if not vals.empty else None
    except Exception:
        return None

def trade_snapshot(reporter: int, year: int = 2022) -> Dict[str, Optional[float]]:
    """Snapshot: imports & exports to world (partner=0)."""
    imp = total_trade_value_usd(comtrade_query(reporter, partner=0, year=year, trade_flow="1"))
    exp = total_trade_value_usd(comtrade_query(reporter, partner=0, year=year, trade_flow="2"))
    tot = (imp + exp) if (imp is not None and exp is not None) else None
    return {"imports_usd": imp, "exports_usd": exp, "total_trade_usd": tot}

def wto_is_configured() -> bool:
    return bool(os.getenv("WTO_SUBSCRIPTION_KEY", "").strip())
'''
with open("agpo_data/trade.py", "w", encoding="utf-8") as f:
    f.write(trade_mod)

real_world_page = '''# SPDX-License-Identifier: MIT
import json
import pandas as pd
import streamlit as st

from agpo_data.actor_map import actor_to_iso3, actor_to_m49, actor_to_comtrade
from agpo_data.worldbank import get_many_indicators
from agpo_data.imf_weo import macro_snapshot
from agpo_data.un_sdg import social_snapshot
from agpo_data.trade import trade_snapshot
from agpo_data.exportcontrol import get_export_control_snapshot

st.set_page_config(page_title="🌍 Real World Data Metrics", layout="wide", initial_sidebar_state="collapsed")
st.markdown("<style>div.block-container{padding-top:0.6rem;}</style>", unsafe_allow_html=True)

st.title("🌍 Real World Data Metrics")
st.caption("""World Bank + IMF (WEO) + UN SDG + UN Comtrade (Trade) — integrated as NOF inputs for Charlie's stress-tests.

Tip: Each source loads on-demand in its own tab to avoid heavy, all-at-once API calls.""")

# ----------------------------
# State cache (per session)
# ----------------------------
if "rw_cache" not in st.session_state:
    st.session_state["rw_cache"] = {}

def _cache_key(actor: str, source: str, extra: str = "") -> str:
    return f"{actor}:{source}:{extra}".strip(":")

def _store(actor: str, source: str, data, extra: str = "") -> None:
    st.session_state["rw_cache"][_cache_key(actor, source, extra)] = data

def _load(actor: str, source: str, extra: str = ""):
    return st.session_state["rw_cache"].get(_cache_key(actor, source, extra))

# ----------------------------
# Actor selection
# ----------------------------
ACTORS = [
    "Dubai", "UK", "US", "Japan", "China", "Brazil", "India", "NATO",
    "Israel", "Paraguay", "Belgium", "Denmark", "Ukraine", "Serbia",
    "Argentina", "Norway", "Switzerland", "Poland", "Global South"
]
actor = st.selectbox("Select Actor", ACTORS, index=2)

iso3 = actor_to_iso3(actor)
m49 = actor_to_m49(actor)
comtrade = actor_to_comtrade(actor)

with st.expander("Actor code mapping", expanded=False):
    st.write({"actor": actor, "iso3": iso3, "m49": m49, "comtrade_code": comtrade})
    if actor == "NATO":
        st.info("NATO is a coalition actor; many national datasets do not map cleanly. Some tabs may be empty.")

tabs = st.tabs(["🏦 World Bank", "📈 IMF (WEO-style)", "🧩 UN SDG", "📦 UN Comtrade", "🛂 Export Controls"])

# ----------------------------
# World Bank tab
# ----------------------------
with tabs[0]:
    st.subheader("🏦 World Bank")

    if not iso3:
        st.warning("No ISO3 mapping available for this actor.")
    else:
        wb_indicators = [
            "NY.GDP.MKTP.CD",        # GDP (current US$)
            "MS.MIL.XPND.GD.ZS",     # Military expenditure (% of GDP)
            "IT.NET.USER.ZS",        # Internet users (% of population)
            "SP.POP.TOTL",           # Population, total
        ]
        wb_labels = {
            "NY.GDP.MKTP.CD": "GDP (current US$)",
            "MS.MIL.XPND.GD.ZS": "Military Expenditure (% of GDP)",
            "IT.NET.USER.ZS": "Internet Users (% of population)",
            "SP.POP.TOTL": "Population, total",
        }

        colA, colB = st.columns([1, 2])
        with colA:
            load_wb = st.button("Load / Refresh World Bank", use_container_width=True)
        with colB:
            st.caption("World Bank series can be sparse by country/year. NULL often means 'not available' rather than a bug.")

        if load_wb or _load(actor, "worldbank") is None:
            with st.spinner("Fetching World Bank indicators..."):
                data = get_many_indicators(iso3, wb_indicators, labels=wb_labels)
            _store(actor, "worldbank", data)

        st.write("Latest values")
        st.json(_load(actor, "worldbank") or {})

# ----------------------------
# IMF tab
# ----------------------------
with tabs[1]:
    st.subheader("📈 IMF (WEO-style)")

    if not iso3:
        st.warning("No ISO3 mapping available for this actor.")
    else:
        colA, colB = st.columns([1, 2])
        with colA:
            load_imf = st.button("Load / Refresh IMF", use_container_width=True)
        with colB:
            st.caption("Indicators used: NGDP_RPCH (GDP growth), PCPIEPCH (inflation), GGXWDG_NGDP (debt/GDP).")

        if load_imf or _load(actor, "imf") is None:
            with st.spinner("Fetching IMF DataMapper series..."):
                data = macro_snapshot(iso3)
            _store(actor, "imf", data)

        st.json(_load(actor, "imf") or {})

        if (_load(actor, "imf") or {}).get("gdp_growth_pct") is None:
            st.info(
                "If values are NULL, it may mean the IMF series isn't available for this actor in the requested window.\n"
                "Next improvement: 'latest available year' fallback + optional debug view of raw response."
            )

# ----------------------------
# UN SDG tab
# ----------------------------
with tabs[2]:
    st.subheader("🧩 UN SDG (social fragility proxy)")

    colA, colB = st.columns([1, 2])
    with colA:
        load_sdg = st.button("Load / Refresh UN SDG", use_container_width=True)
    with colB:
        st.caption("UN SDG endpoints and coverage vary. NULL can mean missing series for the selected actor.")

    if load_sdg or _load(actor, "un_sdg") is None:
        with st.spinner("Fetching UN SDG snapshot..."):
            data = social_snapshot(m49=m49, iso3=iso3)
        _store(actor, "un_sdg", data)

    st.json(_load(actor, "un_sdg") or {})

# ----------------------------
# UN Comtrade tab
# ----------------------------
with tabs[3]:
    st.subheader("📦 UN Comtrade (trade leverage proxy)")

    years = list(range(2010, 2025))
    trade_year = st.select_slider("Trade Year", options=years, value=2024)

    colA, colB = st.columns([1, 2])
    with colA:
        load_trade = st.button("Load / Refresh UN Comtrade", use_container_width=True)
    with colB:
        st.caption("Trade snapshot uses partner=World (0) imports+exports. If endpoint changes, results may be empty (graceful).")

    if load_trade or _load(actor, "comtrade", str(trade_year)) is None:
        with st.spinner("Fetching UN Comtrade trade snapshot..."):
            data = trade_snapshot(reporter=iso3, year=trade_year)
        _store(actor, "comtrade", data, str(trade_year))

    st.json(_load(actor, "comtrade", str(trade_year)) or {})

# ----------------------------
# Export controls tab
# ----------------------------
with tabs[4]:
    st.subheader("🛂 Export Controls (sanctions / screening snapshot)")

    colA, colB = st.columns([1, 2])
    with colA:
        load_ec = st.button("Load / Refresh Export Controls", use_container_width=True)
    with colB:
        st.caption("Data sources may be optional depending on API keys and availability. Results are best-effort.")

    if load_ec or _load(actor, "exportcontrol") is None:
        with st.spinner("Fetching export control snapshot..."):
            data = get_export_control_snapshot(actor=actor, iso3=iso3)
        _store(actor, "exportcontrol", data)

    st.json(_load(actor, "exportcontrol") or {})

st.caption("These metrics are designed as *inputs* to Charlie’s NOF layer — not a predictive oracle.")
'''
pathlib.Path("pages/21_Real_World_Data_Metrics.py").write_text(real_world_page, encoding="utf-8")

print("✅ Patch applied: agpo_data modules updated + Real World Data Metrics page created.")
print("➡️ Re-run your Streamlit launch cell to refresh sidebar pages.")


✅ Patch applied: agpo_data modules updated + Real World Data Metrics page created.
➡️ Re-run your Streamlit launch cell to refresh sidebar pages.


In [10]:
# --- PATCH: overwrite pages/21_Real_World_Data_Metrics.py (signature fix for get_many_indicators) ---
import os
os.makedirs("pages", exist_ok=True)
CONTENT = "import os\nimport json\nimport streamlit as st\n\nfrom agpo_data.actor_map import actor_to_iso3, actor_to_m49, actor_to_comtrade\nfrom agpo_data.worldbank import get_many_indicators, latest_value\nfrom agpo_data.imf_weo import macro_snapshot\nfrom agpo_data.un_sdg import social_snapshot\nfrom agpo_data.trade import trade_snapshot\nfrom agpo_data.exportcontrol import get_export_control_snapshot\n\nst.set_page_config(page_title=\"Auracelle Charlie 3 - Real World Data Metrics\", layout=\"wide\", initial_sidebar_state=\"collapsed\")\nst.markdown(\"<style>div.block-container{padding-top:0.6rem;}</style>\", unsafe_allow_html=True)\n\n# --- Gate: require Session Setup (consent + profile) before viewing metrics ---\nif not st.session_state.get(\"consent\", False):\n    st.warning(\"Session Setup is required before accessing Real-World Data Metrics.\")\n    st.switch_page(\"pages/1_Session_Setup.py\")\n\nst.title(\"\u00f0\u009f\u008c\u008d Real-World Data Metrics\")\nst.caption(\n    \"\"\"World Bank + IMF (WEO-style) + UN SDG + UN Comtrade + Export Controls \u00e2\u0080\u0094 integrated as NOF inputs for Charlie's stress-tests.\n\nEach source loads on-demand in its own tab to avoid heavy, all-at-once API calls.\"\"\"\n)\n\nWB_MAP = {\n    \"GDP (current US$)\": \"NY.GDP.MKTP.CD\",\n    \"Military Expenditure (% of GDP)\": \"MS.MIL.XPND.GD.ZS\",\n    \"Internet Users (% of population)\": \"IT.NET.USER.ZS\",\n    \"Population, total\": \"SP.POP.TOTL\",\n}\n\nactors = st.session_state.get(\"country_options\") or [\"Dubai\", \"UK\", \"US\", \"Japan\", \"China\", \"Brazil\", \"India\", \"NATO\"]\nactor = st.selectbox(\"Select Actor\", actors, index=actors.index(\"US\") if \"US\" in actors else 0)\n\niso3 = actor_to_iso3(actor)\nm49 = actor_to_m49(actor)\nct = actor_to_comtrade(actor)\n\ncolA, colB, colC = st.columns(3)\nwith colA:\n    st.write(\"**ISO3**\")\n    st.code(str(iso3))\nwith colB:\n    st.write(\"**UN M49**\")\n    st.code(str(m49))\nwith colC:\n    st.write(\"**Comtrade reporter/area**\")\n    st.code(str(ct))\n\ntab_wb, tab_imf, tab_un, tab_trade, tab_xc = st.tabs([\"\u00f0\u009f\u008f\u00a6 World Bank\", \"\u00f0\u009f\u0093\u0088 IMF (WEO-style)\", \"\u00f0\u009f\u00a7\u00a9 UN SDG\", \"\u00f0\u009f\u008c\u0090 UN Comtrade\", \"\u00f0\u009f\u009b\u0083 Export Controls\"])\n\ndef _cache_key(prefix: str) -> str:\n    return f\"rwd::{prefix}::{actor}\"\n\n# -----------------------\n# World Bank\n# -----------------------\nwith tab_wb:\n    st.subheader(\"World Bank\")\n    if st.button(\"Load / Refresh World Bank\", use_container_width=True, key=\"wb_refresh\"):\n        wb_df = get_many_indicators(WB_MAP, [iso3])\n        latest = {label: latest_value(wb_df, iso3, code) for label, code in WB_MAP.items()}\n        st.session_state[_cache_key(\"wb\")] = latest\n\n    wb_data = st.session_state.get(_cache_key(\"wb\"))\n    if wb_data is None:\n        st.info(\"Click **Load / Refresh World Bank** to fetch indicators.\")\n    else:\n        st.write(\"Latest values\")\n        st.json(wb_data)\n\n# -----------------------\n# IMF (WEO-style)\n# -----------------------\nwith tab_imf:\n    st.subheader(\"IMF (WEO-style)\")\n    st.caption(\"Indicators used: NGDP_RPCH (GDP growth), PCPIEPCH (inflation), GGXWDG_NGDP (debt/GDP).\")\n\n    if st.button(\"Load / Refresh IMF\", use_container_width=True, key=\"imf_refresh\"):\n        imf = macro_snapshot(iso3)\n        st.session_state[_cache_key(\"imf\")] = imf\n\n    imf_data = st.session_state.get(_cache_key(\"imf\"))\n    if imf_data is None:\n        st.info(\"Click **Load / Refresh IMF** to fetch macro indicators.\")\n    else:\n        st.json(imf_data)\n        if all(imf_data.get(k) is None for k in (\"gdp_growth_pct\", \"inflation_pct\", \"debt_gdp_pct\")):\n            st.info(\n                \"If values are NULL, it may mean the IMF series isn't available for this actor in the requested window. \"\n                \"Next improvement: try a 'latest available year' fallback and add an optional debug view.\"\n            )\n\n# -----------------------\n# UN SDG\n# -----------------------\nwith tab_un:\n    st.subheader(\"UN SDG (social fragility proxy)\")\n    if st.button(\"Load / Refresh UN SDG\", use_container_width=True, key=\"un_refresh\"):\n        un = social_snapshot(m49)\n        st.session_state[_cache_key(\"un\")] = un\n\n    un_data = st.session_state.get(_cache_key(\"un\"))\n    if un_data is None:\n        st.info(\"Click **Load / Refresh UN SDG** to fetch social indicators.\")\n    else:\n        st.json(un_data)\n\n# -----------------------\n# UN Comtrade\n# -----------------------\nwith tab_trade:\n    st.subheader(\"UN Comtrade (trade leverage proxy)\")\n    year = st.slider(\"Trade Year\", min_value=2010, max_value=2024, value=2024, step=1)\n    if st.button(\"Load / Refresh UN Comtrade\", use_container_width=True, key=\"ct_refresh\"):\n        trade = trade_snapshot(ct, year=year)\n        st.session_state[_cache_key(f\"trade::{year}\")] = trade\n\n    trade_data = st.session_state.get(_cache_key(f\"trade::{year}\"))\n    if trade_data is None:\n        st.info(\"Click **Load / Refresh UN Comtrade** to fetch trade snapshot.\")\n        st.caption(\"Trade snapshot uses partner=World (0) imports+exports. If the endpoint changes, results may be empty (graceful).\")\n    else:\n        st.json(trade_data)\n\n# -----------------------\n# Export Controls\n# -----------------------\nwith tab_xc:\n    st.subheader(\"Export Controls (snapshot)\")\n    st.caption(\"Uses available sources configured in the exportcontrol module. Some sources may require optional API keys.\")\n    if st.button(\"Load / Refresh Export Controls\", use_container_width=True, key=\"xc_refresh\"):\n        xc = get_export_control_snapshot(actor)\n        st.session_state[_cache_key(\"xc\")] = xc\n\n    xc_data = st.session_state.get(_cache_key(\"xc\"))\n    if xc_data is None:\n        st.info(\"Click **Load / Refresh Export Controls** to fetch export control snapshot.\")\n    else:\n        st.json(xc_data)\n"
with open("pages/21_Real_World_Data_Metrics.py", "w", encoding="utf-8") as f:
    f.write(CONTENT)
print("✅ Patched pages/21_Real_World_Data_Metrics.py")


✅ Patched pages/21_Real_World_Data_Metrics.py


In [11]:
import os

# --- Hard-coded keys for Colab runtime (loaded BEFORE FastAPI starts) ---
os.environ["COMTRADE_SUBSCRIPTION_KEY"] = "617366b02a264811b5a34f2a0a9d0f69"
os.environ["TRADEGOV_API_KEY"] = "68db2c4484df4e9498dfa421a9e0ce5a"
os.environ["X_BEARER_TOKEN"] = "AAAAAAAAAAAAAAAAAAAAK%2Bf7QEAAAAAP4YCumQBIO1109m3hxaA1K8%2Bw2A%3D3n2y1NjOO5SkY2ien6EvzjItuuyWoKbTwBaeU3FlXCufHCY2TV"

print("COMTRADE_SUBSCRIPTION_KEY set:", bool(os.getenv("COMTRADE_SUBSCRIPTION_KEY")))
print("TRADEGOV_API_KEY set:", bool(os.getenv("TRADEGOV_API_KEY")))
print("X_BEARER_TOKEN set:", bool(os.getenv("X_BEARER_TOKEN")))

COMTRADE_SUBSCRIPTION_KEY set: True
TRADEGOV_API_KEY set: True
X_BEARER_TOKEN set: True


In [12]:
# ========================================
# AURACELLE FASTAPI RESEARCH BACKEND (v2.0)
# - Adds WB time-series + Comtrade partner breakdown (for charts/Sankey/map)
# - Adds "flagged partners only" filter (sanctions/export controls/high tariffs)
# - Keeps defensive timeouts to avoid Streamlit 45s client timeout
# ========================================

!pip install -q fastapi uvicorn nest-asyncio python-multipart requests wbgapi pandas numpy

import nest_asyncio
import uvicorn
import threading
import logging
import os
import time
from datetime import datetime
from typing import Optional, Dict, Any, List
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

import requests
import pandas as pd
import numpy as np
import wbgapi as wb

from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field

nest_asyncio.apply()

logging.basicConfig(level=logging.INFO)
log = logging.getLogger("auracelle-fastapi")

# ---- Shared HTTP session + worker pool ----
HTTP = requests.Session()
try:
    from requests.adapters import HTTPAdapter
    HTTP.mount('https://', HTTPAdapter(pool_connections=32, pool_maxsize=32, max_retries=1))
    HTTP.mount('http://', HTTPAdapter(pool_connections=32, pool_maxsize=32, max_retries=1))
except Exception:
    pass

EXEC = ThreadPoolExecutor(max_workers=12)

# Latency budgets (seconds)
WB_TIMEOUT = 10
IMF_TIMEOUT = 8
CSL_TIMEOUT = 10
COMTRADE_TIMEOUT = 18
WITS_TIMEOUT = 25
SANCTIONS_TIMEOUT = 10
TARIFF_TIMEOUT = 12
CULTURE_TIMEOUT = 8

# -----------------------------
# Actor registry (UI + backend)
# -----------------------------
# Notes:
# - "wb_code" can be ISO3 (e.g., USA) or a World Bank aggregate (e.g., LMY for Low & middle income)
# - "comtrade_r" uses UN M49 numeric reporter codes (e.g., USA=840). Some entities don't have Comtrade reporting.
ACTORS: Dict[str, Dict[str, Any]] = {
    # Existing baseline actors
    "Dubai": {"iso3": "ARE", "wb_code": "ARE", "comtrade_r": 784},
    "UK": {"iso3": "GBR", "wb_code": "GBR", "comtrade_r": 826},
    "US": {"iso3": "USA", "wb_code": "USA", "comtrade_r": 840},
    "Japan": {"iso3": "JPN", "wb_code": "JPN", "comtrade_r": 392},
    "China": {"iso3": "CHN", "wb_code": "CHN", "comtrade_r": 156},
    "Brazil": {"iso3": "BRA", "wb_code": "BRA", "comtrade_r": 76},
    "India": {"iso3": "IND", "wb_code": "IND", "comtrade_r": 356},
    "NATO": {"iso3": None, "wb_code": None, "comtrade_r": None},

    # New requested actors
    "Israel": {"iso3": "ISR", "wb_code": "ISR", "comtrade_r": 376},
    "Paraguay": {"iso3": "PRY", "wb_code": "PRY", "comtrade_r": 600},
    "Belgium": {"iso3": "BEL", "wb_code": "BEL", "comtrade_r": 56},
    "Denmark": {"iso3": "DNK", "wb_code": "DNK", "comtrade_r": 208},
    "Ukraine": {"iso3": "UKR", "wb_code": "UKR", "comtrade_r": 804},
    "Serbia": {"iso3": "SRB", "wb_code": "SRB", "comtrade_r": 688},
    "Argentina": {"iso3": "ARG", "wb_code": "ARG", "comtrade_r": 32},
    "Norway": {"iso3": "NOR", "wb_code": "NOR", "comtrade_r": 578},
    "Switzerland": {"iso3": "CHE", "wb_code": "CHE", "comtrade_r": 756},
    "Poland": {"iso3": "POL", "wb_code": "POL", "comtrade_r": 616},

    # "Global South" as an aggregate proxy:
    # Using World Bank "Low & middle income" aggregate (LMY).
    # Trade partners breakdown is not available as a single reporter in legacy Comtrade.
    "Global South": {"iso3": None, "wb_code": "LMY", "comtrade_r": None, "note": "World Bank aggregate (Low & middle income)."},
}

def resolve_actor(actor_name: str) -> Dict[str, Any]:
    if actor_name in ACTORS:
        return {"name": actor_name, **ACTORS[actor_name]}
    # Allow user to pass ISO3 directly
    a = actor_name.strip().upper()
    if len(a) == 3:
        return {"name": actor_name, "iso3": a, "wb_code": a, "comtrade_r": None}
    raise ValueError(f"Unknown actor '{actor_name}'")

# -----------------------------
# World Bank indicators
# -----------------------------
WB_INDICATORS = {
    "GDP (current US$)": "NY.GDP.MKTP.CD",
    "Population, total": "SP.POP.TOTL",
    "Internet Users (% of population)": "IT.NET.USER.ZS",
    "Military Expenditure (% of GDP)": "MS.MIL.XPND.GD.ZS",
    "Military Expenditure (current US$)": "MS.MIL.XPND.CD",
}

def wb_latest_and_series(wb_code: str, start_year: int, end_year: int) -> Dict[str, Any]:
    if not wb_code:
        return {"enabled": False, "note": "No World Bank code for this actor.", "latest": {}, "series": {}, "missing": list(WB_INDICATORS.keys())}

    years = list(range(start_year, end_year + 1))
    wb_times = [f"YR{y}" for y in years]
    out_series: Dict[str, List[Dict[str, Any]]] = {}
    latest: Dict[str, Any] = {}
    missing: List[str] = []

    try:
        df = wb.data.DataFrame(list(WB_INDICATORS.values()), economy=wb_code, time=wb_times, labels=False)

        if isinstance(df.index, pd.MultiIndex):
            df = df.reset_index()
            time_col = 'time' if 'time' in df.columns else df.columns[1]
        else:
            df = df.reset_index()
            time_col = 'time' if 'time' in df.columns else df.columns[0]

        def parse_year(x):
            if pd.isna(x):
                return None
            s = str(x)
            if s.startswith("YR"):
                s = s.replace("YR", "")
            try:
                return int(s)
            except Exception:
                return None

        df["year"] = df[time_col].apply(parse_year)

        for label, code in WB_INDICATORS.items():
            if code not in df.columns:
                missing.append(label)
                out_series[label] = []
                latest[label] = None
                continue
            sub = df[["year", code]].dropna()
            rows = [{"year": int(r.year), "value": float(r[code])} for r in sub.itertuples(index=False)]
            out_series[label] = rows
            latest[label] = rows[-1]["value"] if rows else None
            if not rows:
                missing.append(label)

    except Exception as e:
        return {"enabled": True, "note": f"World Bank fetch failed: {e}", "latest": {}, "series": {}, "missing": list(WB_INDICATORS.keys())}

    return {"enabled": True, "note": None, "latest": latest, "series": out_series, "missing": missing}

# -----------------------------
# Comtrade (legacy public API) partner breakdown
# -----------------------------
COMTRADE_BASE = "https://comtrade.un.org/api/get"

# ---- WITS SDMX (World Bank) fallback for partner trade flows ----
WITS_SDMX_BASE = "https://wits.worldbank.org/API/V1/SDMX/V21/datasource/tradestats-trade"

def _parse_wits_sdmx_partner_series(payload: Dict[str, Any]) -> Dict[str, float]:
    # Returns mapping partner_iso3 -> value
    try:
        structure = payload.get("structure", {})
        dims = (structure.get("dimensions", {}) or {}).get("series", []) or []
        partner_dim_pos = None
        partner_values = None
        for i, d in enumerate(dims):
            if str(d.get("id", "")).lower() == "partner":
                partner_dim_pos = i
                partner_values = d.get("values", []) or []
                break
        if partner_dim_pos is None or partner_values is None:
            return {}
        dataSets = payload.get("dataSets", []) or []
        if not dataSets:
            return {}
        series = (dataSets[0].get("series", {}) or {})
        out = {}
        for key, s in series.items():
            parts = key.split(":")
            if partner_dim_pos >= len(parts):
                continue
            p_idx = int(parts[partner_dim_pos])
            if p_idx >= len(partner_values):
                continue
            partner = str(partner_values[p_idx].get("id", "")).upper()
            obs = s.get("observations", {}) or {}
            if not obs:
                continue
            # Grab first observation value
            first_k = sorted(obs.keys(), key=lambda x: int(x) if str(x).isdigit() else 0)[0]
            val = obs[first_k][0] if isinstance(obs[first_k], list) and obs[first_k] else obs[first_k]
            if val is None:
                continue
            out[partner] = float(val)
        return out
    except Exception:
        return {}

def wits_partner_flows(reporter_iso3: str, year: int, indicator: str, timeout: int = 45) -> Dict[str, float]:
    # indicator examples: XPRT-TRD-VL, MPRT-TRD-VL
    rep = (reporter_iso3 or "").lower()
    url = f"{WITS_SDMX_BASE}/reporter/{rep}/year/{year}/partner/all/indicator/{indicator}?format=JSON"
    r = HTTP.get(url, timeout=timeout)
    r.raise_for_status()
    payload = r.json()
    return _parse_wits_sdmx_partner_series(payload)

def _comtrade_call(params: Dict[str, Any], timeout: int) -> Dict[str, Any]:
    r = HTTP.get(COMTRADE_BASE, params=params, timeout=timeout)
    r.raise_for_status()
    try:
        return r.json()
    except Exception as e:
        txt = (r.text or "")[:300].replace("\n", " ")
        raise ValueError(f"Comtrade non-JSON response (status={r.status_code}): {txt}") from e

def comtrade_partners(reporting_code: Optional[int], actor_iso3: Optional[str], year: int, top_n: int = 20) -> Dict[str, Any]:
    if (not reporting_code) and (not actor_iso3):
        return {"enabled": False, "note": "No reporter code/ISO3 for this actor.", "year": year, "imports": [], "exports": [], "totals": {}}

    common = {
        "max": 50000,
        "type": "C",
        "freq": "A",
        "px": "HS",
        "ps": year,
        "r": reporting_code,
        "p": "all",
        "cc": "TOTAL",
        "fmt": "json",
    }

    def parse_partner_rows(dataset: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        if not dataset:
            return []
        df = pd.DataFrame(dataset)

        value_col = next((c for c in ["TradeValue", "tradeValue", "Trade Value"] if c in df.columns), None)
        if value_col is None:
            return []

        partner_code_col = next((c for c in ["ptCode", "PartnerCode", "partnerCode", "p"] if c in df.columns), None)
        partner_name_col = next((c for c in ["ptTitle", "Partner", "partnerDesc"] if c in df.columns), None)
        partner_iso3_col = next((c for c in ["pt3ISO", "partnerISO", "partnerISO3", "partnerISO3Code"] if c in df.columns), None)

        df = df.dropna(subset=[value_col])

        group_cols = [c for c in [partner_iso3_col, partner_code_col, partner_name_col] if c]
        if not group_cols:
            return []

        agg = (
            df.groupby(group_cols, dropna=True)[value_col]
              .sum()
              .reset_index()
              .sort_values(value_col, ascending=False)
              .head(top_n)
        )

        out = []
        for row in agg.itertuples(index=False):
            d = row._asdict()
            out.append({
                "partner_iso3": str(d.get(partner_iso3_col) or "").upper() if partner_iso3_col else None,
                "partner_code": int(d.get(partner_code_col)) if partner_code_col and pd.notna(d.get(partner_code_col)) else None,
                "partner_name": str(d.get(partner_name_col)) if partner_name_col else None,
                "usd": float(d.get(value_col)) if pd.notna(d.get(value_col)) else None,
            })
        return out

    imp_params = dict(common); imp_params["rg"] = 1
    exp_params = dict(common); exp_params["rg"] = 2

    try:
        imp = _comtrade_call(imp_params, timeout=COMTRADE_TIMEOUT)
        exp = _comtrade_call(exp_params, timeout=COMTRADE_TIMEOUT)

        imp_rows = parse_partner_rows(imp.get("dataset", []))
        exp_rows = parse_partner_rows(exp.get("dataset", []))

        totals = {
            "imports_total_usd": sum(r.get("value_usd", 0.0) for r in imp_rows),
            "exports_total_usd": sum(r.get("value_usd", 0.0) for r in exp_rows),
        }
        totals["total_trade_usd"] = totals["imports_total_usd"] + totals["exports_total_usd"]

        return {"enabled": True, "note": None, "year": year, "imports": imp_rows, "exports": exp_rows, "totals": totals}

    except Exception as e:
        # Fallback: WITS SDMX partner breakdown (no key required; JSON SDMX)
        try:
            exp_map = wits_partner_flows(actor_iso3 or "", year, "XPRT-TRD-VL", timeout=WITS_TIMEOUT)
            imp_map = wits_partner_flows(actor_iso3 or "", year, "MPRT-TRD-VL", timeout=WITS_TIMEOUT)

            def top_rows(m: Dict[str, float]) -> List[Dict[str, Any]]:
                rows = [{"partner_iso3": k, "value_usd": float(v)} for k, v in (m or {}).items() if k and k != "WLD"]
                rows.sort(key=lambda x: x["value_usd"], reverse=True)
                return rows[:top_n]

            exp_rows = top_rows(exp_map)
            imp_rows = top_rows(imp_map)

            totals = {
                "imports_total_usd": sum(r["value_usd"] for r in imp_rows),
                "exports_total_usd": sum(r["value_usd"] for r in exp_rows),
            }
            totals["total_trade_usd"] = totals["imports_total_usd"] + totals["exports_total_usd"]

            note = f"Comtrade failed; used WITS fallback. ({e})"
            return {"enabled": True, "note": note, "year": year, "imports": imp_rows, "exports": exp_rows, "totals": totals}
        except Exception as e2:
            note = f"Comtrade fetch failed: {e}; WITS fallback failed: {e2}"
            return {"enabled": True, "note": note, "year": year, "imports": [], "exports": [], "totals": {}}


# -----------------------------
# Export controls (CSL) - country extraction for "flagged partners"
# -----------------------------
def csl_flagged_countries(actor_iso3: Optional[str], query: str) -> Dict[str, Any]:
    api_key = os.environ.get("TRADEGOV_API_KEY", "").strip()
    if not api_key:
        return {"enabled": False, "note": "TRADEGOV_API_KEY not set; export controls flagging disabled.", "countries": []}

    url = "https://api.trade.gov/consolidated_screening_list/search"
    params = {"api_key": api_key, "q": query, "size": 100}

    try:
        r = HTTP.get(url, params=params, timeout=CSL_TIMEOUT)
        r.raise_for_status()
        try:
            data = r.json()
        except Exception as e:
            txt = (r.text or "")[:300].replace("\n", " ")
            raise ValueError(f"CSL non-JSON response (status={r.status_code}): {txt}") from e
        results = data.get("results", []) or data.get("results", {}).get("items", []) or []
        countries = set()
        for item in results:
            for k in ["country", "source_country", "address_country"]:
                v = item.get(k)
                if v:
                    countries.add(str(v).upper())
        return {"enabled": True, "note": None if countries else "No CSL results returned.", "countries": sorted(countries)}
    except Exception as e:
        return {"enabled": True, "note": f"CSL query failed: {e}", "countries": []}

# -----------------------------
# Sanctions / Tariffs / Culture (placeholders for v2)
# -----------------------------
def sanctions_flagged_partners(actor_iso3: Optional[str]) -> Dict[str, Any]:
    return {"enabled": False, "note": "Sanctions endpoint not yet wired. See step-by-step checklist in chat.", "countries": []}

def tariff_flagged_partners(actor_iso3: Optional[str], year: int) -> Dict[str, Any]:
    return {"enabled": False, "note": "Tariff endpoint not yet wired. See step-by-step checklist in chat.", "countries": []}

def cultural_friction(actor_iso3: Optional[str], partner_iso3_list: List[str]) -> Dict[str, Any]:
    return {"enabled": False, "note": "Cultural index not yet wired. See step-by-step checklist in chat.", "by_partner": []}

# -----------------------------
# Policy overlay stub (replace with your policy registry)
# -----------------------------
POLICY_EFFECTS = {
    "Export Controls Tightening": [
        {"indicator": "Exports", "direction": "down", "rationale": "Reduced partner set / higher compliance frictions."},
        {"indicator": "Security risk", "direction": "down", "rationale": "Lower leakage to restricted entities."},
    ],
    "AI Data Localization": [
        {"indicator": "Internet Users (% of population)", "direction": "mixed", "rationale": "Depends on implementation and platform responses."},
        {"indicator": "FDI / trade openness", "direction": "down", "rationale": "Friction for cross-border services/data flows."},
    ],
}



def compute_stress_test(policy_name: str,
                        actor: Dict[str, Any],
                        wb_block: Dict[str, Any],
                        imf_block: Dict[str, Any],
                        trade_block: Dict[str, Any],
                        flags: Dict[str, Any]) -> Dict[str, Any]:
    # Heuristic outcomes (decision-support), updated as more datasets become available.
    latest = (wb_block or {}).get("latest", {}) or {}
    snap = (imf_block or {}).get("snapshot", {}) or {}

    gdp = latest.get("GDP (current US$)")
    internet = latest.get("Internet Users (% of population)")
    mil_pct = latest.get("Military Expenditure (% of GDP)")
    debt = snap.get("debt_gdp_pct")

    totals = (trade_block or {}).get("totals", {}) or {}
    total_trade = totals.get("total_trade_usd")

    # Concentration proxy (top-3 share) if trade rows available
    flows = (trade_block or {}).get("imports_flagged_only") or (trade_block or {}).get("imports") or []
    flows2 = (trade_block or {}).get("exports_flagged_only") or (trade_block or {}).get("exports") or []
    rows = sorted((flows + flows2), key=lambda x: float(x.get("value_usd") or 0.0), reverse=True)
    top3 = sum(float(r.get("value_usd") or 0.0) for r in rows[:3])
    denom = float(total_trade) if total_trade else (sum(float(r.get("value_usd") or 0.0) for r in rows) or 1.0)
    concentration = (top3 / denom) if denom else 0.0

    flagged_iso3 = (trade_block or {}).get("flagged_iso3", []) or []
    flagged_n = len(flagged_iso3)

    # Missingness drives confidence down
    missing = (wb_block or {}).get("missing", []) or []
    missing_n = len(missing)

    base_risk = 40.0
    base_reward = 40.0

    # Debt pressure increases risk
    if isinstance(debt, (int, float)):
        base_risk += min(25.0, max(0.0, (float(debt) - 60.0) / 4.0))
    # High internet penetration improves implementation capacity
    if isinstance(internet, (int, float)):
        base_reward += min(20.0, max(0.0, (float(internet) - 50.0) / 2.0))
    # High concentration increases fragility
    base_risk += min(20.0, 50.0 * concentration)
    # Flag count as friction proxy
    base_risk += min(30.0, 5.0 * flagged_n)

    # Policy-specific tilts
    key_risks = []
    mitigations = []
    recs = []
    horizon = "6-18 months"
    conf = "medium"

    if policy_name == "AI Data Localization":
        base_risk += 8.0
        key_risks += [
            "Cross-border services friction (data transfer approvals, compliance overhead).",
            "Partner retaliation risk where dependencies are high (cloud, payments, SaaS, R&D)."
        ]
        mitigations += [
            "Adopt tiered localization (sensitive classes only) + explicit adequacy pathways.",
            "Pre-negotiate data corridors with top partners; publish transparent compliance playbooks."
        ]
        recs += [
            "Run partner-specific impact drills for the top 5 trade/service partners (substitution + coordination cost).",
            "Track leading indicators: internet penetration trend, digital trade share proxy, and export concentration."
        ]
        horizon = "3-12 months"
    else:
        base_risk += 10.0
        key_risks += [
            "Supply-chain substitution shock (controls shift sourcing and compliance cost).",
            "Escalation risk if controls intersect with existing sanctions regimes."
        ]
        mitigations += [
            "Scope controls narrowly (risk-based lists) + include licensing pathways to reduce blanket shocks.",
            "Create compliance tooling and shared screening workflows for industry partners."
        ]
        recs += [
            "Stress-test targeted partners first (flagged list) and model substitution to nearest unflagged suppliers.",
            "Add sanctions + tariff endpoints to calibrate pressure, not just screening hits."
        ]
        horizon = "6-24 months"

    if missing_n >= 3:
        conf = "low"
    if flagged_n == 0 and (flags or {}).get("sanctions", {}).get("enabled") is False:
        conf = "low"

    risk_score = max(0.0, min(100.0, base_risk))
    reward_score = max(0.0, min(100.0, base_reward))

    return {
        "policy_name": policy_name,
        "risk_score": risk_score,
        "reward_score": reward_score,
        "horizon": horizon,
        "confidence": conf,
        "key_risks": key_risks,
        "mitigations": mitigations,
        "foresight_recommendations": recs,
        "signals": {
            "flagged_partners_n": flagged_n,
            "trade_concentration_top3_share": concentration,
            "missing_worldbank_indicators_n": missing_n
        }
    }


# -----------------------------
# FastAPI app
# -----------------------------
app = FastAPI(title="Auracelle Research Backend", version="2.0")

class MetricsRequestV2(BaseModel):
    actor: str = Field(..., description="Actor/country label from UI dropdown (e.g., US).")
    start_year: int = 2010
    end_year: int = 2024
    trade_year: int = 2024
    csl_query: str = "bank"
    flagged_only: bool = True
    top_n_partners: int = 20
    policy_name: str = "AI Data Localization"

def run_with_timeout(fn, timeout_s: int, *args, **kwargs):
    fut = EXEC.submit(fn, *args, **kwargs)
    try:
        return fut.result(timeout=timeout_s)
    except FuturesTimeoutError:
        raise TimeoutError(f"Timed out after {timeout_s}s")

@app.post("/v2/metrics")
def metrics_v2(req: MetricsRequestV2):
    try:
        actor = resolve_actor(req.actor)
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

    if req.start_year > req.end_year:
        raise HTTPException(status_code=400, detail="start_year must be <= end_year")

    # World Bank
    try:
        wb_block = run_with_timeout(wb_latest_and_series, WB_TIMEOUT, actor.get("wb_code"), req.start_year, req.end_year)
    except Exception as e:
        wb_block = {"enabled": True, "note": f"World Bank timed out/failed: {e}", "latest": {}, "series": {}, "missing": list(WB_INDICATORS.keys())}

    # IMF (optional)
    imf_block = {"enabled": False, "note": "IMF module not wired in backend v2 (optional).", "snapshot": {}, "series": {}}
    try:
        from agpo_data.imf_weo import macro_snapshot
        snap = run_with_timeout(macro_snapshot, IMF_TIMEOUT, actor.get("iso3"))
        if isinstance(snap, dict):
            imf_block = {"enabled": True, "note": None, "snapshot": snap, "series": {}}
    except Exception as e:
        imf_block = {"enabled": True, "note": f"IMF snapshot unavailable: {e}", "snapshot": {}, "series": {}}

    # Trade partners
    try:
        trade_block = run_with_timeout(comtrade_partners, COMTRADE_TIMEOUT, actor.get("comtrade_r"), actor.get("iso3"), req.trade_year, req.top_n_partners)
    except Exception as e:
        trade_block = {"enabled": True, "note": f"Comtrade timed out/failed: {e}", "year": req.trade_year, "imports": [], "exports": [], "totals": {}}

    # Flags
    csl_block = csl_flagged_countries(actor.get("iso3"), req.csl_query)
    sanctions_block = sanctions_flagged_partners(actor.get("iso3"))
    tariff_block = tariff_flagged_partners(actor.get("iso3"), req.trade_year)

    # Compute flagged ISO3 set (best-effort)
    flagged_iso3 = set()
    for v in (csl_block.get("countries") or []):
        if isinstance(v, str) and len(v.strip()) == 3:
            flagged_iso3.add(v.strip().upper())
    for v in (sanctions_block.get("countries") or []):
        if isinstance(v, str) and len(v.strip()) == 3:
            flagged_iso3.add(v.strip().upper())
    for v in (tariff_block.get("countries") or []):
        if isinstance(v, str) and len(v.strip()) == 3:
            flagged_iso3.add(v.strip().upper())

    def filt(rows):
        if not req.flagged_only:
            return rows
        # If we have no flags yet (common early on), fall back to unfiltered so charts/maps still render.
        # Streamlit will display a warning that sanctions/tariffs are not wired.
        if not flagged_iso3:
            return rows
        out = []
        for x in rows:
            iso3 = (x.get("partner_iso3") or "").strip().upper()
            if iso3 and iso3 in flagged_iso3:
                out.append(x)
        return out

    trade_filtered = dict(trade_block)
    trade_filtered["imports_flagged_only"] = filt(trade_block.get("imports", []))
    trade_filtered["exports_flagged_only"] = filt(trade_block.get("exports", []))
    trade_filtered["flagged_iso3"] = sorted(flagged_iso3)

    partner_list = [x.get("partner_iso3") for x in (trade_filtered["imports_flagged_only"] + trade_filtered["exports_flagged_only"]) if x.get("partner_iso3")]
    culture_block = cultural_friction(actor.get("iso3"), partner_list)

    flags = {
        "export_controls_csl": csl_block,
        "sanctions": sanctions_block,
        "tariffs": tariff_block,
        "cultural_index": culture_block,
    }

    outcomes = compute_stress_test(req.policy_name, actor, wb_block, imf_block, trade_filtered, flags)

    policy_overlay = {
        "policy_name": req.policy_name,
        "effects": POLICY_EFFECTS.get(req.policy_name, []),
        "outcomes": outcomes,
        "note": None if req.policy_name in POLICY_EFFECTS else "No effects stub found for this policy."
    }

    payload = {
        "ok": True,
        "ts_utc": datetime.utcnow().isoformat() + "Z",
        "request": req.dict(),
        "actor": actor,
        "worldbank": wb_block,
        "imf": imf_block,
        "trade": trade_filtered,
        "flags": {
            "export_controls_csl": csl_block,
            "sanctions": sanctions_block,
            "tariffs": tariff_block,
            "cultural_index": culture_block,
        },
        "policy_overlay": policy_overlay,
    }
    return JSONResponse(payload)

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

if not globals().get("_AURACELLE_FASTAPI_V2_STARTED", False):
    _AURACELLE_FASTAPI_V2_STARTED = True
    th = threading.Thread(target=_run, daemon=True)
    th.start()
    time.sleep(1.0)
    print("✅ FastAPI backend v2 running on http://localhost:8000 (POST /v2/metrics)")
else:
    print("ℹ️ FastAPI backend v2 already started.")


✅ FastAPI backend v2 running on http://localhost:8000 (POST /v2/metrics)


In [13]:
# ========================================
# PATCH: Real-World Data Metrics now calls FastAPI backend
# ========================================

import pathlib
metrics_via_api = r'''
import streamlit as st
import requests
import uuid

st.set_page_config(page_title="Auracelle Charlie - Real World Data (API Mode)", layout="wide")

st.title("🌍 Real-World Data Metrics (FastAPI Backend)")
st.caption("This page now calls the Auracelle Research Engine backend (/v1/metrics).")

# Actor selection
actors = ["Dubai", "UK", "US", "Japan", "China", "Brazil", "India", "NATO"]
actor = st.selectbox("Select Actor", actors, index=actors.index("US"))

# Session ID for research logging
session_id = st.session_state.get("research_session_id")
if not session_id:
    session_id = str(uuid.uuid4())
    st.session_state["research_session_id"] = session_id

st.write("Research Session ID:", session_id)

if st.button("Run Metrics via Backend", use_container_width=True):

    try:
        response = requests.post(
            "http://localhost:8000/v1/metrics",
            json={
                "actor": actor,
                "session_id": session_id
            },
            timeout=30
        )

        if response.status_code == 200:
            data = response.json()
            st.success("Backend responded successfully.")
            st.json(data)
        else:
            st.error(f"Backend error: {response.status_code}")
            st.json(response.json())

    except requests.exceptions.RequestException as e:
        st.error("Backend connection failed.")
        st.write(str(e))

st.markdown("---")
st.markdown("### Backend Health Check")

if st.button("Check Backend Health"):
    try:
        r = requests.get("http://localhost:8000/v1/health", timeout=10)
        st.json(r.json())
    except Exception as e:
        st.error("Health check failed")
        st.write(str(e))
'''
pathlib.Path("pages/21_Real_World_Data_Metrics.py").write_text(metrics_via_api, encoding="utf-8")

print("✅ Real-World Data Metrics page now calls /v1/metrics backend endpoint.")

✅ Real-World Data Metrics page now calls /v1/metrics backend endpoint.


In [14]:
# ========================================
# PATCH v2.2: Real-World Data Metrics page (Plotly visuals + flagged partners)
# - Fixes StreamlitSecretNotFoundError (no secrets.toml required)
# - Mirrors selected policy from Simulation via st.session_state when available
# - Renames label to "Stress Test Outcomes"
# - Adds requested Actors/Countries and 'Global South'
# - Renders KPIs -> trends -> Sankey -> map -> policy overlay
# - Calls FastAPI backend /v2/metrics with flagged_only=True (fallback if no flags available)
# ========================================

import pathlib

pathlib.Path("pages").mkdir(parents=True, exist_ok=True)

PAGE = r'''# SPDX-License-Identifier: MIT
import os
import uuid
import requests
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(page_title="Auracelle Charlie - Real World Data (API v2)", layout="wide")

st.title("Real-World Data Metrics (FastAPI Backend v2)")
st.caption(
    "Backend schema v2 adds time-series + trade partners so we can render KPIs -> trends -> Sankey -> map -> policy overlay. "
    "Visuals prioritize *flagged* partners (sanctions/export controls/high tariffs)."
)

ACTORS = [
    "US", "UK", "Dubai", "Japan", "China", "Brazil", "India", "NATO",
    "Israel", "Paraguay", "Belgium", "Denmark", "Ukraine", "Serbia", "Argentina",
    "Norway", "Switzerland", "Poland", "Global South"
]

ACTOR_TO_ISO3 = {
    "US": "USA",
    "UK": "GBR",
    "Dubai": "ARE",          # proxy for UAE
    "Japan": "JPN",
    "China": "CHN",
    "Brazil": "BRA",
    "India": "IND",
    "NATO": "NATO",
    "Israel": "ISR",
    "Paraguay": "PRY",
    "Belgium": "BEL",
    "Denmark": "DNK",
    "Ukraine": "UKR",
    "Serbia": "SRB",
    "Argentina": "ARG",
    "Norway": "NOR",
    "Switzerland": "CHE",
    "Poland": "POL",
    "Global South": "LMY",   # WB aggregate: Low & middle income
}

def get_base_url(default="http://localhost:8000"):
    # Streamlit raises StreamlitSecretNotFoundError when secrets.toml doesn't exist.
    # Gracefully fall back to env var or default.
    try:
        return st.secrets.get("AURACELLE_API_BASE", os.environ.get("AURACELLE_API_BASE", default))
    except Exception:
        return os.environ.get("AURACELLE_API_BASE", default)

BASE_URL = get_base_url()

POSSIBLE_POLICY_KEYS = [
    "policy_scenario", "policy_selected", "selected_policy",
    "policy", "policy_name", "scenario_policy"
]

def get_policy_from_session(default="AI Data Localization"):
    for k in POSSIBLE_POLICY_KEYS:
        if k in st.session_state and st.session_state[k]:
            return str(st.session_state[k])
    return default

colA, colB, colC, colD = st.columns([1.2, 1.0, 1.0, 1.2])
with colA:
    actor = st.selectbox("Select Actor", ACTORS, index=0)
with colB:
    start_year, end_year = st.slider("Year Range", 1990, 2024, (2010, 2024))
with colC:
    trade_year = st.slider("Trade Year", 1990, 2024, 2024)
with colD:
    options = ["AI Data Localization", "Export Controls Tightening"]
    default_policy = get_policy_from_session(default=options[0])
    idx = options.index(default_policy) if default_policy in options else 0
    policy_name = st.selectbox("Stress Test Outcomes", options, index=idx)
    st.caption("Mirrors Simulation Policy Scenario when available.")

session_id = st.session_state.get("research_session_id") or str(uuid.uuid4())
st.session_state["research_session_id"] = session_id
st.write(f"Research Session ID: {session_id}")

q = st.text_input("CSL test query", value="bank")

def post_metrics():
    payload = {
        "actor": actor,
        "iso3": ACTOR_TO_ISO3.get(actor, "USA"),
        "start_year": int(start_year),
        "end_year": int(end_year),
        "trade_year": int(trade_year),
        "csl_query": q,
        "policy": policy_name,
        "flagged_only": True,
        "max_partners": 12,
        "session_id": session_id,
    }
    r = requests.post(f"{BASE_URL}/v2/metrics", json=payload, timeout=120)
    r.raise_for_status()
    return r.json()

def _safe_num(x):
    try:
        if x is None:
            return None
        return float(x)
    except Exception:
        return None

def render_kpis(wb_latest: dict, imf: dict):
    col1, col2, col3, col4, col5 = st.columns(5)
    gdp = _safe_num(wb_latest.get("GDP (current US$)"))
    pop = _safe_num(wb_latest.get("Population, total"))
    inet = _safe_num(wb_latest.get("Internet Users (% of population)"))
    mil = _safe_num(wb_latest.get("Military Expenditure (% of GDP) (SIPRI)"))
    debt = _safe_num(imf.get("debt_gdp_pct"))

    col1.metric("GDP (US$)", f"{gdp:,.0f}" if gdp else "—")
    col2.metric("Population", f"{pop:,.0f}" if pop else "—")
    col3.metric("Internet Users (%)", f"{inet:.1f}" if inet else "—")
    col4.metric("Mil Exp (% GDP)", f"{mil:.2f}" if mil else "—")
    col5.metric("Debt/GDP (%)", f"{debt:.1f}" if debt else "—")

def render_trends(wb_series: dict):
    st.subheader("Trends (World Bank time series)")
    if not wb_series:
        st.info("No time series returned yet.")
        return

    indicator = st.selectbox("Indicator", list(wb_series.keys()), index=0)
    series = wb_series.get(indicator, {})
    if not series:
        st.info("Selected indicator has no series.")
        return
    df = pd.DataFrame({"year": list(series.keys()), "value": list(series.values())}).sort_values("year")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna()
    if df.empty:
        st.info("Series is empty after cleaning nulls.")
        return
    st.line_chart(df.set_index("year"))

def render_sankey(trade: dict):
    st.subheader("Trade Flows (Flagged partners prioritized)")
    exports = (trade or {}).get("exports_by_partner", []) or []
    imports = (trade or {}).get("imports_by_partner", []) or []
    note = (trade or {}).get("note") or ""

    if not exports and not imports:
        st.info("No partner flows returned yet. (Comtrade partner breakdown may be empty.)")
        if note:
            st.caption(note)
        return

    actor_node = ACTOR_TO_ISO3.get(actor, "USA")
    partners = sorted({x.get("partner_iso3") for x in (exports + imports) if x.get("partner_iso3")})
    nodes = [actor_node] + partners
    idx = {n:i for i,n in enumerate(nodes)}

    sources, targets, values = [], [], []
    for row in exports:
        p = row.get("partner_iso3"); v = row.get("usd")
        if p and v:
            sources.append(idx[actor_node]); targets.append(idx[p]); values.append(float(v))
    for row in imports:
        p = row.get("partner_iso3"); v = row.get("usd")
        if p and v:
            sources.append(idx[p]); targets.append(idx[actor_node]); values.append(float(v))

    if not values:
        st.info("Flows exist but are all null/zero after cleaning.")
        if note:
            st.caption(note)
        return

    fig = go.Figure(data=[go.Sankey(
        node=dict(label=nodes, pad=15, thickness=18),
        link=dict(source=sources, target=targets, value=values),
    )])
    st.plotly_chart(fig, use_container_width=True)
    if note:
        st.caption(note)

def render_map(trade: dict, flags: dict):
    st.subheader("Partner Risk Map (Flagged only)")
    flags = flags or {}
    flagged = set(flags.get("flagged_iso3", []) or [])

    exports = (trade or {}).get("exports_by_partner", []) or []
    imports = (trade or {}).get("imports_by_partner", []) or []

    vols = {}
    for row in exports:
        p = row.get("partner_iso3"); v = row.get("usd") or 0
        if p: vols[p] = vols.get(p, 0) + float(v or 0)
    for row in imports:
        p = row.get("partner_iso3"); v = row.get("usd") or 0
        if p: vols[p] = vols.get(p, 0) + float(v or 0)

    partners = sorted(flagged) if flagged else sorted(vols.keys())
    if not partners:
        st.info("No partner countries available to map yet.")
        return

    df = pd.DataFrame([{"iso3": p, "trade_usd": vols.get(p, 0), "flagged": p in flagged} for p in partners])
    fig = px.choropleth(df, locations="iso3", color="trade_usd", hover_data=["flagged"],
                        title="Trade volume for flagged partners (or fallback partners)")
    st.plotly_chart(fig, use_container_width=True)

def render_policy_overlay(policy_overlay: dict, flags: dict):
    policy_name = (policy_overlay or {}).get("policy_name") or "Unknown"
    outcomes = (policy_overlay or {}).get("outcomes") or {}
    st.subheader("Stress Test Outcomes")
    st.caption("Data-driven heuristics (not a predictive oracle). Outcomes update as additional flag datasets (sanctions/tariffs/culture) come online.")

    if outcomes:
        risk = outcomes.get("risk_score")
        reward = outcomes.get("reward_score")
        horizon = outcomes.get("horizon")
        conf = outcomes.get("confidence")

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Risk score (0-100)", "—" if risk is None else f"{risk:.0f}")
        c2.metric("Reward score (0-100)", "—" if reward is None else f"{reward:.0f}")
        c3.metric("Time horizon", horizon or "—")
        c4.metric("Confidence", conf or "—")

        # Gauge
        if risk is not None or reward is not None:
            fig = go.Figure()
            if risk is not None:
                fig.add_trace(go.Indicator(mode="gauge+number", value=float(risk), title={"text": "Risk"}, domain={"x": [0, 0.48], "y": [0, 1]}))
            if reward is not None:
                fig.add_trace(go.Indicator(mode="gauge+number", value=float(reward), title={"text": "Reward"}, domain={"x": [0.52, 1], "y": [0, 1]}))
            fig.update_layout(height=260, margin=dict(l=10, r=10, t=30, b=10))
            st.plotly_chart(fig, use_container_width=True)

        risks = outcomes.get("key_risks", []) or []
        mitigations = outcomes.get("mitigations", []) or []
        recs = outcomes.get("foresight_recommendations", []) or []

        cL, cR = st.columns(2)
        with cL:
            st.markdown("**Key risks**")
            if risks:
                for r in risks:
                    st.write(f"- {r}")
            else:
                st.write("—")
        with cR:
            st.markdown("**Mitigations**")
            if mitigations:
                for r in mitigations:
                    st.write(f"- {r}")
            else:
                st.write("—")

        st.markdown("**Foresight recommendations**")
        if recs:
            for r in recs:
                st.write(f"- {r}")
        else:
            st.write("—")

    else:
        flagged = (flags or {}).get("flagged_iso3", []) or []
        if policy_name == "AI Data Localization":
            st.write("- Expected pressure: services trade friction up; data-transfer compliance cost up; partner coordination complexity up.")
        else:
            st.write("- Expected pressure: export/import constraints up; enforcement intensity up; partner substitution pressure up.")
        st.write(f"- Flagged partners count: {len(flagged)}")

if st.button("Run Metrics via Backend (v2)", type="primary"):
    try:
        payload = post_metrics()
        st.success("Backend responded successfully.")
    except Exception as e:
        st.error(f"Backend connection failed: {e}")
        st.stop()

    wb_latest = payload.get("worldbank", {}).get("latest", {}) or {}
    wb_series = payload.get("worldbank", {}).get("series", {}) or {}
    imf = payload.get("imf", {}) or {}
    trade = payload.get("trade", {}) or {}
    flags = payload.get("flags", {}) or {}

    render_kpis(wb_latest, imf)
    render_trends(wb_series)
    render_sankey(trade)
    render_map(trade, flags)
    render_policy_overlay(payload.get('policy_overlay', {}), flags)
'''

pathlib.Path("pages/21_Real_World_Data_Metrics.py").write_text(PAGE, encoding="utf-8")
print("✅ Patched pages/21_Real_World_Data_Metrics.py (secrets-safe + policy mirroring)")


✅ Patched pages/21_Real_World_Data_Metrics.py (secrets-safe + policy mirroring)
